# 13 — Forecast Volatility Model Research v0.1

Goal: improve the forecast-volatility denominator used in forecast VRP signals.

This notebook loads existing cleaned panels and compares better forecast-vol candidates against:
- trailing realized variance
- existing EWMA forecast
- existing simple hybrid forecast

Focus:
1. Inspect current forecast target/features.
2. Build robust baseline models.
3. Use walk-forward / expanding-window evaluation.
4. Recompute forecast VRP using improved denominator.
5. Check whether signal quality improves.

In [2]:
# ============================================================
# Cell 1: Setup
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

PROJECT_DIR = Path(r"C:\Users\patri\vrp_project")

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
AUDIT_DIR = DATA_DIR / "audit"

print("Project dir:", PROJECT_DIR)
print("Processed data dir:", PROCESSED_DATA_DIR)
print("Audit dir:", AUDIT_DIR)

assert PROJECT_DIR.exists(), f"Project dir does not exist: {PROJECT_DIR}"
assert PROCESSED_DATA_DIR.exists(), f"Processed data dir does not exist: {PROCESSED_DATA_DIR}"
assert AUDIT_DIR.exists(), f"Audit dir does not exist: {AUDIT_DIR}"

Project dir: C:\Users\patri\vrp_project
Processed data dir: C:\Users\patri\vrp_project\data\processed
Audit dir: C:\Users\patri\vrp_project\data\audit


In [4]:
# ============================================================
# Cell 2: Input / output paths
# ============================================================

# Existing input panels
FORECAST_FEATURE_TARGET_PARQUET = (
    PROCESSED_DATA_DIR / "vol_forecast_feature_target_panel_v0_1.parquet"
)

VIX_TERM_STRUCTURE_PARQUET = (
    PROCESSED_DATA_DIR / "vix_term_structure_history_v0_7_1_repaired_total_variance.parquet"
)

VOL_OF_VOL_FEATURES_PARQUET = (
    PROCESSED_DATA_DIR / "put_vol_of_vol_features_v0_1.parquet"
)

REALIZED_VARIANCE_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "realized_variance_panel_v0_1.parquet"
)

VRP_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "vrp_panel_v0_1.parquet"
)

CURRENT_SIMPLE_FORECAST_PARQUET = (
    PROCESSED_DATA_DIR / "vol_forecast_simple_panel_v0_1.parquet"
)

CURRENT_FORECAST_VRP_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vrp_panel_v0_1.parquet"
)

PHASE3_STRATEGY_PARQUET = (
    PROCESSED_DATA_DIR / "put_phase3_portfolio_strategy_v0_1.parquet"
)

# New outputs from this notebook
FORECAST_MODEL_COMPARISON_CSV = (
    AUDIT_DIR / "forecast_vol_model_comparison_v0_1.csv"
)

FORECAST_MODEL_PREDICTIONS_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_model_predictions_v0_1.parquet"
)

FORECAST_MODEL_CONFIG_JSON = (
    AUDIT_DIR / "forecast_vol_model_config_v0_1.json"
)

required_inputs = [
    FORECAST_FEATURE_TARGET_PARQUET,
    VIX_TERM_STRUCTURE_PARQUET,
    VOL_OF_VOL_FEATURES_PARQUET,
    REALIZED_VARIANCE_PANEL_PARQUET,
    VRP_PANEL_PARQUET,
    CURRENT_SIMPLE_FORECAST_PARQUET,
    CURRENT_FORECAST_VRP_PANEL_PARQUET,
    PHASE3_STRATEGY_PARQUET,
]

print("Checking required inputs...\n")

for path in required_inputs:
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:8s} {path}")

Checking required inputs...

OK       C:\Users\patri\vrp_project\data\processed\vol_forecast_feature_target_panel_v0_1.parquet
OK       C:\Users\patri\vrp_project\data\processed\vix_term_structure_history_v0_7_1_repaired_total_variance.parquet
OK       C:\Users\patri\vrp_project\data\processed\put_vol_of_vol_features_v0_1.parquet
OK       C:\Users\patri\vrp_project\data\processed\realized_variance_panel_v0_1.parquet
OK       C:\Users\patri\vrp_project\data\processed\vrp_panel_v0_1.parquet
OK       C:\Users\patri\vrp_project\data\processed\vol_forecast_simple_panel_v0_1.parquet
OK       C:\Users\patri\vrp_project\data\processed\forecast_vrp_panel_v0_1.parquet
OK       C:\Users\patri\vrp_project\data\processed\put_phase3_portfolio_strategy_v0_1.parquet


In [3]:
# ============================================================
# Cell 3: Inspect forecast feature / target panel
# ============================================================

forecast_feature_target_df = pd.read_parquet(
    FORECAST_FEATURE_TARGET_PARQUET
).copy()

print("Rows:", len(forecast_feature_target_df))
print("Columns:", len(forecast_feature_target_df.columns))

print("\nColumn names and dtypes:")
for col in forecast_feature_target_df.columns:
    print(f"{repr(col):60s} {forecast_feature_target_df[col].dtype}")

print("\nFirst 5 rows:")
display(forecast_feature_target_df.head())

print("\nFirst row transposed:")
display(forecast_feature_target_df.head(1).T)

print("\nMissingness:")
display(
    forecast_feature_target_df
    .isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_pct")
    .head(80)
)

print("\nNumeric summary:")
numeric_cols = forecast_feature_target_df.select_dtypes(include="number").columns.tolist()
display(forecast_feature_target_df[numeric_cols].describe().T)

Rows: 18099
Columns: 75

Column names and dtypes:
'trade_date'                                                 int64
'tenor'                                                      int64
'spx_close'                                                  float64
'spx_log_return'                                             float64
'spx_rsi_14'                                                 float64
'vix_style_vol'                                              float64
'trailing_realized_variance'                                 float64
'trailing_realized_vol'                                      float64
'vrp_trailing_variance_ratio'                                float64
'vrp_trailing_log_variance_ratio'                            float64
'vrp_trailing_vol_ratio'                                     float64
'primary_vrp_signal'                                         float64
'trailing_rv_5d_variance'                                    float64
'trailing_rv_5d_vol'                                     

,trade_date,tenor,spx_close,spx_log_return,spx_rsi_14,vix_style_vol,trailing_realized_variance,trailing_realized_vol,vrp_trailing_variance_ratio,vrp_trailing_log_variance_ratio,...,trailing_rv_252d_vol,trailing_rv_252d_num_returns,log_trailing_rv_252d_variance,forward_realized_variance,forward_realized_vol,forward_num_returns,log_forward_realized_variance,implied_variance,log_implied_variance,log_trailing_realized_variance
0,20180625,9,2717.07,-0.01382,41.761584,17.348587,0.010491,10.242504,2.868903,1.053930,...,1.663212,1,-8.192839,0.006165,7.851987,6,-5.088807,0.030097,-3.503318,-4.557248
1,20180625,12,2717.07,-0.01382,41.761584,17.490509,0.008085,8.991663,3.783771,1.330721,...,1.663212,1,-8.192839,0.009035,9.505109,8,-4.706681,0.030592,-3.487024,-4.817745
2,20180625,15,2717.07,-0.01382,41.761584,17.490890,0.006966,8.346020,4.392027,1.479791,...,1.663212,1,-8.192839,0.009398,9.694309,10,-4.667262,0.030593,-3.486980,-4.966771
3,20180625,18,2717.07,-0.01382,41.761584,17.491144,0.006002,7.747385,5.097135,1.628679,...,1.663212,1,-8.192839,0.010422,10.208696,13,-4.563861,0.030594,-3.486951,-5.115630
4,20180625,21,2717.07,-0.01382,41.761584,17.426522,0.006427,8.016921,4.725048,1.552878,...,1.663212,1,-8.192839,0.008951,9.461139,14,-4.715955,0.030368,-3.494354,-5.047231



First row transposed:


,0
trade_date,2.018062e+07
tenor,9.000000e+00
spx_close,2.717070e+03
spx_log_return,-1.381979e-02
spx_rsi_14,4.176158e+01
...,...
forward_num_returns,6.000000e+00
log_forward_realized_variance,-5.088807e+00
implied_variance,3.009735e-02
log_implied_variance,-3.503318e+00



Missingness:


,missing_pct
log_forward_realized_variance,0.000497
forward_realized_vol,0.000497
forward_realized_variance,0.000497
spx_log_return,0.000000
spx_rsi_14,0.000000
...,...
log_trailing_rv_252d_variance,0.000000
forward_num_returns,0.000000
implied_variance,0.000000
log_implied_variance,0.000000



Numeric summary:


,count,mean,std,min,25%,50%,75%,max
trade_date,18099.0,2.022040e+07,23386.664163,2.018062e+07,2.020062e+07,2.022062e+07,2.024062e+07,2.026062e+07
tenor,18099.0,2.100000e+01,7.746181,9.000000e+00,1.500000e+01,2.100000e+01,2.700000e+01,3.300000e+01
spx_close,18099.0,4.403070e+03,1300.634213,2.237400e+03,3.258440e+03,4.199120e+03,5.344160e+03,7.609780e+03
spx_log_return,18099.0,4.884858e-04,0.012306,-1.276521e-01,-4.425587e-03,9.027860e-04,6.441868e-03,9.089490e-02
spx_rsi_14,18099.0,5.568864e+01,11.523337,1.765821e+01,4.743810e+01,5.741491e+01,6.444746e+01,8.290031e+01
...,...,...,...,...,...,...,...,...
forward_num_returns,18099.0,1.422609e+01,5.385815,0.000000e+00,1.000000e+01,1.400000e+01,1.900000e+01,2.400000e+01
log_forward_realized_variance,18090.0,-3.978484e+00,1.058721,-1.603308e+01,-4.656307e+00,-4.054243e+00,-3.339190e+00,6.066532e-01
implied_variance,18099.0,4.466358e-02,0.054276,7.376624e-03,2.103699e-02,3.099163e-02,4.990936e-02,1.146081e+00
log_implied_variance,18099.0,-3.379605e+00,0.652099,-4.909439e+00,-3.861473e+00,-3.474038e+00,-2.997547e+00,1.363481e-01


In [4]:
# ============================================================
# Cell 4: Quick schema detection
# ============================================================

df = forecast_feature_target_df.copy()

date_candidates = [
    "trade_dt",
    "trade_date",
    "date",
    "asof_dt",
]

tenor_candidates = [
    "tenor",
    "target_tenor",
    "target_days",
    "forecast_horizon",
    "horizon",
]

target_candidates = [
    "forward_realized_variance",
    "forward_realized_var",
    "target_forward_realized_variance",
    "future_realized_variance",
    "realized_variance_forward",
    "fwd_realized_variance",
    "target_variance",
]

trailing_candidates = [
    "trailing_realized_variance",
    "trailing_realized_var",
    "realized_variance",
    "rv_trailing",
    "trailing_variance",
]

implied_candidates = [
    "implied_variance",
    "vix_implied_variance",
    "synthetic_implied_variance",
]

def first_existing(candidates, columns):
    for c in candidates:
        if c in columns:
            return c
    return None

date_col = first_existing(date_candidates, df.columns)
tenor_col = first_existing(tenor_candidates, df.columns)
target_col = first_existing(target_candidates, df.columns)
trailing_col = first_existing(trailing_candidates, df.columns)
implied_col = first_existing(implied_candidates, df.columns)

print("Detected columns:")
print("date_col:    ", date_col)
print("tenor_col:   ", tenor_col)
print("target_col:  ", target_col)
print("trailing_col:", trailing_col)
print("implied_col: ", implied_col)

print("\nColumns containing 'forward', 'target', 'realized', 'variance', or 'forecast':")
interesting = [
    c for c in df.columns
    if any(token in c.lower() for token in ["forward", "target", "realized", "variance", "forecast", "ewma", "rv"])
]

for c in interesting:
    print(c)

Detected columns:
date_col:     trade_date
tenor_col:    tenor
target_col:   forward_realized_variance
trailing_col: trailing_realized_variance
implied_col:  implied_variance

Columns containing 'forward', 'target', 'realized', 'variance', or 'forecast':
trailing_realized_variance
trailing_realized_vol
vrp_trailing_variance_ratio
vrp_trailing_log_variance_ratio
trailing_rv_5d_variance
trailing_rv_5d_vol
trailing_rv_5d_num_returns
log_trailing_rv_5d_variance
trailing_rv_9d_variance
trailing_rv_9d_vol
trailing_rv_9d_num_returns
log_trailing_rv_9d_variance
trailing_rv_12d_variance
trailing_rv_12d_vol
trailing_rv_12d_num_returns
log_trailing_rv_12d_variance
trailing_rv_15d_variance
trailing_rv_15d_vol
trailing_rv_15d_num_returns
log_trailing_rv_15d_variance
trailing_rv_18d_variance
trailing_rv_18d_vol
trailing_rv_18d_num_returns
log_trailing_rv_18d_variance
trailing_rv_21d_variance
trailing_rv_21d_vol
trailing_rv_21d_num_returns
log_trailing_rv_21d_variance
trailing_rv_24d_variance
trailin

In [5]:
# ============================================================
# Cell 5: Inspect existing simple forecast panel
# ============================================================

current_simple_forecast_df = pd.read_parquet(
    CURRENT_SIMPLE_FORECAST_PARQUET
).copy()

print("Rows:", len(current_simple_forecast_df))
print("Columns:", len(current_simple_forecast_df.columns))

print("\nColumn names and dtypes:")
for col in current_simple_forecast_df.columns:
    print(f"{repr(col):60s} {current_simple_forecast_df[col].dtype}")

print("\nFirst 5 rows:")
display(current_simple_forecast_df.head())

print("\nColumns containing forecast / variance / vol:")
for col in current_simple_forecast_df.columns:
    if any(token in col.lower() for token in ["forecast", "variance", "vol"]):
        print(col)

Rows: 18099
Columns: 130

Column names and dtypes:
'trade_date'                                                 int64
'tenor'                                                      int64
'spx_close'                                                  float64
'spx_log_return'                                             float64
'spx_rsi_14'                                                 float64
'vix_style_vol'                                              float64
'trailing_realized_variance'                                 float64
'trailing_realized_vol'                                      float64
'vrp_trailing_variance_ratio'                                float64
'vrp_trailing_log_variance_ratio'                            float64
'vrp_trailing_vol_ratio'                                     float64
'primary_vrp_signal'                                         float64
'trailing_rv_5d_variance'                                    float64
'trailing_rv_5d_vol'                                    

,trade_date,tenor,spx_close,spx_log_return,spx_rsi_14,vix_style_vol,trailing_realized_variance,trailing_realized_vol,vrp_trailing_variance_ratio,vrp_trailing_log_variance_ratio,...,ewma_10_scheduled_bias_adjusted_forecast_variance,ewma_10_scheduled_bias_adjusted_forecast_vol,ewma_10_scheduled_bias_adjusted_vrp_signal,ewma_10_scheduled_bias_adjusted_vrp_vol_ratio,simple_forecast_model,simple_forecast_variance,simple_forecast_vol,log_simple_forecast_variance,simple_forecast_vrp_signal,simple_forecast_vrp_vol_ratio
0,20180625,9,2717.07,-0.01382,41.761584,17.348587,0.010491,10.242504,2.868903,1.053930,...,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,NaN
1,20180625,12,2717.07,-0.01382,41.761584,17.490509,0.008085,8.991663,3.783771,1.330721,...,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,NaN
2,20180625,15,2717.07,-0.01382,41.761584,17.490890,0.006966,8.346020,4.392027,1.479791,...,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,NaN
3,20180625,18,2717.07,-0.01382,41.761584,17.491144,0.006002,7.747385,5.097135,1.628679,...,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,NaN
4,20180625,21,2717.07,-0.01382,41.761584,17.426522,0.006427,8.016921,4.725048,1.552878,...,NaN,NaN,NaN,NaN,ewma_10_scheduled_raw,NaN,NaN,NaN,NaN,NaN



Columns containing forecast / variance / vol:
vix_style_vol
trailing_realized_variance
trailing_realized_vol
vrp_trailing_variance_ratio
vrp_trailing_log_variance_ratio
vrp_trailing_vol_ratio
trailing_rv_5d_variance
trailing_rv_5d_vol
log_trailing_rv_5d_variance
trailing_rv_9d_variance
trailing_rv_9d_vol
log_trailing_rv_9d_variance
trailing_rv_12d_variance
trailing_rv_12d_vol
log_trailing_rv_12d_variance
trailing_rv_15d_variance
trailing_rv_15d_vol
log_trailing_rv_15d_variance
trailing_rv_18d_variance
trailing_rv_18d_vol
log_trailing_rv_18d_variance
trailing_rv_21d_variance
trailing_rv_21d_vol
log_trailing_rv_21d_variance
trailing_rv_24d_variance
trailing_rv_24d_vol
log_trailing_rv_24d_variance
trailing_rv_27d_variance
trailing_rv_27d_vol
log_trailing_rv_27d_variance
trailing_rv_30d_variance
trailing_rv_30d_vol
log_trailing_rv_30d_variance
trailing_rv_33d_variance
trailing_rv_33d_vol
log_trailing_rv_33d_variance
trailing_rv_42d_variance
trailing_rv_42d_vol
log_trailing_rv_42d_variance

In [6]:
# ============================================================
# Cell 6: Build normalized forecast-modeling panel
# ============================================================

import numpy as np
import pandas as pd

forecast_feature_target_df = pd.read_parquet(
    FORECAST_FEATURE_TARGET_PARQUET
).copy()

vov_df = pd.read_parquet(
    VOL_OF_VOL_FEATURES_PARQUET
).copy()

current_simple_forecast_df = pd.read_parquet(
    CURRENT_SIMPLE_FORECAST_PARQUET
).copy()

# ----------------------------
# Normalize main panel
# ----------------------------

model_df = forecast_feature_target_df.copy()

model_df["trade_dt"] = pd.to_datetime(
    model_df["trade_date"].astype(str),
    format="%Y%m%d",
    errors="coerce",
)

model_df["tenor"] = model_df["tenor"].astype(int)

# Keep rows with usable target.
model_df = model_df.dropna(
    subset=[
        "trade_dt",
        "tenor",
        "forward_realized_variance",
        "log_forward_realized_variance",
        "trailing_realized_variance",
        "log_trailing_realized_variance",
        "implied_variance",
        "log_implied_variance",
    ]
).copy()

# ----------------------------
# Merge vol-of-vol / shape features
# ----------------------------

vov_df["trade_dt"] = pd.to_datetime(vov_df["trade_dt"])
vov_df["tenor"] = vov_df["tenor"].astype(int)

vov_cols_to_merge = [
    "trade_dt",
    "tenor",
    "iv_change_1d",
    "iv_change_3d",
    "iv_change_5d",
    "iv_change_10d",
    "iv_change_21d",
    "iv_abs_change_1d",
    "iv_abs_change_3d",
    "iv_abs_change_5d",
    "iv_change_1d_std_10d",
    "iv_change_1d_std_21d",
    "iv_change_1d_std_42d",
    "iv_change_1d_std_63d",
    "iv_abs_change_1d_mean_10d",
    "iv_abs_change_1d_mean_21d",
    "iv_abs_change_1d_mean_42d",
    "iv_abs_change_1d_mean_63d",
    "iv_abs_change_1d_z_1y",
    "iv_abs_change_3d_z_1y",
    "iv_abs_change_5d_z_1y",
    "iv_change_1d_std_21d_z_1y",
    "iv_change_1d_std_42d_z_1y",
    "iv_abs_change_1d_mean_21d_z_1y",
    "iv_abs_change_1d_mean_42d_z_1y",
    "iv_slope_33_minus_9",
    "iv_slope_30_minus_12",
    "iv_slope_24_minus_9",
    "iv_curve_21_vs_9_33",
    "iv_slope_33_minus_9_z_1y",
    "iv_slope_30_minus_12_z_1y",
    "iv_slope_24_minus_9_z_1y",
    "iv_curve_21_vs_9_33_z_1y",
]

vov_cols_to_merge = [c for c in vov_cols_to_merge if c in vov_df.columns]

model_df = model_df.merge(
    vov_df[vov_cols_to_merge],
    on=["trade_dt", "tenor"],
    how="left",
)

# ----------------------------
# Merge existing simple forecast if possible
# ----------------------------

simple_df = current_simple_forecast_df.copy()

if "trade_dt" not in simple_df.columns:
    if "trade_date" in simple_df.columns:
        simple_df["trade_dt"] = pd.to_datetime(
            simple_df["trade_date"].astype(str),
            format="%Y%m%d",
            errors="coerce",
        )
    else:
        simple_df["trade_dt"] = pd.NaT

if "tenor" not in simple_df.columns:
    if "target_tenor" in simple_df.columns:
        simple_df["tenor"] = simple_df["target_tenor"]
    elif "target_days" in simple_df.columns:
        simple_df["tenor"] = simple_df["target_days"]

simple_df["tenor"] = simple_df["tenor"].astype(int)

simple_var_candidates = [
    "simple_forecast_variance",
    "forecast_variance",
    "forecasted_variance",
    "hybrid_forecast_variance",
    "scheduled_forecast_variance",
    "bias_adjusted_forecast_variance",
    "ewma_forecast_variance",
]

simple_var_col = None

for col in simple_var_candidates:
    if col in simple_df.columns:
        simple_var_col = col
        break

if simple_var_col is None:
    print("Could not detect simple forecast variance column.")
    print("Available forecast-like columns:")
    for col in simple_df.columns:
        if any(token in col.lower() for token in ["forecast", "variance", "vol"]):
            print(col)
else:
    print("Using existing simple forecast variance column:", simple_var_col)

    simple_merge_df = simple_df[
        [
            "trade_dt",
            "tenor",
            simple_var_col,
        ]
    ].copy()

    simple_merge_df = simple_merge_df.rename(
        columns={simple_var_col: "existing_simple_forecast_variance"}
    )

    simple_merge_df["existing_simple_forecast_variance"] = pd.to_numeric(
        simple_merge_df["existing_simple_forecast_variance"],
        errors="coerce",
    )

    simple_merge_df["log_existing_simple_forecast_variance"] = np.log(
        simple_merge_df["existing_simple_forecast_variance"].clip(lower=1e-8)
    )

    model_df = model_df.merge(
        simple_merge_df,
        on=["trade_dt", "tenor"],
        how="left",
    )

# ----------------------------
# Final cleanup
# ----------------------------

model_df = model_df.sort_values(["trade_dt", "tenor"]).reset_index(drop=True)

# Core target naming.
model_df["target_log_variance"] = model_df["log_forward_realized_variance"]
model_df["target_variance"] = model_df["forward_realized_variance"]
model_df["target_vol_pct"] = model_df["forward_realized_vol"]

# Baseline predictions.
model_df["pred_logvar_trailing_same_tenor"] = model_df["log_trailing_realized_variance"]
model_df["pred_var_trailing_same_tenor"] = model_df["trailing_realized_variance"]

model_df["pred_logvar_implied"] = model_df["log_implied_variance"]
model_df["pred_var_implied"] = model_df["implied_variance"]

if "existing_simple_forecast_variance" in model_df.columns:
    model_df["pred_var_existing_simple"] = model_df["existing_simple_forecast_variance"]
    model_df["pred_logvar_existing_simple"] = model_df["log_existing_simple_forecast_variance"]

FORECAST_MODELING_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_v0_1.parquet"
)

model_df.to_parquet(FORECAST_MODELING_PANEL_PARQUET, index=False)

print("Saved modeling panel:")
print(FORECAST_MODELING_PANEL_PARQUET)

print("\nRows:", len(model_df))
print("Dates:", model_df["trade_dt"].min(), "to", model_df["trade_dt"].max())
print("Tenors:", sorted(model_df["tenor"].unique()))

print("\nColumns:", len(model_df.columns))

print("\nKey missingness:")
key_cols = [
    "target_log_variance",
    "pred_logvar_trailing_same_tenor",
    "pred_logvar_implied",
    "pred_logvar_existing_simple" if "pred_logvar_existing_simple" in model_df.columns else None,
    "iv_change_1d_std_21d_z_1y",
    "iv_slope_33_minus_9_z_1y",
]

key_cols = [c for c in key_cols if c is not None and c in model_df.columns]

display(
    model_df[key_cols]
    .isna()
    .mean()
    .to_frame("missing_pct")
)

display(model_df.head())

Using existing simple forecast variance column: simple_forecast_variance
Saved modeling panel:
C:\Users\patri\vrp_project\data\processed\forecast_vol_modeling_panel_v0_1.parquet

Rows: 18090
Dates: 2018-06-25 00:00:00 to 2026-06-24 00:00:00
Tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]

Columns: 118

Key missingness:


,missing_pct
target_log_variance,0.000000
pred_logvar_trailing_same_tenor,0.000000
pred_logvar_implied,0.000000
pred_logvar_existing_simple,0.004478
iv_change_1d_std_21d_z_1y,0.034328
iv_slope_33_minus_9_z_1y,0.029353


,trade_date,tenor,spx_close,spx_log_return,spx_rsi_14,vix_style_vol,trailing_realized_variance,trailing_realized_vol,vrp_trailing_variance_ratio,vrp_trailing_log_variance_ratio,...,log_existing_simple_forecast_variance,target_log_variance,target_variance,target_vol_pct,pred_logvar_trailing_same_tenor,pred_var_trailing_same_tenor,pred_logvar_implied,pred_var_implied,pred_var_existing_simple,pred_logvar_existing_simple
0,20180625,9,2717.07,-0.01382,41.761584,17.348587,0.010491,10.242504,2.868903,1.053930,...,NaN,-5.088807,0.006165,7.851987,-4.557248,0.010491,-3.503318,0.030097,NaN,NaN
1,20180625,12,2717.07,-0.01382,41.761584,17.490509,0.008085,8.991663,3.783771,1.330721,...,NaN,-4.706681,0.009035,9.505109,-4.817745,0.008085,-3.487024,0.030592,NaN,NaN
2,20180625,15,2717.07,-0.01382,41.761584,17.490890,0.006966,8.346020,4.392027,1.479791,...,NaN,-4.667262,0.009398,9.694309,-4.966771,0.006966,-3.486980,0.030593,NaN,NaN
3,20180625,18,2717.07,-0.01382,41.761584,17.491144,0.006002,7.747385,5.097135,1.628679,...,NaN,-4.563861,0.010422,10.208696,-5.115630,0.006002,-3.486951,0.030594,NaN,NaN
4,20180625,21,2717.07,-0.01382,41.761584,17.426522,0.006427,8.016921,4.725048,1.552878,...,NaN,-4.715955,0.008951,9.461139,-5.047231,0.006427,-3.494354,0.030368,NaN,NaN


In [10]:
# ============================================================
# Cell 7: Define forecast model feature sets
# ============================================================

# Realized-vol / HAR-style features.
rv_feature_cols = [
    "log_trailing_realized_variance",
    "log_trailing_rv_5d_variance",
    "log_trailing_rv_9d_variance",
    "log_trailing_rv_12d_variance",
    "log_trailing_rv_15d_variance",
    "log_trailing_rv_21d_variance",
    "log_trailing_rv_30d_variance",
    "log_trailing_rv_42d_variance",
    "log_trailing_rv_63d_variance",
    "log_trailing_rv_126d_variance",
    "log_trailing_rv_252d_variance",
]

# Implied-vol / market features.
iv_feature_cols = [
    "log_implied_variance",
    "vix_style_vol",
    "spx_rsi_14",
    "iv_slope_33_minus_9",
    "iv_slope_30_minus_12",
    "iv_slope_24_minus_9",
    "iv_curve_21_vs_9_33",
    "iv_slope_33_minus_9_z_1y",
    "iv_slope_30_minus_12_z_1y",
    "iv_slope_24_minus_9_z_1y",
    "iv_curve_21_vs_9_33_z_1y",
]

# Vol-of-vol features.
vov_feature_cols = [
    "iv_change_1d",
    "iv_change_3d",
    "iv_change_5d",
    "iv_abs_change_1d",
    "iv_abs_change_3d",
    "iv_abs_change_5d",
    "iv_change_1d_std_21d",
    "iv_change_1d_std_42d",
    "iv_abs_change_1d_mean_21d",
    "iv_abs_change_1d_mean_42d",
    "iv_abs_change_1d_z_1y",
    "iv_abs_change_3d_z_1y",
    "iv_abs_change_5d_z_1y",
    "iv_change_1d_std_21d_z_1y",
    "iv_change_1d_std_42d_z_1y",
    "iv_abs_change_1d_mean_21d_z_1y",
    "iv_abs_change_1d_mean_42d_z_1y",
]

# Keep only columns that exist.
rv_feature_cols = [c for c in rv_feature_cols if c in model_df.columns]
iv_feature_cols = [c for c in iv_feature_cols if c in model_df.columns]
vov_feature_cols = [c for c in vov_feature_cols if c in model_df.columns]

MODEL_FEATURE_SETS = {
    "har_rv": rv_feature_cols,
    "har_rv_iv": rv_feature_cols + iv_feature_cols,
    "har_rv_iv_vov": rv_feature_cols + iv_feature_cols + vov_feature_cols,
}

for name, cols in MODEL_FEATURE_SETS.items():
    print(f"\n{name}: {len(cols)} features")
    for c in cols:
        print("  ", c)

NameError: name 'model_df' is not defined

In [8]:
# ============================================================
# Replacement Cell 8: Monthly expanding walk-forward ridge forecasts
# Pure numpy version — no sklearn dependency
# ============================================================

import numpy as np
import pandas as pd

modeling_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_complete_targets_v0_1.parquet"
).copy()

modeling_df["trade_dt"] = pd.to_datetime(modeling_df["trade_dt"])
modeling_df = modeling_df.sort_values(["trade_dt", "tenor"]).reset_index(drop=True)

target_col = "target_log_variance"

MIN_TRAIN_DATES = 504   # roughly two years
REFIT_FREQ = "MS"       # month-start refit
RIDGE_ALPHA = 5.0


def fit_predict_standardized_ridge(
    X_train,
    y_train,
    X_test,
    alpha=5.0,
    min_std=1e-12,
):
    """
    Standardized ridge regression with unpenalized intercept.
    Pure numpy implementation.

    Model:
        y = intercept + X_standardized @ beta

    Ridge penalty applies only to beta, not intercept.
    """

    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)

    x_mean = np.nanmean(X_train, axis=0)
    x_std = np.nanstd(X_train, axis=0)

    usable = np.isfinite(x_mean) & np.isfinite(x_std) & (x_std > min_std)

    if usable.sum() == 0:
        return np.full(X_test.shape[0], np.nan)

    X_train_u = X_train[:, usable]
    X_test_u = X_test[:, usable]

    x_mean_u = x_mean[usable]
    x_std_u = x_std[usable]

    X_train_z = (X_train_u - x_mean_u) / x_std_u
    X_test_z = (X_test_u - x_mean_u) / x_std_u

    # Add intercept column.
    X_design = np.column_stack([
        np.ones(len(X_train_z)),
        X_train_z,
    ])

    X_test_design = np.column_stack([
        np.ones(len(X_test_z)),
        X_test_z,
    ])

    # Ridge penalty, no penalty on intercept.
    penalty = np.eye(X_design.shape[1]) * alpha
    penalty[0, 0] = 0.0

    lhs = X_design.T @ X_design + penalty
    rhs = X_design.T @ y_train

    try:
        beta = np.linalg.solve(lhs, rhs)
    except np.linalg.LinAlgError:
        beta = np.linalg.lstsq(lhs, rhs, rcond=None)[0]

    return X_test_design @ beta


unique_dates = np.array(sorted(modeling_df["trade_dt"].unique()))
date_counts = pd.Series(unique_dates).sort_values().reset_index(drop=True)

first_test_date = date_counts.iloc[min(MIN_TRAIN_DATES, len(date_counts) - 1)]

print("First test date:", first_test_date)

refit_dates = pd.date_range(
    start=pd.Timestamp(first_test_date).replace(day=1),
    end=modeling_df["trade_dt"].max(),
    freq=REFIT_FREQ,
)

refit_dates = [d for d in refit_dates if d >= first_test_date]

print("Refit dates:", len(refit_dates))

prediction_blocks = []

for i, refit_date in enumerate(refit_dates):
    if i == len(refit_dates) - 1:
        next_refit_date = modeling_df["trade_dt"].max() + pd.Timedelta(days=1)
    else:
        next_refit_date = refit_dates[i + 1]

    test_mask = (
        (modeling_df["trade_dt"] >= refit_date)
        & (modeling_df["trade_dt"] < next_refit_date)
    )

    test_block = modeling_df[test_mask].copy()

    if test_block.empty:
        continue

    train_block = modeling_df[
        modeling_df["trade_dt"] < refit_date
    ].copy()

    train_block = train_block.dropna(subset=[target_col]).copy()

    if train_block["trade_dt"].nunique() < MIN_TRAIN_DATES:
        continue

    pred_block = test_block[
        [
            "trade_date",
            "trade_dt",
            "tenor",
            "target_variance",
            "target_vol_pct",
            "target_log_variance",
            "trailing_realized_variance",
            "trailing_realized_vol",
            "implied_variance",
            "vix_style_vol",
            "spx_close",
            "spx_rsi_14",
        ]
    ].copy()

    # Baselines.
    pred_block["pred_logvar_trailing_same_tenor"] = test_block["pred_logvar_trailing_same_tenor"]
    pred_block["pred_var_trailing_same_tenor"] = test_block["pred_var_trailing_same_tenor"]

    pred_block["pred_logvar_implied"] = test_block["pred_logvar_implied"]
    pred_block["pred_var_implied"] = test_block["pred_var_implied"]

    if "pred_logvar_existing_simple" in test_block.columns:
        pred_block["pred_logvar_existing_simple"] = test_block["pred_logvar_existing_simple"]
        pred_block["pred_var_existing_simple"] = test_block["pred_var_existing_simple"]

    # Model forecasts.
    for model_name, feature_cols in MODEL_FEATURE_SETS.items():
        usable_feature_cols = [c for c in feature_cols if c in train_block.columns]

        pred_col_log = f"pred_logvar_{model_name}"
        pred_col_var = f"pred_var_{model_name}"

        pred_block[pred_col_log] = np.nan
        pred_block[pred_col_var] = np.nan

        if len(usable_feature_cols) == 0:
            continue

        train_needed = usable_feature_cols + [target_col]
        test_needed = usable_feature_cols

        train_xy = train_block.dropna(subset=train_needed).copy()
        test_x = test_block.dropna(subset=test_needed).copy()

        if len(train_xy) < 500 or len(test_x) == 0:
            continue

        X_train = train_xy[usable_feature_cols].values
        y_train = train_xy[target_col].values
        X_test = test_x[usable_feature_cols].values

        y_pred_log = fit_predict_standardized_ridge(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            alpha=RIDGE_ALPHA,
        )

        # Clip log variance to prevent absurd model outputs.
        y_pred_log = np.clip(y_pred_log, -10.0, 1.0)

        pred_block.loc[test_x.index, pred_col_log] = y_pred_log
        pred_block.loc[test_x.index, pred_col_var] = np.exp(y_pred_log)

    prediction_blocks.append(pred_block)

forecast_predictions_df = pd.concat(
    prediction_blocks,
    ignore_index=True,
)

# Convert all variance forecasts to vol pct.
pred_var_cols = [
    c for c in forecast_predictions_df.columns
    if c.startswith("pred_var_")
]

for col in pred_var_cols:
    vol_col = col.replace("pred_var_", "pred_vol_pct_")
    forecast_predictions_df[vol_col] = (
        np.sqrt(forecast_predictions_df[col].clip(lower=1e-10)) * 100.0
    )

forecast_predictions_df.to_parquet(
    FORECAST_MODEL_PREDICTIONS_PARQUET,
    index=False,
)

print("Saved forecast predictions:")
print(FORECAST_MODEL_PREDICTIONS_PARQUET)

print("Rows:", len(forecast_predictions_df))
print("Dates:", forecast_predictions_df["trade_dt"].min(), "to", forecast_predictions_df["trade_dt"].max())

print("\nPrediction columns:")
for col in forecast_predictions_df.columns:
    if col.startswith("pred_"):
        print(col)

display(forecast_predictions_df.tail())

First test date: 2020-06-29 00:00:00
Refit dates: 72
Saved forecast predictions:
C:\Users\patri\vrp_project\data\processed\forecast_vol_model_predictions_v0_1.parquet
Rows: 12260
Dates: 2020-07-01 00:00:00 to 2026-06-17 00:00:00

Prediction columns:
pred_logvar_trailing_same_tenor
pred_var_trailing_same_tenor
pred_logvar_implied
pred_var_implied
pred_logvar_existing_simple
pred_var_existing_simple
pred_logvar_har_rv
pred_var_har_rv
pred_logvar_har_rv_iv
pred_var_har_rv_iv
pred_logvar_har_rv_iv_vov
pred_var_har_rv_iv_vov
pred_vol_pct_trailing_same_tenor
pred_vol_pct_implied
pred_vol_pct_existing_simple
pred_vol_pct_har_rv
pred_vol_pct_har_rv_iv
pred_vol_pct_har_rv_iv_vov


,trade_date,trade_dt,tenor,target_variance,target_vol_pct,target_log_variance,trailing_realized_variance,trailing_realized_vol,implied_variance,vix_style_vol,...,pred_logvar_har_rv_iv,pred_var_har_rv_iv,pred_logvar_har_rv_iv_vov,pred_var_har_rv_iv_vov,pred_vol_pct_trailing_same_tenor,pred_vol_pct_implied,pred_vol_pct_existing_simple,pred_vol_pct_har_rv,pred_vol_pct_har_rv_iv,pred_vol_pct_har_rv_iv_vov
12255,20260612,2026-06-12,12,0.024061,15.511565,-3.727169,0.042795,20.687030,0.029767,17.253141,...,-3.970069,0.018872,-4.001255,0.018293,20.687030,17.253141,12.977627,16.266215,13.737584,13.525039
12256,20260615,2026-06-15,9,0.021182,14.554132,-3.854590,0.035597,18.867091,0.023138,15.211022,...,-4.252548,0.014228,-4.340559,0.013029,18.867091,15.211022,14.363650,15.748135,11.928093,11.414573
12257,20260615,2026-06-15,12,0.015887,12.604367,-4.142254,0.049044,22.145953,0.023006,15.167816,...,-4.253881,0.014209,-4.314098,0.013379,22.145953,15.167816,14.655751,15.976215,11.920142,11.566593
12258,20260616,2026-06-16,9,0.019865,14.094268,-3.918804,0.036915,19.213159,0.024148,15.539618,...,-4.192595,0.015107,-4.313910,0.013381,19.213159,15.539618,14.019805,15.516091,12.291067,11.567684
12259,20260617,2026-06-17,9,0.013806,11.749932,-4.282646,0.042615,20.643319,0.034480,18.568801,...,-3.800086,0.022369,-3.967935,0.018912,20.643319,18.568801,14.214190,16.092084,14.956218,13.752256


In [5]:
# ============================================================
# Cell 9: Evaluate forecast model candidates
# ============================================================

import numpy as np
import pandas as pd

forecast_predictions_df = pd.read_parquet(
    FORECAST_MODEL_PREDICTIONS_PARQUET
).copy()

forecast_predictions_df["trade_dt"] = pd.to_datetime(forecast_predictions_df["trade_dt"])
forecast_predictions_df["year"] = forecast_predictions_df["trade_dt"].dt.year

pred_log_cols = [
    c for c in forecast_predictions_df.columns
    if c.startswith("pred_logvar_")
]

eval_rows = []

for pred_log_col in pred_log_cols:
    model_name = pred_log_col.replace("pred_logvar_", "")
    pred_var_col = f"pred_var_{model_name}"
    pred_vol_col = f"pred_vol_pct_{model_name}"

    temp = forecast_predictions_df.dropna(
        subset=[
            "target_log_variance",
            "target_variance",
            "target_vol_pct",
            pred_log_col,
            pred_var_col,
            pred_vol_col,
        ]
    ).copy()

    if temp.empty:
        continue

    log_err = temp[pred_log_col] - temp["target_log_variance"]
    var_err = temp[pred_var_col] - temp["target_variance"]
    vol_err = temp[pred_vol_col] - temp["target_vol_pct"]

    eval_rows.append({
        "model": model_name,
        "rows": len(temp),
        "start_date": temp["trade_dt"].min(),
        "end_date": temp["trade_dt"].max(),

        "log_rmse": np.sqrt(np.mean(log_err ** 2)),
        "log_mae": np.mean(np.abs(log_err)),
        "log_bias": np.mean(log_err),

        "variance_rmse": np.sqrt(np.mean(var_err ** 2)),
        "variance_mae": np.mean(np.abs(var_err)),
        "variance_bias": np.mean(var_err),

        "vol_rmse_pct": np.sqrt(np.mean(vol_err ** 2)),
        "vol_mae_pct": np.mean(np.abs(vol_err)),
        "vol_bias_pct": np.mean(vol_err),

        "corr_log_prediction_target": temp[pred_log_col].corr(temp["target_log_variance"]),
        "corr_vol_prediction_target": temp[pred_vol_col].corr(temp["target_vol_pct"]),
    })

forecast_model_comparison_df = pd.DataFrame(eval_rows)

forecast_model_comparison_df = forecast_model_comparison_df.sort_values(
    ["log_rmse", "vol_rmse_pct"]
).reset_index(drop=True)

forecast_model_comparison_df.to_csv(
    FORECAST_MODEL_COMPARISON_CSV,
    index=False,
)

print("Saved model comparison:")
print(FORECAST_MODEL_COMPARISON_CSV)

display(forecast_model_comparison_df)

# By-tenor comparison.
tenor_rows = []

for pred_log_col in pred_log_cols:
    model_name = pred_log_col.replace("pred_logvar_", "")
    pred_var_col = f"pred_var_{model_name}"
    pred_vol_col = f"pred_vol_pct_{model_name}"

    for tenor, temp in forecast_predictions_df.groupby("tenor"):
        temp = temp.dropna(
            subset=[
                "target_log_variance",
                "target_vol_pct",
                pred_log_col,
                pred_vol_col,
            ]
        )

        if temp.empty:
            continue

        log_err = temp[pred_log_col] - temp["target_log_variance"]
        vol_err = temp[pred_vol_col] - temp["target_vol_pct"]

        tenor_rows.append({
            "model": model_name,
            "tenor": tenor,
            "rows": len(temp),
            "log_rmse": np.sqrt(np.mean(log_err ** 2)),
            "log_bias": np.mean(log_err),
            "vol_rmse_pct": np.sqrt(np.mean(vol_err ** 2)),
            "vol_bias_pct": np.mean(vol_err),
            "corr_log_prediction_target": temp[pred_log_col].corr(temp["target_log_variance"]),
        })

forecast_model_by_tenor_df = pd.DataFrame(tenor_rows)

FORECAST_MODEL_BY_TENOR_CSV = (
    AUDIT_DIR / "forecast_vol_model_comparison_by_tenor_v0_1.csv"
)

forecast_model_by_tenor_df.to_csv(
    FORECAST_MODEL_BY_TENOR_CSV,
    index=False,
)

print("\nSaved by-tenor comparison:")
print(FORECAST_MODEL_BY_TENOR_CSV)

display(
    forecast_model_by_tenor_df
    .sort_values(["tenor", "log_rmse"])
)

Saved model comparison:
C:\Users\patri\vrp_project\data\audit\forecast_vol_model_comparison_v0_1.csv


,model,rows,start_date,end_date,log_rmse,log_mae,log_bias,variance_rmse,variance_mae,variance_bias,vol_rmse_pct,vol_mae_pct,vol_bias_pct,corr_log_prediction_target,corr_vol_prediction_target
0,har_rv_iv,12260,2020-07-01,2026-06-17,0.715400,0.557820,0.114267,0.034428,0.016251,-0.001487,6.370327,4.337304,0.239952,0.587839,0.547804
1,har_rv_iv_vov,12260,2020-07-01,2026-06-17,0.749615,0.573505,0.042090,0.084043,0.019439,-0.000532,8.104263,4.616721,-0.144069,0.543019,0.388418
2,har_rv,12260,2020-07-01,2026-06-17,0.765510,0.604364,0.041096,0.034286,0.016278,-0.005292,6.542045,4.541975,-0.554804,0.478708,0.474514
3,existing_simple,12260,2020-07-01,2026-06-17,0.782745,0.616213,0.000935,0.036366,0.017279,-0.002986,6.858764,4.688955,-0.395064,0.518946,0.490687
4,trailing_same_tenor,12260,2020-07-01,2026-06-17,0.869052,0.677820,0.010616,0.041225,0.019130,0.000029,7.438907,5.101243,0.055842,0.482898,0.482378
5,implied,12260,2020-07-01,2026-06-17,0.900340,0.738969,0.610317,0.035714,0.021894,0.012925,7.418711,5.892595,4.307434,0.635569,0.601334



Saved by-tenor comparison:
C:\Users\patri\vrp_project\data\audit\forecast_vol_model_comparison_by_tenor_v0_1.csv


,model,tenor,rows,log_rmse,log_bias,vol_rmse_pct,vol_bias_pct,corr_log_prediction_target
36,har_rv_iv,9,1440,0.790661,0.100642,6.652603,-0.129048,0.625950
45,har_rv_iv_vov,9,1440,0.823239,-0.011225,8.163074,-0.869594,0.581202
18,existing_simple,9,1440,0.880198,-0.086434,7.437750,-1.445511,0.520101
27,har_rv,9,1440,0.902649,0.133567,7.358861,-0.296218,0.461382
9,implied,9,1440,0.970304,0.615075,7.471859,3.888830,0.663000
0,trailing_same_tenor,9,1440,1.026564,0.048436,8.335793,0.277187,0.464173
37,har_rv_iv,12,1481,0.753927,0.146297,6.289042,0.341254,0.617239
46,har_rv_iv_vov,12,1481,0.774044,0.051137,7.771790,-0.258199,0.580066
19,existing_simple,12,1481,0.825467,-0.071105,6.861241,-1.125253,0.525006
28,har_rv,12,1481,0.834850,0.148816,6.732400,0.022582,0.483039


In [10]:
# ============================================================
# Cell 10: Diagnose incomplete forward realized-vol targets
# ============================================================

import numpy as np
import pandas as pd

modeling_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_v0_1.parquet"
).copy()

modeling_df["trade_dt"] = pd.to_datetime(modeling_df["trade_dt"])
modeling_df["tenor"] = modeling_df["tenor"].astype(int)

print("Forward num returns by tenor:")
display(
    modeling_df
    .groupby("tenor")
    .agg(
        rows=("trade_dt", "count"),
        min_forward_returns=("forward_num_returns", "min"),
        p01_forward_returns=("forward_num_returns", lambda x: x.quantile(0.01)),
        p05_forward_returns=("forward_num_returns", lambda x: x.quantile(0.05)),
        median_forward_returns=("forward_num_returns", "median"),
        max_forward_returns=("forward_num_returns", "max"),
    )
    .reset_index()
)

# Conservative minimum required returns by tenor.
# Approx business days in a calendar-day window = tenor * 5 / 7.
# Allow one missing day for holidays.
min_required_by_tenor = {
    9: 5,
    12: 7,
    15: 9,
    18: 11,
    21: 14,
    24: 16,
    27: 18,
    30: 20,
    33: 22,
}

modeling_df["min_required_forward_returns"] = (
    modeling_df["tenor"].map(min_required_by_tenor)
)

modeling_df["has_complete_forward_target"] = (
    modeling_df["forward_num_returns"] >= modeling_df["min_required_forward_returns"]
)

print("\nComplete target coverage:")
display(
    modeling_df
    .groupby("tenor")
    .agg(
        rows=("trade_dt", "count"),
        complete_rows=("has_complete_forward_target", "sum"),
        complete_pct=("has_complete_forward_target", "mean"),
        incomplete_rows=("has_complete_forward_target", lambda x: (~x).sum()),
    )
    .reset_index()
)

print("\nWorst incomplete rows:")
display(
    modeling_df
    .loc[~modeling_df["has_complete_forward_target"]]
    .sort_values(["trade_dt", "tenor"])
    [
        [
            "trade_dt",
            "tenor",
            "forward_num_returns",
            "min_required_forward_returns",
            "forward_realized_variance",
            "forward_realized_vol",
            "log_forward_realized_variance",
        ]
    ]
    .tail(50)
)

Forward num returns by tenor:


,tenor,rows,min_forward_returns,p01_forward_returns,p05_forward_returns,median_forward_returns,max_forward_returns
0,9,2010,1,4.0,5.0,6.0,7
1,12,2010,1,6.0,7.0,8.0,9
2,15,2010,1,9.0,9.0,10.0,11
3,18,2010,1,10.0,11.0,12.0,14
4,21,2010,1,13.0,13.0,14.0,15
5,24,2010,1,14.0,15.0,16.0,18
6,27,2010,1,16.0,17.0,18.0,19
7,30,2010,1,17.0,19.0,21.0,22
8,33,2010,1,19.0,21.0,22.0,24



Complete target coverage:


,tenor,rows,complete_rows,complete_pct,incomplete_rows
0,9,2010,1928,0.959204,82
1,12,2010,1985,0.987562,25
2,15,2010,1994,0.992040,16
3,18,2010,1955,0.972637,55
4,21,2010,1876,0.933333,134
5,24,2010,1518,0.755224,492
6,27,2010,1801,0.896020,209
7,30,2010,1666,0.828856,344
8,33,2010,1735,0.863184,275



Worst incomplete rows:


,trade_dt,tenor,forward_num_returns,min_required_forward_returns,forward_realized_variance,forward_realized_vol,log_forward_realized_variance
18039,2026-06-16,18,6,11,9.932419e-03,9.966152,-4.611951
18040,2026-06-16,21,6,14,8.513502e-03,9.226864,-4.766102
18041,2026-06-16,24,6,16,7.449314e-03,8.630941,-4.899633
18042,2026-06-16,27,6,18,6.621613e-03,8.137329,-5.017416
18043,2026-06-16,30,6,20,5.959451e-03,7.719748,-5.122777
18044,2026-06-16,33,6,22,5.417683e-03,7.360491,-5.218087
18046,2026-06-17,12,5,7,1.035457e-02,10.175739,-4.570328
18047,2026-06-17,15,5,9,8.283654e-03,9.101458,-4.793471
18048,2026-06-17,18,5,11,6.903045e-03,8.308456,-4.975793
18049,2026-06-17,21,5,14,5.916895e-03,7.692136,-5.129943


In [11]:
# ============================================================
# Cell 11: Save complete-target forecast modeling panel
# ============================================================

FORECAST_MODELING_PANEL_COMPLETE_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_complete_targets_v0_1.parquet"
)

modeling_complete_df = modeling_df[
    modeling_df["has_complete_forward_target"]
].copy()

modeling_complete_df = modeling_complete_df.dropna(
    subset=[
        "target_log_variance",
        "target_variance",
        "target_vol_pct",
    ]
).copy()

modeling_complete_df.to_parquet(
    FORECAST_MODELING_PANEL_COMPLETE_PARQUET,
    index=False,
)

print("Saved complete-target modeling panel:")
print(FORECAST_MODELING_PANEL_COMPLETE_PARQUET)

print("\nRows before:", len(modeling_df))
print("Rows after:", len(modeling_complete_df))
print("Rows removed:", len(modeling_df) - len(modeling_complete_df))

print("\nDates after filtering:")
print(modeling_complete_df["trade_dt"].min(), "to", modeling_complete_df["trade_dt"].max())

display(
    modeling_complete_df
    .groupby("tenor")
    .agg(
        rows=("trade_dt", "count"),
        start=("trade_dt", "min"),
        end=("trade_dt", "max"),
        avg_forward_returns=("forward_num_returns", "mean"),
        min_forward_returns=("forward_num_returns", "min"),
    )
    .reset_index()
)

Saved complete-target modeling panel:
C:\Users\patri\vrp_project\data\processed\forecast_vol_modeling_panel_complete_targets_v0_1.parquet

Rows before: 18090
Rows after: 16458
Rows removed: 1632

Dates after filtering:
2018-06-25 00:00:00 to 2026-06-17 00:00:00


,tenor,rows,start,end,avg_forward_returns,min_forward_returns
0,9,1928,2018-06-25,2026-06-17,6.258817,5
1,12,1985,2018-06-25,2026-06-15,7.893703,7
2,15,1994,2018-06-25,2026-06-11,10.412738,9
3,18,1955,2018-06-25,2026-06-09,12.156522,11
4,21,1876,2018-06-25,2026-06-04,14.532516,14
5,24,1518,2018-06-25,2026-06-02,16.830698,16
6,27,1801,2018-06-25,2026-05-29,18.441421,18
7,30,1666,2018-06-25,2026-05-27,20.965786,20
8,33,1735,2018-06-25,2026-05-18,22.544669,22


In [6]:
# ============================================================
# Cell 12: Save champion forecast model config
# ============================================================

import json
import pandas as pd

CHAMPION_FORECAST_MODEL_NAME = "har_rv_iv"

CHAMPION_FORECAST_CONFIG_JSON = (
    AUDIT_DIR / "forecast_vol_champion_model_config_v0_1.json"
)

champion_forecast_config = {
    "strategy_version": "forecast_vol_champion_model_v0_1",

    "champion_model": CHAMPION_FORECAST_MODEL_NAME,

    "target": {
        "model_target": "log_forward_realized_variance",
        "economic_target": "forward realized variance over matching tenor",
    },

    "model_type": {
        "type": "monthly expanding walk-forward ridge regression",
        "implementation": "pure numpy standardized ridge",
        "ridge_alpha": 5.0,
        "min_train_dates": 504,
        "refit_frequency": "monthly",
    },

    "features": {
        "realized_vol_features": rv_feature_cols,
        "implied_vol_features": iv_feature_cols,
        "excluded": {
            "vol_of_vol_features": "excluded from champion because har_rv_iv_vov underperformed",
        },
    },

    "evaluation_summary": {
        "log_rmse": 0.715400,
        "vol_rmse_pct": 6.370327,
        "log_bias": 0.114267,
        "vol_bias_pct": 0.239952,
        "corr_log_prediction_target": 0.587839,
        "corr_vol_prediction_target": 0.547804,
        "comparison": "improves materially over existing_simple and trailing_same_tenor",
    },

    "intended_use": {
        "primary_use": "forecast-vol denominator for forecast VRP",
        "signal_formula": "forecast_vrp_signal = log_implied_variance - champion_pred_log_variance",
        "status": "research champion; not yet promoted to production signal",
    },
}

with open(CHAMPION_FORECAST_CONFIG_JSON, "w") as f:
    json.dump(champion_forecast_config, f, indent=4)

print("Saved champion forecast model config:")
print(CHAMPION_FORECAST_CONFIG_JSON)

NameError: name 'rv_feature_cols' is not defined

In [13]:
# ============================================================
# Cell 13: Build champion forecast panel
# ============================================================

import numpy as np
import pandas as pd

forecast_predictions_df = pd.read_parquet(
    FORECAST_MODEL_PREDICTIONS_PARQUET
).copy()

forecast_predictions_df["trade_dt"] = pd.to_datetime(forecast_predictions_df["trade_dt"])

champion_df = forecast_predictions_df.copy()

champion_df["champion_model"] = CHAMPION_FORECAST_MODEL_NAME

champion_df["champion_forecast_log_variance"] = champion_df[
    f"pred_logvar_{CHAMPION_FORECAST_MODEL_NAME}"
]

champion_df["champion_forecast_variance"] = champion_df[
    f"pred_var_{CHAMPION_FORECAST_MODEL_NAME}"
]

champion_df["champion_forecast_vol"] = (
    np.sqrt(champion_df["champion_forecast_variance"].clip(lower=1e-10)) * 100.0
)

champion_df["champion_forecast_vrp_signal"] = (
    champion_df["pred_logvar_implied"]
    - champion_df["champion_forecast_log_variance"]
)

champion_df["champion_forecast_vrp_variance_ratio"] = (
    champion_df["implied_variance"]
    / champion_df["champion_forecast_variance"].clip(lower=1e-10)
)

champion_df["champion_forecast_vrp_vol_ratio"] = (
    champion_df["vix_style_vol"]
    / champion_df["champion_forecast_vol"].clip(lower=1e-10)
)

CHAMPION_FORECAST_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_champion_panel_v0_1.parquet"
)

CHAMPION_FORECAST_PANEL_CSV = (
    AUDIT_DIR / "forecast_vol_champion_panel_sample_v0_1.csv"
)

champion_df.to_parquet(CHAMPION_FORECAST_PANEL_PARQUET, index=False)
champion_df.head(1000).to_csv(CHAMPION_FORECAST_PANEL_CSV, index=False)

print("Saved champion forecast panel:")
print(CHAMPION_FORECAST_PANEL_PARQUET)
print(CHAMPION_FORECAST_PANEL_CSV)

print("\nRows:", len(champion_df))
print("Dates:", champion_df["trade_dt"].min(), "to", champion_df["trade_dt"].max())

display(
    champion_df[
        [
            "trade_dt",
            "tenor",
            "target_vol_pct",
            "trailing_realized_vol",
            "vix_style_vol",
            "champion_forecast_vol",
            "champion_forecast_vrp_signal",
            "champion_forecast_vrp_vol_ratio",
        ]
    ].tail(20)
)

Saved champion forecast panel:
C:\Users\patri\vrp_project\data\processed\forecast_vol_champion_panel_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\forecast_vol_champion_panel_sample_v0_1.csv

Rows: 12260
Dates: 2020-07-01 00:00:00 to 2026-06-17 00:00:00


,trade_dt,tenor,target_vol_pct,trailing_realized_vol,vix_style_vol,champion_forecast_vol,champion_forecast_vrp_signal,champion_forecast_vrp_vol_ratio
12240,2026-06-05,18,16.843361,14.588992,20.748302,17.433261,0.348169,1.190156
12241,2026-06-08,9,20.643319,18.089011,19.136591,15.718719,0.393500,1.217440
12242,2026-06-08,12,18.841655,16.026641,18.920381,15.528669,0.395103,1.218416
12243,2026-06-08,15,18.392573,14.646265,18.768702,15.394473,0.396364,1.219184
12244,2026-06-08,18,16.795944,13.474683,18.666899,15.300000,0.397798,1.220059
12245,2026-06-09,9,21.694354,18.163459,21.327388,18.179182,0.319430,1.173176
12246,2026-06-09,12,18.787862,15.775473,20.839859,17.762732,0.319530,1.173235
12247,2026-06-09,15,18.354911,14.701443,20.541788,17.511506,0.319206,1.173045
12248,2026-06-09,18,16.755724,13.420520,20.333162,17.326304,0.320055,1.173543
12249,2026-06-10,9,19.038582,20.863863,25.510495,22.925986,0.213638,1.112733


In [14]:
# ============================================================
# Cell 14: Compare champion forecast VRP to current forecast VRP
# ============================================================

import numpy as np
import pandas as pd

current_forecast_vrp_df = pd.read_parquet(
    CURRENT_FORECAST_VRP_PANEL_PARQUET
).copy()

champion_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "forecast_vol_champion_panel_v0_1.parquet"
).copy()

current_forecast_vrp_df["trade_dt"] = pd.to_datetime(
    current_forecast_vrp_df["trade_date"].astype(str),
    format="%Y%m%d",
    errors="coerce",
)

champion_df["trade_dt"] = pd.to_datetime(champion_df["trade_dt"])

if "tenor" not in current_forecast_vrp_df.columns:
    if "target_tenor" in current_forecast_vrp_df.columns:
        current_forecast_vrp_df["tenor"] = current_forecast_vrp_df["target_tenor"]
    elif "target_days" in current_forecast_vrp_df.columns:
        current_forecast_vrp_df["tenor"] = current_forecast_vrp_df["target_days"]
    else:
        raise ValueError("Could not find tenor column in current_forecast_vrp_df.")

current_forecast_vrp_df["tenor"] = current_forecast_vrp_df["tenor"].astype(int)
champion_df["tenor"] = champion_df["tenor"].astype(int)

print("Current forecast VRP columns:")
for col in current_forecast_vrp_df.columns:
    if any(token in col.lower() for token in ["forecast", "vrp", "variance", "vol"]):
        print(col)

# Detect current forecast VRP signal column.
current_signal_candidates = [
    "simple_forecast_vrp_signal",
    "forecast_vrp_signal",
    "forecast_vrp_log_variance_ratio",
    "vrp_forecast_log_variance_ratio",
]

current_signal_col = None

for col in current_signal_candidates:
    if col in current_forecast_vrp_df.columns:
        current_signal_col = col
        break

if current_signal_col is None:
    raise ValueError("Could not detect current forecast VRP signal column.")

print("\nUsing current forecast VRP signal column:", current_signal_col)

compare_vrp_df = champion_df.merge(
    current_forecast_vrp_df[
        [
            "trade_dt",
            "tenor",
            current_signal_col,
        ]
    ],
    on=["trade_dt", "tenor"],
    how="left",
)

compare_vrp_df = compare_vrp_df.rename(
    columns={current_signal_col: "current_forecast_vrp_signal"}
)

compare_vrp_df["champion_minus_current_vrp_signal"] = (
    compare_vrp_df["champion_forecast_vrp_signal"]
    - compare_vrp_df["current_forecast_vrp_signal"]
)

FORECAST_VRP_CHAMPION_COMPARISON_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vrp_champion_comparison_v0_1.parquet"
)

FORECAST_VRP_CHAMPION_COMPARISON_CSV = (
    AUDIT_DIR / "forecast_vrp_champion_comparison_summary_v0_1.csv"
)

compare_vrp_df.to_parquet(
    FORECAST_VRP_CHAMPION_COMPARISON_PARQUET,
    index=False,
)

summary_df = (
    compare_vrp_df
    .groupby("tenor")
    .agg(
        rows=("trade_dt", "count"),
        avg_current_signal=("current_forecast_vrp_signal", "mean"),
        avg_champion_signal=("champion_forecast_vrp_signal", "mean"),
        avg_diff=("champion_minus_current_vrp_signal", "mean"),
        median_diff=("champion_minus_current_vrp_signal", "median"),
        corr_current_champion=(
            "current_forecast_vrp_signal",
            lambda x: x.corr(
                compare_vrp_df.loc[x.index, "champion_forecast_vrp_signal"]
            )
        ),
    )
    .reset_index()
)

summary_df.to_csv(FORECAST_VRP_CHAMPION_COMPARISON_CSV, index=False)

print("Saved champion/current VRP comparison:")
print(FORECAST_VRP_CHAMPION_COMPARISON_PARQUET)
print(FORECAST_VRP_CHAMPION_COMPARISON_CSV)

display(summary_df)

print("\nDistribution of champion minus current signal:")
display(
    compare_vrp_df["champion_minus_current_vrp_signal"]
    .describe()
    .to_frame("champion_minus_current")
)

Current forecast VRP columns:
vix_style_vol
implied_variance
trailing_realized_variance
trailing_realized_vol
primary_vrp_signal
ewma_10_scheduled_forecast_variance
ewma_10_scheduled_forecast_vol
ewma_10_scheduled_vrp_signal
ewma_10_scheduled_bias_adjusted_forecast_variance
ewma_10_scheduled_bias_adjusted_forecast_vol
ewma_10_scheduled_bias_adjusted_vrp_signal
simple_forecast_model
simple_forecast_variance
simple_forecast_vol
simple_forecast_vrp_signal
simple_forecast_vrp_vol_ratio
forward_realized_variance
forward_realized_vol

Using current forecast VRP signal column: simple_forecast_vrp_signal
Saved champion/current VRP comparison:
C:\Users\patri\vrp_project\data\processed\forecast_vrp_champion_comparison_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\forecast_vrp_champion_comparison_summary_v0_1.csv


,tenor,rows,avg_current_signal,avg_champion_signal,avg_diff,median_diff,corr_current_champion
0,9,1440,0.701509,0.514433,-0.187076,-0.161638,-0.277263
1,12,1481,0.729141,0.511740,-0.217401,-0.185217,-0.281806
2,15,1488,0.653270,0.505992,-0.147278,-0.122906,-0.263752
3,18,1455,0.680211,0.501554,-0.178657,-0.144611,-0.221975
4,21,1398,0.485570,0.498498,0.012928,0.022641,-0.205173
5,24,1129,0.520124,0.479667,-0.040457,-0.016521,-0.150015
6,27,1343,0.543440,0.488127,-0.055313,-0.041411,-0.174654
7,30,1240,0.541388,0.476239,-0.065149,-0.048068,-0.155367
8,33,1286,0.584773,0.478770,-0.106003,-0.088589,-0.159077



Distribution of champion minus current signal:


,champion_minus_current
count,12260.000000
mean,-0.113332
std,0.565944
min,-2.780485
25%,-0.425569
50%,-0.089113
75%,0.259134
max,1.651761


In [15]:
# ============================================================
# Cell 15: Validate current forecast signal formula and compare denominators
# ============================================================

import numpy as np
import pandas as pd

current_forecast_vrp_df = pd.read_parquet(
    CURRENT_FORECAST_VRP_PANEL_PARQUET
).copy()

champion_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "forecast_vol_champion_panel_v0_1.parquet"
).copy()

current_forecast_vrp_df["trade_dt"] = pd.to_datetime(
    current_forecast_vrp_df["trade_date"].astype(str),
    format="%Y%m%d",
    errors="coerce",
)

champion_df["trade_dt"] = pd.to_datetime(champion_df["trade_dt"])

current_forecast_vrp_df["tenor"] = current_forecast_vrp_df["tenor"].astype(int)
champion_df["tenor"] = champion_df["tenor"].astype(int)

diagnostic_df = champion_df.merge(
    current_forecast_vrp_df[
        [
            "trade_dt",
            "tenor",
            "implied_variance",
            "vix_style_vol",
            "simple_forecast_model",
            "simple_forecast_variance",
            "simple_forecast_vol",
            "simple_forecast_vrp_signal",
            "simple_forecast_vrp_vol_ratio",
            "trailing_realized_variance",
            "trailing_realized_vol",
            "primary_vrp_signal",
        ]
    ].rename(
        columns={
            "implied_variance": "current_implied_variance",
            "vix_style_vol": "current_vix_style_vol",
            "trailing_realized_variance": "current_trailing_realized_variance",
            "trailing_realized_vol": "current_trailing_realized_vol",
        }
    ),
    on=["trade_dt", "tenor"],
    how="left",
)

# Validate current simple signal formula.
diagnostic_df["calc_simple_forecast_vrp_signal"] = np.log(
    diagnostic_df["current_implied_variance"]
    / diagnostic_df["simple_forecast_variance"].clip(lower=1e-10)
)

diagnostic_df["simple_signal_formula_error"] = (
    diagnostic_df["simple_forecast_vrp_signal"]
    - diagnostic_df["calc_simple_forecast_vrp_signal"]
)

# Champion formula check.
diagnostic_df["calc_champion_forecast_vrp_signal"] = np.log(
    diagnostic_df["implied_variance"]
    / diagnostic_df["champion_forecast_variance"].clip(lower=1e-10)
)

diagnostic_df["champion_signal_formula_error"] = (
    diagnostic_df["champion_forecast_vrp_signal"]
    - diagnostic_df["calc_champion_forecast_vrp_signal"]
)

# Denominator comparison.
diagnostic_df["simple_minus_champion_forecast_vol"] = (
    diagnostic_df["simple_forecast_vol"]
    - diagnostic_df["champion_forecast_vol"]
)

diagnostic_df["simple_div_champion_forecast_variance"] = (
    diagnostic_df["simple_forecast_variance"]
    / diagnostic_df["champion_forecast_variance"].clip(lower=1e-10)
)

diagnostic_df["champion_minus_simple_signal"] = (
    diagnostic_df["champion_forecast_vrp_signal"]
    - diagnostic_df["simple_forecast_vrp_signal"]
)

print("Signal formula validation:")
display(
    diagnostic_df[
        [
            "simple_signal_formula_error",
            "champion_signal_formula_error",
        ]
    ].describe().T
)

print("\nForecast denominator comparison by tenor:")
display(
    diagnostic_df
    .groupby("tenor")
    .agg(
        rows=("trade_dt", "count"),
        avg_simple_forecast_vol=("simple_forecast_vol", "mean"),
        avg_champion_forecast_vol=("champion_forecast_vol", "mean"),
        avg_trailing_realized_vol=("current_trailing_realized_vol", "mean"),
        avg_implied_vol=("current_vix_style_vol", "mean"),
        avg_simple_minus_champion_vol=("simple_minus_champion_forecast_vol", "mean"),
        median_simple_minus_champion_vol=("simple_minus_champion_forecast_vol", "median"),
        avg_simple_div_champion_var=("simple_div_champion_forecast_variance", "mean"),
        corr_simple_champion_forecast_vol=(
            "simple_forecast_vol",
            lambda x: x.corr(
                diagnostic_df.loc[x.index, "champion_forecast_vol"]
            )
        ),
        corr_simple_champion_signal=(
            "simple_forecast_vrp_signal",
            lambda x: x.corr(
                diagnostic_df.loc[x.index, "champion_forecast_vrp_signal"]
            )
        ),
    )
    .reset_index()
)

print("\nSimple forecast model distribution:")
display(
    diagnostic_df
    .groupby(["tenor", "simple_forecast_model"])
    .size()
    .reset_index(name="rows")
    .sort_values(["tenor", "rows"], ascending=[True, False])
)

print("\nLargest champion/current signal disagreements:")
display(
    diagnostic_df
    .sort_values("champion_minus_simple_signal")
    [
        [
            "trade_dt",
            "tenor",
            "simple_forecast_model",
            "current_vix_style_vol",
            "current_trailing_realized_vol",
            "simple_forecast_vol",
            "champion_forecast_vol",
            "simple_forecast_vrp_signal",
            "champion_forecast_vrp_signal",
            "champion_minus_simple_signal",
        ]
    ]
    .head(30)
)

print("\nLargest cases where champion is richer than current:")
display(
    diagnostic_df
    .sort_values("champion_minus_simple_signal", ascending=False)
    [
        [
            "trade_dt",
            "tenor",
            "simple_forecast_model",
            "current_vix_style_vol",
            "current_trailing_realized_vol",
            "simple_forecast_vol",
            "champion_forecast_vol",
            "simple_forecast_vrp_signal",
            "champion_forecast_vrp_signal",
            "champion_minus_simple_signal",
        ]
    ]
    .head(30)
)

Signal formula validation:


,count,mean,std,min,25%,50%,75%,max
simple_signal_formula_error,12260.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.0,0.000000e+00,0.000000e+00
champion_signal_formula_error,12260.0,1.102552e-18,1.660441e-16,-5.551115e-16,-1.110223e-16,0.0,1.110223e-16,5.551115e-16



Forecast denominator comparison by tenor:


,tenor,rows,avg_simple_forecast_vol,avg_champion_forecast_vol,avg_trailing_realized_vol,avg_implied_vol,avg_simple_minus_champion_vol,median_simple_minus_champion_vol,avg_simple_div_champion_var,corr_simple_champion_forecast_vol,corr_simple_champion_signal
0,9,1440,13.477266,14.793730,15.199965,18.811608,-1.316463,-0.945705,0.974558,0.607346,-0.277263
1,12,1481,13.456043,14.922550,14.831473,18.974132,-1.466507,-1.100909,0.939820,0.611050,-0.281806
2,15,1488,14.142007,15.084612,15.210329,19.137966,-0.942605,-0.727966,1.004604,0.612045,-0.263752
3,18,1455,14.141984,15.259096,15.135618,19.325963,-1.117112,-0.926310,0.972570,0.613930,-0.221975
4,21,1398,15.796341,15.459970,15.337553,19.564383,0.336370,0.140560,1.171201,0.616812,-0.205173
5,24,1129,15.803061,15.835359,15.274822,19.884898,-0.032298,-0.115053,1.101742,0.619927,-0.150015
6,27,1343,15.689290,15.838051,15.461772,19.945678,-0.148761,-0.282621,1.096653,0.604026,-0.174654
7,30,1240,15.845176,16.035572,15.624401,20.092588,-0.190397,-0.293626,1.080772,0.611108,-0.155367
8,33,1286,15.676532,16.211213,15.514860,20.313371,-0.534681,-0.586411,1.042497,0.599258,-0.159077



Simple forecast model distribution:


,tenor,simple_forecast_model,rows
0,9,ewma_10_scheduled_bias_adjusted,1440
1,12,ewma_10_scheduled_bias_adjusted,1481
2,15,ewma_10_scheduled_bias_adjusted,1488
3,18,ewma_10_scheduled_bias_adjusted,1455
4,21,ewma_10_scheduled_raw,1398
5,24,ewma_10_scheduled_raw,1129
6,27,ewma_10_scheduled_raw,1343
7,30,ewma_10_scheduled_raw,1240
8,33,ewma_10_scheduled_raw,1286



Largest champion/current signal disagreements:


,trade_dt,tenor,simple_forecast_model,current_vix_style_vol,current_trailing_realized_vol,simple_forecast_vol,champion_forecast_vol,simple_forecast_vrp_signal,champion_forecast_vrp_signal,champion_minus_simple_signal
2999,2021-11-26,9,ewma_10_scheduled_bias_adjusted,30.600000,15.070567,8.942860,35.912953,2.460289,-0.320196,-2.780485
3000,2021-11-26,12,ewma_10_scheduled_bias_adjusted,29.767740,13.301592,10.091209,34.951971,2.163521,-0.321099,-2.484620
2526,2021-09-10,9,ewma_10_scheduled_bias_adjusted,22.011754,6.459748,6.054343,20.229395,2.581601,0.168880,-2.412722
3001,2021-11-26,15,ewma_10_scheduled_bias_adjusted,29.294650,12.415661,10.341760,34.403412,2.082430,-0.321502,-2.403931
3002,2021-11-26,18,ewma_10_scheduled_bias_adjusted,29.063187,12.035121,10.506346,34.134427,2.034986,-0.321668,-2.356654
9309,2024-12-18,9,ewma_10_scheduled_bias_adjusted,35.420000,20.438139,12.920706,38.376192,2.016891,-0.160321,-2.177212
2527,2021-09-10,12,ewma_10_scheduled_bias_adjusted,21.904760,6.123503,6.842643,20.136815,2.327060,0.168308,-2.158752
386,2020-09-01,33,ewma_10_scheduled_raw,26.453528,9.709902,11.655492,33.936314,1.639224,-0.498192,-2.137416
9310,2024-12-18,12,ewma_10_scheduled_bias_adjusted,32.849406,18.023370,12.329511,35.793881,1.959876,-0.171687,-2.131563
3026,2021-12-01,12,ewma_10_scheduled_bias_adjusted,34.174661,19.313657,12.568331,36.251175,2.000608,-0.117975,-2.118583



Largest cases where champion is richer than current:


,trade_dt,tenor,simple_forecast_model,current_vix_style_vol,current_trailing_realized_vol,simple_forecast_vol,champion_forecast_vol,simple_forecast_vrp_signal,champion_forecast_vrp_signal,champion_minus_simple_signal
10101,2025-05-16,21,ewma_10_scheduled_raw,16.555064,16.614169,27.152106,11.888529,-0.989525,0.662236,1.651761
10067,2025-05-12,24,ewma_10_scheduled_raw,18.038561,22.681341,31.763875,14.194027,-1.131636,0.479381,1.611017
10069,2025-05-12,30,ewma_10_scheduled_raw,18.216455,21.960012,31.576476,14.319648,-1.100174,0.481385,1.581560
10076,2025-05-13,24,ewma_10_scheduled_raw,18.106981,22.855539,30.828694,14.009040,-1.064297,0.513189,1.577486
10102,2025-05-16,27,ewma_10_scheduled_raw,17.082359,21.756896,27.152106,12.338678,-0.926817,0.650615,1.577431
10066,2025-05-12,21,ewma_10_scheduled_raw,17.842601,22.114955,30.815485,14.035321,-1.092857,0.480024,1.572880
10070,2025-05-12,33,ewma_10_scheduled_raw,18.251010,24.721798,31.508054,14.385467,-1.092046,0.476004,1.568050
10093,2025-05-15,21,ewma_10_scheduled_raw,17.463163,16.641514,27.957821,12.792675,-0.941207,0.622442,1.563648
10103,2025-05-16,33,ewma_10_scheduled_raw,17.264331,21.249895,27.152106,12.461064,-0.905624,0.652067,1.557691
10065,2025-05-12,18,ewma_10_scheduled_bias_adjusted,17.577923,17.574389,29.931424,13.758300,-1.064530,0.490003,1.554533


In [16]:
# ============================================================
# Cell 16: Build champion forecast VRP z-score panel
# ============================================================

import numpy as np
import pandas as pd

champion_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "forecast_vol_champion_panel_v0_1.parquet"
).copy()

champion_df["trade_dt"] = pd.to_datetime(champion_df["trade_dt"])
champion_df["tenor"] = champion_df["tenor"].astype(int)

champion_z_df = champion_df[
    [
        "trade_date",
        "trade_dt",
        "tenor",
        "spx_close",
        "spx_rsi_14",
        "vix_style_vol",
        "implied_variance",
        "pred_logvar_implied",
        "champion_model",
        "champion_forecast_log_variance",
        "champion_forecast_variance",
        "champion_forecast_vol",
        "champion_forecast_vrp_signal",
        "champion_forecast_vrp_variance_ratio",
        "champion_forecast_vrp_vol_ratio",
        "target_variance",
        "target_vol_pct",
        "target_log_variance",
    ]
].copy()

champion_z_df = champion_z_df.sort_values(["tenor", "trade_dt"]).reset_index(drop=True)

# Rolling z-scores by tenor.
# Use same basic convention as prior VRP research: current observation included in rolling window.
g = champion_z_df.groupby("tenor", group_keys=False)

for window, label in [(63, "3m"), (252, "1y")]:
    mean_col = f"champion_forecast_vrp_signal_mean_{label}"
    std_col = f"champion_forecast_vrp_signal_std_{label}"
    z_col = f"champion_forecast_vrp_signal_z_{label}"

    champion_z_df[mean_col] = (
        g["champion_forecast_vrp_signal"]
        .rolling(window, min_periods=max(20, window // 3))
        .mean()
        .reset_index(level=0, drop=True)
    )

    champion_z_df[std_col] = (
        g["champion_forecast_vrp_signal"]
        .rolling(window, min_periods=max(20, window // 3))
        .std()
        .reset_index(level=0, drop=True)
    )

    champion_z_df[z_col] = (
        (
            champion_z_df["champion_forecast_vrp_signal"]
            - champion_z_df[mean_col]
        )
        / champion_z_df[std_col].replace(0, np.nan)
    )

# Combined conservative z-score: require richness on both short and long history.
champion_z_df["champion_forecast_vrp_min_z"] = champion_z_df[
    [
        "champion_forecast_vrp_signal_z_3m",
        "champion_forecast_vrp_signal_z_1y",
    ]
].min(axis=1)

CHAMPION_FORECAST_VRP_ZSCORE_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vrp_champion_zscore_panel_v0_1.parquet"
)

CHAMPION_FORECAST_VRP_ZSCORE_CSV = (
    AUDIT_DIR / "forecast_vrp_champion_zscore_panel_sample_v0_1.csv"
)

champion_z_df.to_parquet(CHAMPION_FORECAST_VRP_ZSCORE_PARQUET, index=False)
champion_z_df.head(1000).to_csv(CHAMPION_FORECAST_VRP_ZSCORE_CSV, index=False)

print("Saved champion forecast VRP z-score panel:")
print(CHAMPION_FORECAST_VRP_ZSCORE_PARQUET)
print(CHAMPION_FORECAST_VRP_ZSCORE_CSV)

print("\nRows:", len(champion_z_df))
print("Dates:", champion_z_df["trade_dt"].min(), "to", champion_z_df["trade_dt"].max())

display(
    champion_z_df[
        [
            "trade_dt",
            "tenor",
            "vix_style_vol",
            "champion_forecast_vol",
            "champion_forecast_vrp_signal",
            "champion_forecast_vrp_signal_z_3m",
            "champion_forecast_vrp_signal_z_1y",
            "champion_forecast_vrp_min_z",
        ]
    ].tail(20)
)

Saved champion forecast VRP z-score panel:
C:\Users\patri\vrp_project\data\processed\forecast_vrp_champion_zscore_panel_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\forecast_vrp_champion_zscore_panel_sample_v0_1.csv

Rows: 12260
Dates: 2020-07-01 00:00:00 to 2026-06-17 00:00:00


,trade_dt,tenor,vix_style_vol,champion_forecast_vol,champion_forecast_vrp_signal,champion_forecast_vrp_signal_z_3m,champion_forecast_vrp_signal_z_1y,champion_forecast_vrp_min_z
12240,2026-04-21,33,20.322406,17.036314,0.352754,-0.735562,-0.780841,-0.780841
12241,2026-04-22,33,18.775624,14.953868,0.455179,0.380692,-0.274292,-0.274292
12242,2026-04-23,33,19.032528,15.744579,0.379307,-0.420143,-0.647835,-0.647835
12243,2026-04-24,33,18.654144,15.046258,0.429878,0.150374,-0.396505,-0.396505
12244,2026-04-27,33,18.210269,15.091030,0.375770,-0.444141,-0.664455,-0.664455
12245,2026-04-28,33,17.951269,14.875393,0.375905,-0.435300,-0.664112,-0.664112
12246,2026-04-29,33,18.561158,15.269007,0.390492,-0.266447,-0.592407,-0.592407
12247,2026-04-30,33,17.289005,13.226654,0.535673,1.301580,0.122621,0.122621
12248,2026-05-01,33,17.275351,12.893531,0.585110,1.780449,0.364829,0.364829
12249,2026-05-04,33,18.343626,14.245817,0.505638,0.953474,-0.034546,-0.034546


In [17]:
# ============================================================
# Cell 17: Compare champion forecast signal on Phase 3 accepted trades
# ============================================================

import numpy as np
import pandas as pd

phase3_trades_df = pd.read_parquet(
    PHASE3_STRATEGY_PARQUET
).copy()

champion_z_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "forecast_vrp_champion_zscore_panel_v0_1.parquet"
).copy()

current_forecast_vrp_df = pd.read_parquet(
    CURRENT_FORECAST_VRP_PANEL_PARQUET
).copy()

phase3_trades_df["trade_dt"] = pd.to_datetime(phase3_trades_df["trade_dt"])
champion_z_df["trade_dt"] = pd.to_datetime(champion_z_df["trade_dt"])

current_forecast_vrp_df["trade_dt"] = pd.to_datetime(
    current_forecast_vrp_df["trade_date"].astype(str),
    format="%Y%m%d",
    errors="coerce",
)

for df in [phase3_trades_df, champion_z_df, current_forecast_vrp_df]:
    if "tenor" in df.columns:
        df["tenor"] = df["tenor"].astype(int)

# Use target_tenor as selected signal tenor.
if "target_tenor" in phase3_trades_df.columns:
    phase3_trades_df["entry_tenor"] = phase3_trades_df["target_tenor"].astype(int)
elif "tenor" in phase3_trades_df.columns:
    phase3_trades_df["entry_tenor"] = phase3_trades_df["tenor"].astype(int)
else:
    raise ValueError("Could not find target_tenor / tenor in Phase 3 trades.")

trade_champion_df = phase3_trades_df.merge(
    champion_z_df[
        [
            "trade_dt",
            "tenor",
            "champion_forecast_vol",
            "champion_forecast_vrp_signal",
            "champion_forecast_vrp_signal_z_3m",
            "champion_forecast_vrp_signal_z_1y",
            "champion_forecast_vrp_min_z",
        ]
    ],
    left_on=["trade_dt", "entry_tenor"],
    right_on=["trade_dt", "tenor"],
    how="left",
    suffixes=("", "_champion"),
)

trade_champion_df = trade_champion_df.merge(
    current_forecast_vrp_df[
        [
            "trade_dt",
            "tenor",
            "simple_forecast_vol",
            "simple_forecast_vrp_signal",
        ]
    ],
    left_on=["trade_dt", "entry_tenor"],
    right_on=["trade_dt", "tenor"],
    how="left",
    suffixes=("", "_current"),
)

trade_champion_df["champion_minus_simple_signal"] = (
    trade_champion_df["champion_forecast_vrp_signal"]
    - trade_champion_df["simple_forecast_vrp_signal"]
)

# Broad diagnostic buckets, not optimized.
trade_champion_df["champion_signal_bucket"] = pd.cut(
    trade_champion_df["champion_forecast_vrp_signal"],
    bins=[-np.inf, 0.0, 0.25, 0.50, 0.75, 1.00, np.inf],
    labels=["neg", "0_25", "25_50", "50_75", "75_100", "gt_100"],
)

trade_champion_df["champion_min_z_bucket"] = pd.cut(
    trade_champion_df["champion_forecast_vrp_min_z"],
    bins=[-np.inf, -1.0, 0.0, 0.5, 1.0, 1.5, np.inf],
    labels=["lt_neg1", "neg1_0", "0_0p5", "0p5_1", "1_1p5", "gt_1p5"],
)

TRADE_CHAMPION_DIAGNOSTIC_PARQUET = (
    PROCESSED_DATA_DIR / "put_phase3_trade_champion_forecast_diagnostic_v0_1.parquet"
)

TRADE_CHAMPION_DIAGNOSTIC_CSV = (
    AUDIT_DIR / "put_phase3_trade_champion_forecast_diagnostic_v0_1.csv"
)

trade_champion_df.to_parquet(TRADE_CHAMPION_DIAGNOSTIC_PARQUET, index=False)
trade_champion_df.to_csv(TRADE_CHAMPION_DIAGNOSTIC_CSV, index=False)

print("Saved Phase 3 trade champion diagnostic:")
print(TRADE_CHAMPION_DIAGNOSTIC_PARQUET)
print(TRADE_CHAMPION_DIAGNOSTIC_CSV)

def summarize_trades(df, group_cols):
    return (
        df
        .groupby(group_cols, dropna=False, observed=False)
        .agg(
            trades=("trade_dt", "count"),
            total_pnl_pct=("actual_pnl_pct_of_nw", "sum"),
            avg_pnl_pct=("actual_pnl_pct_of_nw", "mean"),
            median_pnl_pct=("actual_pnl_pct_of_nw", "median"),
            win_rate=("actual_pnl_pct_of_nw", lambda x: (x > 0).mean()),
            worst_pnl_pct=("actual_pnl_pct_of_nw", "min"),
            q05_pnl_pct=("actual_pnl_pct_of_nw", lambda x: x.quantile(0.05)),
            avg_simple_signal=("simple_forecast_vrp_signal", "mean"),
            avg_champion_signal=("champion_forecast_vrp_signal", "mean"),
            avg_champion_min_z=("champion_forecast_vrp_min_z", "mean"),
        )
        .reset_index()
    )

print("\nPhase 3 accepted trades by champion signal bucket:")
display(
    summarize_trades(
        trade_champion_df,
        ["champion_signal_bucket", "signal_tier_2level"],
    )
)

print("\nPhase 3 accepted trades by champion min-z bucket:")
display(
    summarize_trades(
        trade_champion_df,
        ["champion_min_z_bucket", "signal_tier_2level"],
    )
)

print("\nPhase 3 accepted trades where champion signal is negative:")
display(
    summarize_trades(
        trade_champion_df.assign(
            champion_signal_negative=trade_champion_df["champion_forecast_vrp_signal"] < 0
        ),
        ["champion_signal_negative", "signal_tier_2level"],
    )
)

print("\nWorst accepted trades by champion signal:")
display(
    trade_champion_df
    .sort_values("actual_pnl_pct_of_nw")
    [
        [
            "trade_dt",
            "signal_tier_2level",
            "tenor_group",
            "entry_tenor",
            "actual_pnl_pct_of_nw",
            "simple_forecast_vrp_signal",
            "champion_forecast_vrp_signal",
            "champion_forecast_vrp_min_z",
            "simple_forecast_vol",
            "champion_forecast_vol",
            "vix_style_vol",
            "spx_rsi_14",
        ]
    ]
    .head(30)
)

Saved Phase 3 trade champion diagnostic:
C:\Users\patri\vrp_project\data\processed\put_phase3_trade_champion_forecast_diagnostic_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\put_phase3_trade_champion_forecast_diagnostic_v0_1.csv

Phase 3 accepted trades by champion signal bucket:


,champion_signal_bucket,signal_tier_2level,trades,total_pnl_pct,avg_pnl_pct,median_pnl_pct,win_rate,worst_pnl_pct,q05_pnl_pct,avg_simple_signal,avg_champion_signal,avg_champion_min_z
0,neg,Core,9,0.045815,0.005091,0.004500,1.000000,0.002793,0.003248,1.292526,-0.127699,-2.255428
1,neg,Secondary,4,0.017720,0.004430,0.004665,1.000000,0.002752,0.002998,0.820311,-0.340267,-4.478573
2,0_25,Core,17,0.078638,0.004626,0.004948,0.941176,-0.001179,0.000062,1.262597,0.123002,-1.205595
3,0_25,Secondary,14,0.014850,0.001061,0.001375,0.928571,-0.005055,-0.001513,0.946183,0.150988,-1.505079
4,25_50,Core,42,0.044782,0.001066,0.003712,0.738095,-0.014330,-0.011630,1.281213,0.390051,-0.746426
5,25_50,Secondary,66,0.052422,0.000794,0.001459,0.818182,-0.008625,-0.004338,0.838564,0.398226,-0.791488
6,50_75,Core,27,0.063196,0.002341,0.003188,0.851852,-0.009340,-0.001628,1.053343,0.631140,0.385835
7,50_75,Secondary,71,0.045796,0.000645,0.001433,0.788732,-0.008526,-0.003238,0.778988,0.613139,0.113043
8,75_100,Core,6,0.021773,0.003629,0.003667,1.000000,0.001725,0.001983,1.170487,0.817707,1.337384
9,75_100,Secondary,16,0.006054,0.000378,0.001343,0.687500,-0.003701,-0.002834,0.646722,0.825249,1.203824



Phase 3 accepted trades by champion min-z bucket:


,champion_min_z_bucket,signal_tier_2level,trades,total_pnl_pct,avg_pnl_pct,median_pnl_pct,win_rate,worst_pnl_pct,q05_pnl_pct,avg_simple_signal,avg_champion_signal,avg_champion_min_z
0,lt_neg1,Core,31,0.085346,0.002753,0.004515,0.838710,-0.014330,-0.010888,1.285894,0.180574,-1.875332
1,lt_neg1,Secondary,41,0.040040,0.000977,0.001216,0.829268,-0.006114,-0.004686,0.804543,0.277715,-1.900897
2,neg1_0,Core,34,0.046997,0.001382,0.003562,0.735294,-0.014202,-0.009892,1.211110,0.341578,-0.555326
3,neg1_0,Secondary,59,0.036765,0.000623,0.001364,0.762712,-0.008625,-0.003119,0.835974,0.440632,-0.535946
4,0_0p5,Core,19,0.060660,0.003193,0.003400,0.894737,-0.001114,-0.000543,1.150205,0.524788,0.184615
5,0_0p5,Secondary,33,0.033760,0.001023,0.001560,0.848485,-0.007562,-0.002333,0.832614,0.565274,0.224765
6,0p5_1,Core,8,0.028034,0.003504,0.003840,1.000000,0.001520,0.001820,1.183020,0.643436,0.691752
7,0p5_1,Secondary,20,0.005992,0.000300,0.001452,0.800000,-0.008526,-0.008214,0.756514,0.703998,0.784447
8,1_1p5,Core,4,0.016624,0.004156,0.004174,1.000000,0.002759,0.002949,1.122977,0.799048,1.287224
9,1_1p5,Secondary,10,0.002674,0.000267,0.000940,0.700000,-0.002544,-0.002361,0.753216,0.752423,1.191189



Phase 3 accepted trades where champion signal is negative:


,champion_signal_negative,signal_tier_2level,trades,total_pnl_pct,avg_pnl_pct,median_pnl_pct,win_rate,worst_pnl_pct,q05_pnl_pct,avg_simple_signal,avg_champion_signal,avg_champion_min_z
0,False,Core,121,0.299851,0.002478,0.003864,0.851240,-0.014330,-0.009340,1.208514,0.439350,-0.363078
1,False,Secondary,244,0.169986,0.000697,0.001403,0.807377,-0.012191,-0.003698,0.794948,0.519576,-0.257708
2,True,Core,9,0.045815,0.005091,0.004500,1.000000,0.002793,0.003248,1.292526,-0.127699,-2.255428
3,True,Secondary,4,0.017720,0.004430,0.004665,1.000000,0.002752,0.002998,0.820311,-0.340267,-4.478573



Worst accepted trades by champion signal:


,trade_dt,signal_tier_2level,tenor_group,entry_tenor,actual_pnl_pct_of_nw,simple_forecast_vrp_signal,champion_forecast_vrp_signal,champion_forecast_vrp_min_z,simple_forecast_vol,champion_forecast_vol,vix_style_vol,spx_rsi_14
365,2026-03-02,Core,middle_18_24d,24,-0.014330,1.155171,0.351278,-1.591343,11.945207,17.854874,21.283148,48.647576
367,2026-03-04,Core,middle_18_24d,24,-0.014202,1.152274,0.419215,-0.858135,11.924611,17.203865,21.215699,48.264262
51,2020-02-21,Secondary,front_9_15d,12,-0.012191,1.196984,NaN,NaN,9.389261,NaN,17.082566,53.661884
366,2026-03-03,Core,middle_18_24d,24,-0.011712,1.323558,0.289864,-2.062310,12.232689,20.510945,23.709850,43.035216
313,2025-02-26,Core,middle_18_24d,18,-0.010064,1.131838,0.427562,-1.233708,10.652274,15.148652,18.759356,41.127776
312,2025-02-25,Core,middle_18_24d,18,-0.009964,1.045518,0.493949,-0.637845,11.478584,15.123768,19.360640,41.026556
364,2026-02-27,Core,middle_18_24d,18,-0.009853,1.162275,0.438358,-0.720179,10.749466,15.437736,19.220813,48.384049
237,2023-08-01,Core,front_9_15d,15,-0.009340,0.846497,0.597491,-0.440516,8.429449,9.547072,12.871040,66.757074
309,2025-02-18,Secondary,middle_18_24d,21,-0.008625,0.437240,0.482647,-0.021175,11.687632,11.425270,14.543600,59.893776
311,2025-02-20,Secondary,middle_18_24d,24,-0.008526,0.658259,0.592173,0.687853,10.720143,11.080287,14.898404,57.563644


In [18]:
# ============================================================
# Cell 18: Simple champion-confirmation diagnostic on accepted trades
# ============================================================

import numpy as np
import pandas as pd

trade_champion_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "put_phase3_trade_champion_forecast_diagnostic_v0_1.parquet"
).copy()

confirmation_rules = [
    {
        "rule": "champion_signal_gt_0",
        "pass": trade_champion_df["champion_forecast_vrp_signal"] > 0.0,
    },
    {
        "rule": "champion_signal_gt_0p25",
        "pass": trade_champion_df["champion_forecast_vrp_signal"] > 0.25,
    },
    {
        "rule": "champion_signal_gt_0p50",
        "pass": trade_champion_df["champion_forecast_vrp_signal"] > 0.50,
    },
    {
        "rule": "champion_min_z_gt_0",
        "pass": trade_champion_df["champion_forecast_vrp_min_z"] > 0.0,
    },
    {
        "rule": "champion_min_z_gt_0p5",
        "pass": trade_champion_df["champion_forecast_vrp_min_z"] > 0.5,
    },
]

rows = []

for rule in confirmation_rules:
    temp = trade_champion_df.copy()
    temp["passes_rule"] = rule["pass"].values

    for tier in ["Core", "Secondary"]:
        tier_df = temp[temp["signal_tier_2level"] == tier].copy()

        kept = tier_df[tier_df["passes_rule"]].copy()
        removed = tier_df[~tier_df["passes_rule"]].copy()

        rows.append({
            "rule": rule["rule"],
            "tier": tier,

            "total_trades": len(tier_df),
            "kept_trades": len(kept),
            "removed_trades": len(removed),
            "kept_pct": len(kept) / len(tier_df) if len(tier_df) else np.nan,

            "total_pnl_pct": tier_df["actual_pnl_pct_of_nw"].sum(),
            "kept_pnl_pct": kept["actual_pnl_pct_of_nw"].sum(),
            "removed_pnl_pct": removed["actual_pnl_pct_of_nw"].sum(),

            "kept_avg_pnl_pct": kept["actual_pnl_pct_of_nw"].mean(),
            "removed_avg_pnl_pct": removed["actual_pnl_pct_of_nw"].mean(),

            "kept_win_rate": (kept["actual_pnl_pct_of_nw"] > 0).mean() if len(kept) else np.nan,
            "removed_win_rate": (removed["actual_pnl_pct_of_nw"] > 0).mean() if len(removed) else np.nan,

            "kept_worst_pnl_pct": kept["actual_pnl_pct_of_nw"].min() if len(kept) else np.nan,
            "removed_worst_pnl_pct": removed["actual_pnl_pct_of_nw"].min() if len(removed) else np.nan,
        })

champion_confirmation_summary_df = pd.DataFrame(rows)

CHAMPION_CONFIRMATION_SUMMARY_CSV = (
    AUDIT_DIR / "put_phase3_champion_confirmation_summary_v0_1.csv"
)

champion_confirmation_summary_df.to_csv(
    CHAMPION_CONFIRMATION_SUMMARY_CSV,
    index=False,
)

print("Saved champion confirmation summary:")
print(CHAMPION_CONFIRMATION_SUMMARY_CSV)

display(champion_confirmation_summary_df)

Saved champion confirmation summary:
C:\Users\patri\vrp_project\data\audit\put_phase3_champion_confirmation_summary_v0_1.csv


,rule,tier,total_trades,kept_trades,removed_trades,kept_pct,total_pnl_pct,kept_pnl_pct,removed_pnl_pct,kept_avg_pnl_pct,removed_avg_pnl_pct,kept_win_rate,removed_win_rate,kept_worst_pnl_pct,removed_worst_pnl_pct
0,champion_signal_gt_0,Core,130,92,38,0.707692,0.345666,0.208389,0.137277,0.002265,0.003613,0.826087,0.947368,-0.014330,-0.007916
1,champion_signal_gt_0,Secondary,248,170,78,0.685484,0.187706,0.120637,0.067069,0.000710,0.000860,0.800000,0.833333,-0.008625,-0.012191
2,champion_signal_gt_0p25,Core,130,75,55,0.576923,0.345666,0.129751,0.215915,0.001730,0.003926,0.800000,0.945455,-0.014330,-0.007916
3,champion_signal_gt_0p25,Secondary,248,156,92,0.629032,0.187706,0.105787,0.081919,0.000678,0.000890,0.788462,0.847826,-0.008625,-0.012191
4,champion_signal_gt_0p50,Core,130,33,97,0.253846,0.345666,0.084969,0.260697,0.002575,0.002688,0.878788,0.855670,-0.009340,-0.014330
5,champion_signal_gt_0p50,Secondary,248,90,158,0.362903,0.187706,0.053365,0.134341,0.000593,0.000850,0.766667,0.835443,-0.008526,-0.012191
6,champion_min_z_gt_0,Core,130,36,94,0.276923,0.345666,0.121860,0.223806,0.003385,0.002381,0.944444,0.829787,-0.001114,-0.014330
7,champion_min_z_gt_0,Secondary,248,70,178,0.282258,0.187706,0.049383,0.138323,0.000705,0.000777,0.814286,0.808989,-0.008526,-0.012191
8,champion_min_z_gt_0p5,Core,130,17,113,0.130769,0.345666,0.061200,0.284466,0.003600,0.002517,1.000000,0.840708,0.001520,-0.014330
9,champion_min_z_gt_0p5,Secondary,248,37,211,0.149194,0.187706,0.015624,0.172082,0.000422,0.000816,0.783784,0.815166,-0.008526,-0.012191


In [19]:
# ============================================================
# Cell 19: Save champion forecast signal diagnostic conclusion
# ============================================================

import json
import pandas as pd

CHAMPION_SIGNAL_DIAGNOSTIC_CONFIG_JSON = (
    AUDIT_DIR / "forecast_vol_champion_signal_diagnostic_config_v0_1.json"
)

CHAMPION_SIGNAL_DIAGNOSTIC_SUMMARY_CSV = (
    AUDIT_DIR / "forecast_vol_champion_signal_diagnostic_summary_v0_1.csv"
)

champion_signal_diagnostic_config = {
    "strategy_version": "forecast_vol_champion_signal_diagnostic_v0_1",

    "champion_model": "har_rv_iv",

    "forecast_model_conclusion": {
        "best_forecast_model": True,
        "better_than_existing_simple_forecast": True,
        "better_than_trailing_realized_variance": True,
        "useful_as_forecast_denominator": True,
    },

    "trading_signal_conclusion": {
        "replace_current_signal": False,
        "use_as_hard_confirmation_filter": False,
        "use_as_secondary_suppressor": False,
        "use_as_core_suppressor": False,
        "use_as_dashboard_context": True,
        "use_as_quality_tag": True,
    },

    "key_findings": {
        "champion_signal_negative_trades": "All profitable in accepted Phase 3 sample.",
        "champion_signal_gt_0_filter": "Removed trades were profitable and had higher average P&L than kept trades.",
        "champion_min_z_gt_0_core": "Identifies a high-quality Core subset but removes too many profitable trades.",
        "main_interpretation": (
            "Champion model forecasts forward realized variance better, but high forecast realized volatility "
            "does not imply short-vol trades are unattractive if implied volatility is sufficiently rich."
        ),
    },

    "current_policy": {
        "base_phase3_signal": "unchanged",
        "champion_forecast_role": "parallel diagnostic denominator",
        "future_research": "model disagreement / regime context, not hard filter",
    },
}

with open(CHAMPION_SIGNAL_DIAGNOSTIC_CONFIG_JSON, "w") as f:
    json.dump(champion_signal_diagnostic_config, f, indent=4)

champion_signal_diagnostic_summary_df = pd.DataFrame([
    {
        "test": "champion_signal_gt_0",
        "finding": "Did not improve accepted trade selection; removed trades were profitable.",
        "action": "Do not use as hard filter.",
    },
    {
        "test": "champion_signal_gt_0p25",
        "finding": "Too restrictive and removed too much good P&L.",
        "action": "Do not use as hard filter.",
    },
    {
        "test": "champion_min_z_gt_0",
        "finding": "Useful Core quality tag but too restrictive for full strategy.",
        "action": "Display as quality/context metric.",
    },
    {
        "test": "negative champion signal",
        "finding": "Accepted trades with negative champion signal were profitable.",
        "action": "Do not suppress solely because champion signal is negative.",
    },
])

champion_signal_diagnostic_summary_df.to_csv(
    CHAMPION_SIGNAL_DIAGNOSTIC_SUMMARY_CSV,
    index=False,
)

print("Saved champion signal diagnostic config:")
print(CHAMPION_SIGNAL_DIAGNOSTIC_CONFIG_JSON)

print("\nSaved champion signal diagnostic summary:")
print(CHAMPION_SIGNAL_DIAGNOSTIC_SUMMARY_CSV)

display(champion_signal_diagnostic_summary_df)

Saved champion signal diagnostic config:
C:\Users\patri\vrp_project\data\audit\forecast_vol_champion_signal_diagnostic_config_v0_1.json

Saved champion signal diagnostic summary:
C:\Users\patri\vrp_project\data\audit\forecast_vol_champion_signal_diagnostic_summary_v0_1.csv


,test,finding,action
0,champion_signal_gt_0,Did not improve accepted trade selection; remo...,Do not use as hard filter.
1,champion_signal_gt_0p25,Too restrictive and removed too much good P&L.,Do not use as hard filter.
2,champion_min_z_gt_0,Useful Core quality tag but too restrictive fo...,Display as quality/context metric.
3,negative champion signal,Accepted trades with negative champion signal ...,Do not suppress solely because champion signal...


In [21]:
# ============================================================
# Cell 20: Inspect inputs for champion-model recalibration
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

PUT_CANDIDATE_UNIVERSE_PRICING_PARQUET = (
    PROCESSED_DATA_DIR / "put_candidate_universe_pricing_v0_1.parquet"
)

CHAMPION_FORECAST_VRP_ZSCORE_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vrp_champion_zscore_panel_v0_1.parquet"
)

CURRENT_FORECAST_VRP_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vrp_panel_v0_1.parquet"
)

CURRENT_FORECAST_VRP_ZSCORE_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vrp_zscore_panel_v0_1.parquet"
)

TRAILING_VRP_ZSCORE_PARQUET = (
    PROCESSED_DATA_DIR / "vrp_zscore_panel_v0_1.parquet"
)

input_paths = [
    PUT_CANDIDATE_UNIVERSE_PRICING_PARQUET,
    CHAMPION_FORECAST_VRP_ZSCORE_PARQUET,
    CURRENT_FORECAST_VRP_PANEL_PARQUET,
    CURRENT_FORECAST_VRP_ZSCORE_PARQUET,
    TRAILING_VRP_ZSCORE_PARQUET,
]

print("Input file check:")
for path in input_paths:
    print(("OK      " if path.exists() else "MISSING ") + str(path))

put_universe_df = pd.read_parquet(PUT_CANDIDATE_UNIVERSE_PRICING_PARQUET).copy()
champion_z_df = pd.read_parquet(CHAMPION_FORECAST_VRP_ZSCORE_PARQUET).copy()

print("\nPut candidate universe:")
print("Rows:", len(put_universe_df))
print("Columns:", len(put_universe_df.columns))

print("\nPut universe columns:")
for col in put_universe_df.columns:
    print(repr(col))

print("\nUseful-looking put universe columns:")
for col in put_universe_df.columns:
    col_l = col.lower()
    if any(tok in col_l for tok in [
        "trade", "tenor", "tier", "group", "signal", "z", "vrp",
        "rsi", "pnl", "win", "credit", "strike", "dte", "expiry",
        "forecast", "realized", "variance", "vol",
    ]):
        print(f"{repr(col):55s} {put_universe_df[col].dtype}")

print("\nChampion z-score panel:")
print("Rows:", len(champion_z_df))
print("Columns:", len(champion_z_df.columns))

print("\nChampion useful columns:")
for col in champion_z_df.columns:
    col_l = col.lower()
    if any(tok in col_l for tok in ["trade", "tenor", "champion", "vrp", "z", "forecast", "vol", "variance"]):
        print(f"{repr(col):60s} {champion_z_df[col].dtype}")

print("\nPut universe first row transposed:")
display(put_universe_df.head(1).T)

Input file check:
OK      C:\Users\patri\vrp_project\data\processed\put_candidate_universe_pricing_v0_1.parquet
OK      C:\Users\patri\vrp_project\data\processed\forecast_vrp_champion_zscore_panel_v0_1.parquet
OK      C:\Users\patri\vrp_project\data\processed\forecast_vrp_panel_v0_1.parquet
OK      C:\Users\patri\vrp_project\data\processed\forecast_vrp_zscore_panel_v0_1.parquet
OK      C:\Users\patri\vrp_project\data\processed\vrp_zscore_panel_v0_1.parquet

Put candidate universe:
Rows: 18099
Columns: 55

Put universe columns:
'trade_date'
'tenor'
'spx_close'
'spx_rsi_14'
'vix_style_vol'
'implied_variance'
'old_vrp_signal'
'old_vol'
'old_vrp_3m_z'
'old_vrp_1y_z'
'old_vrp_blended_z'
'simple_forecast_model'
'simple_forecast_variance'
'forecast_vol'
'forecast_vrp_signal'
'forecast_vrp_3m_z'
'forecast_vrp_1y_z'
'forecast_vrp_blended_z'
'target_tenor'
'side'
'source_universe'
'trade_date_ts'
'target_expiry_date'
'entry_spx_close'
'tenor_group'
'trade_year'
'expiry_mapping_status'
'selected_

,0
trade_date,20180625
tenor,9
spx_close,2717.07
spx_rsi_14,41.761584
vix_style_vol,17.348587
implied_variance,0.030097
old_vrp_signal,1.05393
old_vol,10.242504
old_vrp_3m_z,NaN
old_vrp_1y_z,NaN


In [22]:
# ============================================================
# Cell 21: Build champion recalibration universe
# ============================================================

import numpy as np
import pandas as pd

PUT_CHAMPION_RECALIBRATION_UNIVERSE_PARQUET = (
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet"
)

PUT_CHAMPION_RECALIBRATION_UNIVERSE_CSV = (
    AUDIT_DIR / "put_champion_recalibration_universe_sample_v0_1.csv"
)

put_df = pd.read_parquet(PUT_CANDIDATE_UNIVERSE_PRICING_PARQUET).copy()
champion_z_df = pd.read_parquet(CHAMPION_FORECAST_VRP_ZSCORE_PARQUET).copy()

def parse_trade_dt_from_trade_date(s):
    return pd.to_datetime(
        s.astype(str).str.replace(r"\.0$", "", regex=True),
        format="%Y%m%d",
        errors="coerce",
    )

def ensure_trade_dt(df):
    df = df.copy()

    if "trade_dt" in df.columns:
        df["trade_dt"] = pd.to_datetime(df["trade_dt"], errors="coerce")
    elif "trade_date_ts" in df.columns:
        df["trade_dt"] = pd.to_datetime(df["trade_date_ts"], errors="coerce")
    elif "trade_date" in df.columns:
        df["trade_dt"] = parse_trade_dt_from_trade_date(df["trade_date"])
    else:
        raise ValueError("Could not find trade date column.")

    return df

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

put_df = ensure_trade_dt(put_df)
champion_z_df = ensure_trade_dt(champion_z_df)

# Normalize tenor.
if "target_tenor" in put_df.columns:
    put_df["entry_tenor"] = put_df["target_tenor"].astype(int)
elif "tenor" in put_df.columns:
    put_df["entry_tenor"] = put_df["tenor"].astype(int)
else:
    raise ValueError("Could not find target_tenor or tenor in put universe.")

if "tenor" not in champion_z_df.columns:
    raise ValueError("Champion z panel does not have tenor column.")

champion_z_df["tenor"] = champion_z_df["tenor"].astype(int)

# Merge champion model signal.
champion_cols = [
    "trade_dt",
    "tenor",
    "champion_forecast_vol",
    "champion_forecast_vrp_signal",
    "champion_forecast_vrp_signal_z_3m",
    "champion_forecast_vrp_signal_z_1y",
    "champion_forecast_vrp_min_z",
]

missing_champion_cols = [c for c in champion_cols if c not in champion_z_df.columns]
if missing_champion_cols:
    raise ValueError(f"Missing champion columns: {missing_champion_cols}")

recal_df = put_df.merge(
    champion_z_df[champion_cols],
    left_on=["trade_dt", "entry_tenor"],
    right_on=["trade_dt", "tenor"],
    how="left",
    suffixes=("", "_champion_panel"),
)

# Detect current/simple forecast signal.
simple_signal_col = first_existing(
    recal_df,
    [
        "simple_forecast_vrp_signal",
        "forecast_vrp_signal",
        "current_forecast_vrp_signal",
        "ewma_10_scheduled_vrp_signal",
        "ewma_10_scheduled_bias_adjusted_vrp_signal",
    ],
)

if simple_signal_col is None:
    # Fall back to merging current forecast VRP panel.
    current_forecast_vrp_df = pd.read_parquet(CURRENT_FORECAST_VRP_PANEL_PARQUET).copy()
    current_forecast_vrp_df = ensure_trade_dt(current_forecast_vrp_df)

    if "tenor" not in current_forecast_vrp_df.columns:
        if "target_tenor" in current_forecast_vrp_df.columns:
            current_forecast_vrp_df["tenor"] = current_forecast_vrp_df["target_tenor"]
        elif "target_days" in current_forecast_vrp_df.columns:
            current_forecast_vrp_df["tenor"] = current_forecast_vrp_df["target_days"]
        else:
            raise ValueError("Could not find tenor in current forecast VRP panel.")

    current_forecast_vrp_df["tenor"] = current_forecast_vrp_df["tenor"].astype(int)

    if "simple_forecast_vrp_signal" not in current_forecast_vrp_df.columns:
        raise ValueError("Could not find simple_forecast_vrp_signal in current forecast VRP panel.")

    recal_df = recal_df.merge(
        current_forecast_vrp_df[
            [
                "trade_dt",
                "tenor",
                "simple_forecast_vrp_signal",
                "simple_forecast_vol",
                "simple_forecast_variance",
            ]
        ],
        left_on=["trade_dt", "entry_tenor"],
        right_on=["trade_dt", "tenor"],
        how="left",
        suffixes=("", "_current_forecast"),
    )

    simple_signal_col = "simple_forecast_vrp_signal"

recal_df["simple_carry_vrp_signal"] = recal_df[simple_signal_col]

# Detect old/trailing VRP signal.
old_signal_col = first_existing(
    recal_df,
    [
        "primary_vrp_signal",
        "vrp_trailing_log_variance_ratio",
        "old_vrp_signal",
        "trailing_vrp_signal",
    ],
)

if old_signal_col is None:
    print("Warning: could not find old/trailing VRP signal in put universe.")
    recal_df["trailing_carry_vrp_signal"] = np.nan
else:
    recal_df["trailing_carry_vrp_signal"] = recal_df[old_signal_col]

# Detect RSI.
rsi_col = first_existing(
    recal_df,
    [
        "spx_rsi_14",
        "rsi_14",
        "RSI_14",
    ],
)

if rsi_col is None:
    raise ValueError("Could not find RSI column.")

recal_df["entry_rsi_14"] = recal_df[rsi_col]

# Detect P&L.
pnl_col = first_existing(
    recal_df,
    [
        "weighted_pnl_dollars",
        "normalized_pnl_dollars",
        "actual_pnl_pct_of_nw",
        "normalized_pnl_pct",
    ],
)

if pnl_col is None:
    raise ValueError("Could not find P&L column.")

recal_df["test_pnl"] = recal_df[pnl_col]
recal_df["test_pnl_source_col"] = pnl_col

# Detect win column or create one.
if "win" in recal_df.columns:
    recal_df["test_win"] = recal_df["win"].astype(bool)
else:
    recal_df["test_win"] = recal_df["test_pnl"] > 0

# Tenor group.
if "tenor_group" not in recal_df.columns:
    def tenor_group_from_tenor(t):
        t = int(t)
        if t in [9, 12, 15]:
            return "front_9_15d"
        if t in [18, 21, 24]:
            return "middle_18_24d"
        if t in [27, 30, 33]:
            return "back_27_33d"
        return "other"

    recal_df["tenor_group"] = recal_df["entry_tenor"].map(tenor_group_from_tenor)

# Compute rolling simple z-scores directly from the universe.
# Use one signal row per date/tenor.
signal_base = (
    recal_df[
        [
            "trade_dt",
            "entry_tenor",
            "simple_carry_vrp_signal",
            "trailing_carry_vrp_signal",
        ]
    ]
    .drop_duplicates(["trade_dt", "entry_tenor"])
    .sort_values(["entry_tenor", "trade_dt"])
    .copy()
)

g = signal_base.groupby("entry_tenor", group_keys=False)

for signal_col in ["simple_carry_vrp_signal", "trailing_carry_vrp_signal"]:
    for window, label in [(63, "3m"), (252, "1y")]:
        mean_col = f"{signal_col}_mean_{label}"
        std_col = f"{signal_col}_std_{label}"
        z_col = f"{signal_col}_z_{label}"

        signal_base[mean_col] = (
            g[signal_col]
            .rolling(window, min_periods=max(20, window // 3))
            .mean()
            .reset_index(level=0, drop=True)
        )

        signal_base[std_col] = (
            g[signal_col]
            .rolling(window, min_periods=max(20, window // 3))
            .std()
            .reset_index(level=0, drop=True)
        )

        signal_base[z_col] = (
            (signal_base[signal_col] - signal_base[mean_col])
            / signal_base[std_col].replace(0, np.nan)
        )

signal_base["simple_carry_vrp_min_z"] = signal_base[
    [
        "simple_carry_vrp_signal_z_3m",
        "simple_carry_vrp_signal_z_1y",
    ]
].min(axis=1)

signal_base["trailing_carry_vrp_min_z"] = signal_base[
    [
        "trailing_carry_vrp_signal_z_3m",
        "trailing_carry_vrp_signal_z_1y",
    ]
].min(axis=1)

z_cols_to_merge = [
    "trade_dt",
    "entry_tenor",
    "simple_carry_vrp_signal_z_3m",
    "simple_carry_vrp_signal_z_1y",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal_z_3m",
    "trailing_carry_vrp_signal_z_1y",
    "trailing_carry_vrp_min_z",
]

recal_df = recal_df.merge(
    signal_base[z_cols_to_merge],
    on=["trade_dt", "entry_tenor"],
    how="left",
)

# Model disagreement.
recal_df["simple_minus_champion_signal"] = (
    recal_df["simple_carry_vrp_signal"]
    - recal_df["champion_forecast_vrp_signal"]
)

recal_df["both_simple_and_champion_rich_flag"] = (
    (recal_df["simple_carry_vrp_signal"] > 0)
    & (recal_df["champion_forecast_vrp_signal"] > 0)
)

recal_df["simple_rich_champion_weak_flag"] = (
    (recal_df["simple_carry_vrp_signal"] > 0)
    & (recal_df["champion_forecast_vrp_signal"] <= 0)
)

recal_df["champion_rich_simple_weak_flag"] = (
    (recal_df["champion_forecast_vrp_signal"] > 0)
    & (recal_df["simple_carry_vrp_signal"] <= 0)
)

recal_df = recal_df.sort_values(["trade_dt", "entry_tenor"]).reset_index(drop=True)

recal_df.to_parquet(PUT_CHAMPION_RECALIBRATION_UNIVERSE_PARQUET, index=False)
recal_df.head(2000).to_csv(PUT_CHAMPION_RECALIBRATION_UNIVERSE_CSV, index=False)

print("Saved recalibration universe:")
print(PUT_CHAMPION_RECALIBRATION_UNIVERSE_PARQUET)
print(PUT_CHAMPION_RECALIBRATION_UNIVERSE_CSV)

print("\nRows:", len(recal_df))
print("Dates:", recal_df["trade_dt"].min(), "to", recal_df["trade_dt"].max())
print("Tenors:", sorted(recal_df["entry_tenor"].dropna().unique()))
print("P&L column used:", pnl_col)
print("Simple signal column used:", simple_signal_col)
print("Old/trailing signal column used:", old_signal_col)
print("RSI column used:", rsi_col)

print("\nKey missingness:")
key_cols = [
    "simple_carry_vrp_signal",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_min_z",
    "champion_forecast_vrp_signal",
    "champion_forecast_vrp_min_z",
    "entry_rsi_14",
    "test_pnl",
]

display(
    recal_df[key_cols]
    .isna()
    .mean()
    .to_frame("missing_pct")
)

print("\nSignal summary:")
display(
    recal_df[
        [
            "simple_carry_vrp_signal",
            "simple_carry_vrp_min_z",
            "trailing_carry_vrp_signal",
            "trailing_carry_vrp_min_z",
            "champion_forecast_vrp_signal",
            "champion_forecast_vrp_min_z",
            "simple_minus_champion_signal",
            "test_pnl",
        ]
    ].describe().T
)

print("\nSample:")
display(
    recal_df[
        [
            "trade_dt",
            "entry_tenor",
            "tenor_group",
            "entry_rsi_14",
            "simple_carry_vrp_signal",
            "simple_carry_vrp_min_z",
            "champion_forecast_vrp_signal",
            "champion_forecast_vrp_min_z",
            "simple_minus_champion_signal",
            "test_pnl",
            "test_win",
        ]
    ].tail(20)
)

Saved recalibration universe:
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_universe_v0_1.parquet
C:\Users\patri\vrp_project\data\audit\put_champion_recalibration_universe_sample_v0_1.csv

Rows: 18099
Dates: 2018-06-25 00:00:00 to 2026-06-25 00:00:00
Tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]
P&L column used: normalized_pnl_dollars
Simple signal column used: forecast_vrp_signal
Old/trailing signal column used: old_vrp_signal
RSI column used: spx_rsi_14

Key missingness:


,missing_pct
simple_carry_vrp_signal,0.004475
simple_carry_vrp_min_z,0.014421
trailing_carry_vrp_signal,0.000000
trailing_carry_vrp_min_z,0.009945
champion_forecast_vrp_signal,0.322615
champion_forecast_vrp_min_z,0.332560
entry_rsi_14,0.000000
test_pnl,0.008454



Signal summary:


,count,mean,std,min,25%,50%,75%,max
simple_carry_vrp_signal,18018.0,0.534672,0.526231,-1.217596,0.218456,0.536724,0.867115,2.581601
simple_carry_vrp_min_z,17838.0,-0.190380,1.139128,-4.277639,-0.979903,-0.193852,0.639771,3.475335
trailing_carry_vrp_signal,18099.0,0.565564,0.636239,-1.669171,0.148520,0.532452,0.985749,3.581716
trailing_carry_vrp_min_z,17919.0,-0.188245,1.087746,-4.767962,-0.924054,-0.198393,0.589688,4.208888
champion_forecast_vrp_signal,12260.0,0.496051,0.213728,-0.601171,0.380700,0.503858,0.626933,1.637116
champion_forecast_vrp_min_z,12080.0,-0.250500,1.020493,-6.587806,-0.915791,-0.223312,0.415318,3.049510
simple_minus_champion_signal,12260.0,0.113332,0.565944,-1.651761,-0.259134,0.089113,0.425569,2.780485
test_pnl,17946.0,2892.945729,27236.650997,-306182.537690,1562.045448,10148.636050,15129.095290,85703.628889



Sample:


,trade_dt,entry_tenor,tenor_group,entry_rsi_14,simple_carry_vrp_signal,simple_carry_vrp_min_z,champion_forecast_vrp_signal,champion_forecast_vrp_min_z,simple_minus_champion_signal,test_pnl,test_win
18079,2026-06-23,30,back_27_33d,46.752076,0.313995,-1.266877,NaN,NaN,NaN,NaN,False
18080,2026-06-23,33,back_27_33d,46.752076,0.377271,-1.208011,NaN,NaN,NaN,NaN,False
18081,2026-06-24,9,front_9_15d,46.342083,0.669279,-0.410467,NaN,NaN,NaN,NaN,False
18082,2026-06-24,12,front_9_15d,46.342083,0.739274,-0.375552,NaN,NaN,NaN,NaN,False
18083,2026-06-24,15,front_9_15d,46.342083,0.550836,-0.696963,NaN,NaN,NaN,NaN,False
18084,2026-06-24,18,middle_18_24d,46.342083,0.621234,-0.638722,NaN,NaN,NaN,NaN,False
18085,2026-06-24,21,middle_18_24d,46.342083,0.395771,-0.840562,NaN,NaN,NaN,NaN,False
18086,2026-06-24,24,middle_18_24d,46.342083,0.407037,-0.902639,NaN,NaN,NaN,NaN,False
18087,2026-06-24,27,back_27_33d,46.342083,0.415448,-0.978299,NaN,NaN,NaN,NaN,False
18088,2026-06-24,30,back_27_33d,46.342083,0.373337,-1.107308,NaN,NaN,NaN,NaN,False


In [15]:
# ============================================================
# Cell 22: First-pass champion recalibration sweep
# Signal-quality sweep, one selected tenor per trade date
# ============================================================

import numpy as np
import pandas as pd
from itertools import product

PUT_CHAMPION_RECALIBRATION_UNIVERSE_PARQUET = (
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet"
)

CHAMPION_RECAL_SWEEP_CSV = (
    AUDIT_DIR / "put_champion_recalibration_sweep_v0_1.csv"
)

CHAMPION_RECAL_SWEEP_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_champion_recalibration_sweep_selected_trades_v0_1.parquet"
)

recal_df = pd.read_parquet(PUT_CHAMPION_RECALIBRATION_UNIVERSE_PARQUET).copy()
recal_df["trade_dt"] = pd.to_datetime(recal_df["trade_dt"])
recal_df["entry_tenor"] = recal_df["entry_tenor"].astype(int)

# Restrict to common sample where all signals and P&L are known.
required_cols = [
    "test_pnl",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_min_z",
    "champion_forecast_vrp_signal",
    "champion_forecast_vrp_min_z",
    "entry_rsi_14",
]

clean_df = recal_df.dropna(subset=required_cols).copy()

clean_df = clean_df.sort_values(["trade_dt", "entry_tenor"]).reset_index(drop=True)

clean_df["trade_year"] = clean_df["trade_dt"].dt.year

total_trade_dates = clean_df["trade_dt"].nunique()

print("Clean recalibration sample:")
print("Rows:", len(clean_df))
print("Dates:", clean_df["trade_dt"].min(), "to", clean_df["trade_dt"].max())
print("Unique trade dates:", total_trade_dates)
print("Tenors:", sorted(clean_df["entry_tenor"].unique()))

display(
    clean_df[
        [
            "test_pnl",
            "simple_carry_vrp_signal",
            "simple_carry_vrp_min_z",
            "trailing_carry_vrp_signal",
            "trailing_carry_vrp_min_z",
            "champion_forecast_vrp_signal",
            "champion_forecast_vrp_min_z",
            "entry_rsi_14",
        ]
    ].describe().T
)


def max_drawdown_from_pnl(pnl_series):
    pnl = pd.Series(pnl_series).fillna(0.0)
    equity = pnl.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    return float(dd.min())


def summarize_selected_trades(selected_df, rule_name, params):
    selected_df = selected_df.sort_values("trade_dt").copy()

    if selected_df.empty:
        return {
            "rule_name": rule_name,
            **params,
            "trades": 0,
            "frequency": 0.0,
            "total_pnl": 0.0,
            "avg_pnl": np.nan,
            "median_pnl": np.nan,
            "win_rate": np.nan,
            "q05_pnl": np.nan,
            "worst_pnl": np.nan,
            "max_drawdown": np.nan,
            "worst_10_trade_sum": np.nan,
            "worst_20_trade_sum": np.nan,
            "positive_year_rate": np.nan,
            "min_year_pnl": np.nan,
            "avg_rsi": np.nan,
            "avg_tenor": np.nan,
            "front_pct": np.nan,
            "middle_pct": np.nan,
            "back_pct": np.nan,
        }

    pnl = selected_df["test_pnl"].astype(float)

    yearly = selected_df.groupby("trade_year")["test_pnl"].sum()

    roll10 = pnl.rolling(10).sum()
    roll20 = pnl.rolling(20).sum()

    return {
        "rule_name": rule_name,
        **params,
        "trades": len(selected_df),
        "frequency": len(selected_df) / total_trade_dates,
        "total_pnl": pnl.sum(),
        "avg_pnl": pnl.mean(),
        "median_pnl": pnl.median(),
        "win_rate": (pnl > 0).mean(),
        "q05_pnl": pnl.quantile(0.05),
        "worst_pnl": pnl.min(),
        "max_drawdown": max_drawdown_from_pnl(pnl),
        "worst_10_trade_sum": roll10.min(),
        "worst_20_trade_sum": roll20.min(),
        "positive_year_rate": (yearly > 0).mean(),
        "min_year_pnl": yearly.min(),
        "avg_rsi": selected_df["entry_rsi_14"].mean(),
        "avg_tenor": selected_df["entry_tenor"].mean(),
        "front_pct": selected_df["tenor_group"].eq("front_9_15d").mean(),
        "middle_pct": selected_df["tenor_group"].eq("middle_18_24d").mean(),
        "back_pct": selected_df["tenor_group"].eq("back_27_33d").mean(),
    }


def select_one_per_day(candidate_df, score_col):
    if candidate_df.empty:
        return candidate_df.copy()

    # Highest score first, then longer tenor as tie-breaker.
    selected = (
        candidate_df
        .sort_values(
            ["trade_dt", score_col, "entry_tenor"],
            ascending=[True, False, False],
        )
        .groupby("trade_dt", as_index=False)
        .head(1)
        .copy()
    )

    return selected.sort_values("trade_dt").reset_index(drop=True)


def add_score_columns(df):
    out = df.copy()

    out["score_simple"] = (
        out["simple_carry_vrp_signal"]
        + 0.50 * out["simple_carry_vrp_min_z"]
    )

    out["score_champion"] = (
        out["champion_forecast_vrp_signal"]
        + 0.50 * out["champion_forecast_vrp_min_z"]
    )

    out["score_old_simple"] = (
        np.minimum(
            out["trailing_carry_vrp_signal"],
            out["simple_carry_vrp_signal"],
        )
        + 0.50 * np.minimum(
            out["trailing_carry_vrp_min_z"],
            out["simple_carry_vrp_min_z"],
        )
    )

    out["score_old_champion"] = (
        np.minimum(
            out["trailing_carry_vrp_signal"],
            out["champion_forecast_vrp_signal"],
        )
        + 0.50 * np.minimum(
            out["trailing_carry_vrp_min_z"],
            out["champion_forecast_vrp_min_z"],
        )
    )

    out["score_simple_champion"] = (
        np.minimum(
            out["simple_carry_vrp_signal"],
            out["champion_forecast_vrp_signal"],
        )
        + 0.50 * np.minimum(
            out["simple_carry_vrp_min_z"],
            out["champion_forecast_vrp_min_z"],
        )
    )

    out["score_three_signal"] = (
        np.minimum.reduce([
            out["trailing_carry_vrp_signal"],
            out["simple_carry_vrp_signal"],
            out["champion_forecast_vrp_signal"],
        ])
        + 0.50 * np.minimum.reduce([
            out["trailing_carry_vrp_min_z"],
            out["simple_carry_vrp_min_z"],
            out["champion_forecast_vrp_min_z"],
        ])
    )

    out["score_disagreement_carry"] = (
        out["simple_carry_vrp_signal"]
        + 0.25 * out["trailing_carry_vrp_signal"]
        - 0.10 * np.maximum(-out["champion_forecast_vrp_signal"], 0)
    )

    return out


clean_df = add_score_columns(clean_df)

# Broad, non-optimized grids.
rsi_caps = [58, 62, 68, 72, 77]

simple_signal_thresholds = [0.40, 0.50, 0.60, 0.75, 0.90, 1.10]
champion_signal_thresholds = [0.00, 0.25, 0.40, 0.50, 0.60, 0.75]
old_signal_thresholds = [0.50, 0.70, 0.90, 1.10]

simple_z_thresholds = [-0.50, 0.00, 0.30, 0.50, 0.70, 1.00]
champion_z_thresholds = [-1.00, -0.50, 0.00, 0.30, 0.50, 0.70]
old_z_thresholds = [0.00, 0.30, 0.50, 0.70, 1.00]

summary_rows = []
selected_trade_blocks = []

rule_id = 0

# ------------------------------------------------------------
# Family A: simple-only carry signal
# ------------------------------------------------------------

for rsi_cap, sig_thr, z_thr in product(
    rsi_caps,
    simple_signal_thresholds,
    simple_z_thresholds,
):
    mask = (
        (clean_df["entry_rsi_14"] < rsi_cap)
        & (clean_df["simple_carry_vrp_signal"] > sig_thr)
        & (clean_df["simple_carry_vrp_min_z"] > z_thr)
    )

    candidates = clean_df[mask].copy()
    selected = select_one_per_day(candidates, "score_simple")

    params = {
        "rule_id": rule_id,
        "rsi_cap": rsi_cap,
        "simple_signal_threshold": sig_thr,
        "simple_z_threshold": z_thr,
        "champion_signal_threshold": np.nan,
        "champion_z_threshold": np.nan,
        "old_signal_threshold": np.nan,
        "old_z_threshold": np.nan,
    }

    summary_rows.append(summarize_selected_trades(selected, "simple_only", params))

    selected["rule_id"] = rule_id
    selected["rule_name"] = "simple_only"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Family B: champion-only model-expected VRP signal
# ------------------------------------------------------------

for rsi_cap, sig_thr, z_thr in product(
    rsi_caps,
    champion_signal_thresholds,
    champion_z_thresholds,
):
    mask = (
        (clean_df["entry_rsi_14"] < rsi_cap)
        & (clean_df["champion_forecast_vrp_signal"] > sig_thr)
        & (clean_df["champion_forecast_vrp_min_z"] > z_thr)
    )

    candidates = clean_df[mask].copy()
    selected = select_one_per_day(candidates, "score_champion")

    params = {
        "rule_id": rule_id,
        "rsi_cap": rsi_cap,
        "simple_signal_threshold": np.nan,
        "simple_z_threshold": np.nan,
        "champion_signal_threshold": sig_thr,
        "champion_z_threshold": z_thr,
        "old_signal_threshold": np.nan,
        "old_z_threshold": np.nan,
    }

    summary_rows.append(summarize_selected_trades(selected, "champion_only", params))

    selected["rule_id"] = rule_id
    selected["rule_name"] = "champion_only"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Family C: current-style old + simple confirmation
# ------------------------------------------------------------

for rsi_cap, old_sig_thr, old_z_thr, simple_sig_thr, simple_z_thr in product(
    rsi_caps,
    old_signal_thresholds,
    old_z_thresholds,
    simple_signal_thresholds,
    simple_z_thresholds,
):
    mask = (
        (clean_df["entry_rsi_14"] < rsi_cap)
        & (clean_df["trailing_carry_vrp_signal"] > old_sig_thr)
        & (clean_df["trailing_carry_vrp_min_z"] > old_z_thr)
        & (clean_df["simple_carry_vrp_signal"] > simple_sig_thr)
        & (clean_df["simple_carry_vrp_min_z"] > simple_z_thr)
    )

    candidates = clean_df[mask].copy()
    selected = select_one_per_day(candidates, "score_old_simple")

    params = {
        "rule_id": rule_id,
        "rsi_cap": rsi_cap,
        "simple_signal_threshold": simple_sig_thr,
        "simple_z_threshold": simple_z_thr,
        "champion_signal_threshold": np.nan,
        "champion_z_threshold": np.nan,
        "old_signal_threshold": old_sig_thr,
        "old_z_threshold": old_z_thr,
    }

    summary_rows.append(summarize_selected_trades(selected, "old_simple_confirm", params))

    selected["rule_id"] = rule_id
    selected["rule_name"] = "old_simple_confirm"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Family D: old + champion confirmation
# ------------------------------------------------------------

for rsi_cap, old_sig_thr, old_z_thr, champ_sig_thr, champ_z_thr in product(
    rsi_caps,
    old_signal_thresholds,
    old_z_thresholds,
    champion_signal_thresholds,
    champion_z_thresholds,
):
    mask = (
        (clean_df["entry_rsi_14"] < rsi_cap)
        & (clean_df["trailing_carry_vrp_signal"] > old_sig_thr)
        & (clean_df["trailing_carry_vrp_min_z"] > old_z_thr)
        & (clean_df["champion_forecast_vrp_signal"] > champ_sig_thr)
        & (clean_df["champion_forecast_vrp_min_z"] > champ_z_thr)
    )

    candidates = clean_df[mask].copy()
    selected = select_one_per_day(candidates, "score_old_champion")

    params = {
        "rule_id": rule_id,
        "rsi_cap": rsi_cap,
        "simple_signal_threshold": np.nan,
        "simple_z_threshold": np.nan,
        "champion_signal_threshold": champ_sig_thr,
        "champion_z_threshold": champ_z_thr,
        "old_signal_threshold": old_sig_thr,
        "old_z_threshold": old_z_thr,
    }

    summary_rows.append(summarize_selected_trades(selected, "old_champion_confirm", params))

    selected["rule_id"] = rule_id
    selected["rule_name"] = "old_champion_confirm"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Family E: simple + champion confirmation
# ------------------------------------------------------------

for rsi_cap, simple_sig_thr, simple_z_thr, champ_sig_thr, champ_z_thr in product(
    rsi_caps,
    simple_signal_thresholds,
    simple_z_thresholds,
    champion_signal_thresholds,
    champion_z_thresholds,
):
    mask = (
        (clean_df["entry_rsi_14"] < rsi_cap)
        & (clean_df["simple_carry_vrp_signal"] > simple_sig_thr)
        & (clean_df["simple_carry_vrp_min_z"] > simple_z_thr)
        & (clean_df["champion_forecast_vrp_signal"] > champ_sig_thr)
        & (clean_df["champion_forecast_vrp_min_z"] > champ_z_thr)
    )

    candidates = clean_df[mask].copy()
    selected = select_one_per_day(candidates, "score_simple_champion")

    params = {
        "rule_id": rule_id,
        "rsi_cap": rsi_cap,
        "simple_signal_threshold": simple_sig_thr,
        "simple_z_threshold": simple_z_thr,
        "champion_signal_threshold": champ_sig_thr,
        "champion_z_threshold": champ_z_thr,
        "old_signal_threshold": np.nan,
        "old_z_threshold": np.nan,
    }

    summary_rows.append(summarize_selected_trades(selected, "simple_champion_confirm", params))

    selected["rule_id"] = rule_id
    selected["rule_name"] = "simple_champion_confirm"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Family F: three-signal confirmation
# ------------------------------------------------------------

for rsi_cap, old_sig_thr, simple_sig_thr, champ_sig_thr, z_floor in product(
    rsi_caps,
    [0.50, 0.70, 0.90],
    [0.40, 0.60, 0.75],
    [0.00, 0.25, 0.40, 0.50],
    [-0.50, 0.00, 0.30, 0.50],
):
    mask = (
        (clean_df["entry_rsi_14"] < rsi_cap)
        & (clean_df["trailing_carry_vrp_signal"] > old_sig_thr)
        & (clean_df["simple_carry_vrp_signal"] > simple_sig_thr)
        & (clean_df["champion_forecast_vrp_signal"] > champ_sig_thr)
        & (clean_df["trailing_carry_vrp_min_z"] > z_floor)
        & (clean_df["simple_carry_vrp_min_z"] > z_floor)
        & (clean_df["champion_forecast_vrp_min_z"] > z_floor)
    )

    candidates = clean_df[mask].copy()
    selected = select_one_per_day(candidates, "score_three_signal")

    params = {
        "rule_id": rule_id,
        "rsi_cap": rsi_cap,
        "simple_signal_threshold": simple_sig_thr,
        "simple_z_threshold": z_floor,
        "champion_signal_threshold": champ_sig_thr,
        "champion_z_threshold": z_floor,
        "old_signal_threshold": old_sig_thr,
        "old_z_threshold": z_floor,
    }

    summary_rows.append(summarize_selected_trades(selected, "three_signal_confirm", params))

    selected["rule_id"] = rule_id
    selected["rule_name"] = "three_signal_confirm"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Family G: carry-rich, champion-not-hostile
# This preserves post-vol-shock carry setups.
# ------------------------------------------------------------

for rsi_cap, simple_sig_thr, simple_z_thr, champ_floor, champ_z_floor in product(
    rsi_caps,
    simple_signal_thresholds,
    simple_z_thresholds,
    [-0.50, -0.25, 0.00, 0.25],
    [-2.00, -1.50, -1.00, -0.50, 0.00],
):
    mask = (
        (clean_df["entry_rsi_14"] < rsi_cap)
        & (clean_df["simple_carry_vrp_signal"] > simple_sig_thr)
        & (clean_df["simple_carry_vrp_min_z"] > simple_z_thr)
        & (clean_df["champion_forecast_vrp_signal"] > champ_floor)
        & (clean_df["champion_forecast_vrp_min_z"] > champ_z_floor)
    )

    candidates = clean_df[mask].copy()
    selected = select_one_per_day(candidates, "score_disagreement_carry")

    params = {
        "rule_id": rule_id,
        "rsi_cap": rsi_cap,
        "simple_signal_threshold": simple_sig_thr,
        "simple_z_threshold": simple_z_thr,
        "champion_signal_threshold": champ_floor,
        "champion_z_threshold": champ_z_floor,
        "old_signal_threshold": np.nan,
        "old_z_threshold": np.nan,
    }

    summary_rows.append(summarize_selected_trades(selected, "simple_rich_champion_not_hostile", params))

    selected["rule_id"] = rule_id
    selected["rule_name"] = "simple_rich_champion_not_hostile"
    selected_trade_blocks.append(selected)

    rule_id += 1


sweep_df = pd.DataFrame(summary_rows)

sweep_df = sweep_df.sort_values(
    ["total_pnl", "avg_pnl", "win_rate"],
    ascending=[False, False, False],
).reset_index(drop=True)

selected_sweep_trades_df = pd.concat(
    selected_trade_blocks,
    ignore_index=True,
) if selected_trade_blocks else pd.DataFrame()

sweep_df.to_csv(CHAMPION_RECAL_SWEEP_CSV, index=False)
selected_sweep_trades_df.to_parquet(CHAMPION_RECAL_SWEEP_TRADES_PARQUET, index=False)

print("Saved champion recalibration sweep:")
print(CHAMPION_RECAL_SWEEP_CSV)
print(CHAMPION_RECAL_SWEEP_TRADES_PARQUET)

print("\nSweep rows:", len(sweep_df))
print("Selected trade rows:", len(selected_sweep_trades_df))

print("\nTop 30 by total P&L, requiring at least 100 trades:")
display(
    sweep_df
    .query("trades >= 100")
    .sort_values(["total_pnl", "avg_pnl"], ascending=[False, False])
    .head(30)
)

print("\nTop 30 by avg P&L, requiring 50-250 trades:")
display(
    sweep_df
    .query("trades >= 50 and trades <= 250")
    .sort_values(["avg_pnl", "total_pnl"], ascending=[False, False])
    .head(30)
)

print("\nBest by rule family, requiring at least 50 trades:")
display(
    sweep_df
    .query("trades >= 50")
    .sort_values(["rule_name", "total_pnl"], ascending=[True, False])
    .groupby("rule_name", as_index=False)
    .head(5)
    .sort_values(["rule_name", "total_pnl"], ascending=[True, False])
)

Clean recalibration sample:
Rows: 12053
Dates: 2020-07-30 00:00:00 to 2026-06-11 00:00:00
Unique trade dates: 1471
Tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]


,count,mean,std,min,25%,50%,75%,max
test_pnl,12053.0,3487.680824,21498.866685,-129837.937694,192.161403,10225.785340,15026.561070,45095.398744
simple_carry_vrp_signal,12053.0,0.612708,0.485220,-1.131636,0.322324,0.601934,0.906852,2.581601
simple_carry_vrp_min_z,12053.0,-0.174461,1.125261,-4.277639,-0.954878,-0.190863,0.655277,3.475335
trailing_carry_vrp_signal,12053.0,0.597963,0.612509,-1.669171,0.194554,0.555591,0.983707,3.581716
trailing_carry_vrp_min_z,12053.0,-0.145413,1.070632,-4.767962,-0.868811,-0.172995,0.615527,4.208888
champion_forecast_vrp_signal,12053.0,0.490926,0.208507,-0.601171,0.378592,0.500130,0.622912,1.224966
champion_forecast_vrp_min_z,12053.0,-0.249698,1.020669,-6.587806,-0.915394,-0.222605,0.418753,3.049510
entry_rsi_14,12053.0,55.650813,11.334729,21.362549,47.430045,57.286866,64.384988,82.900307


Saved champion recalibration sweep:
C:\Users\patri\vrp_project\data\audit\put_champion_recalibration_sweep_v0_1.csv
C:\Users\patri\vrp_project\data\processed\put_champion_recalibration_sweep_selected_trades_v0_1.parquet

Sweep rows: 18360
Selected trade rows: 3388621

Top 30 by total P&L, requiring at least 100 trades:


,rule_name,rule_id,rsi_cap,simple_signal_threshold,simple_z_threshold,champion_signal_threshold,champion_z_threshold,old_signal_threshold,old_z_threshold,trades,...,max_drawdown,worst_10_trade_sum,worst_20_trade_sum,positive_year_rate,min_year_pnl,avg_rsi,avg_tenor,front_pct,middle_pct,back_pct
0,old_simple_confirm,2521,72,0.40,0.0,NaN,NaN,0.5,0.0,601,...,-9.432520e+05,-533576.106123,-9.412136e+05,0.857143,-31920.406471,53.938737,21.229617,0.391015,0.204659,0.404326
1,old_simple_confirm,3241,77,0.40,0.0,NaN,NaN,0.5,0.0,632,...,-9.432520e+05,-533576.106123,-9.412136e+05,0.857143,-10196.734323,54.944099,21.256329,0.392405,0.200949,0.406646
2,old_simple_confirm,2700,72,0.40,-0.5,NaN,NaN,0.7,0.0,660,...,-1.001149e+06,-533576.106123,-9.412136e+05,1.000000,34242.438039,54.575211,20.563636,0.419697,0.218182,0.362121
3,old_simple_confirm,2526,72,0.50,-0.5,NaN,NaN,0.5,0.0,694,...,-1.095646e+06,-604181.933966,-1.039157e+06,0.857143,-92804.983935,54.404644,20.178674,0.440922,0.208934,0.350144
4,old_simple_confirm,3420,77,0.40,-0.5,NaN,NaN,0.7,0.0,700,...,-1.001149e+06,-533576.106123,-9.412136e+05,1.000000,34242.438039,55.706739,20.567143,0.422857,0.212857,0.364286
5,old_simple_confirm,2527,72,0.50,0.0,NaN,NaN,0.5,0.0,584,...,-9.432520e+05,-533576.106123,-9.412136e+05,0.857143,-87452.412813,53.772924,21.061644,0.398973,0.207192,0.393836
6,old_simple_confirm,2520,72,0.40,-0.5,NaN,NaN,0.5,0.0,723,...,-1.164424e+06,-604181.933966,-1.049804e+06,0.714286,-133115.223752,54.544887,20.423237,0.427386,0.208852,0.363762
7,old_simple_confirm,3246,77,0.50,-0.5,NaN,NaN,0.5,0.0,733,...,-1.095646e+06,-604181.933966,-1.039157e+06,0.857143,-100644.296369,55.465030,20.160982,0.444748,0.204638,0.350614
8,simple_only,108,72,0.40,-0.5,NaN,NaN,NaN,NaN,948,...,-7.058454e+05,-435565.121706,-4.868790e+05,0.857143,-182972.730331,53.631524,15.322785,0.686709,0.159283,0.154008
9,old_simple_confirm,3240,77,0.40,-0.5,NaN,NaN,0.5,0.0,764,...,-1.164424e+06,-604181.933966,-1.049804e+06,0.714286,-133115.223752,55.606461,20.426702,0.430628,0.204188,0.365183



Top 30 by avg P&L, requiring 50-250 trades:


,rule_name,rule_id,rsi_cap,simple_signal_threshold,simple_z_threshold,champion_signal_threshold,champion_z_threshold,old_signal_threshold,old_z_threshold,trades,...,max_drawdown,worst_10_trade_sum,worst_20_trade_sum,positive_year_rate,min_year_pnl,avg_rsi,avg_tenor,front_pct,middle_pct,back_pct
7739,simple_champion_confirm,7957,58,0.5,1.0,0.00,-0.5,NaN,NaN,87,...,-127739.397244,-108785.704020,-24359.084848,0.833333,-96020.719965,47.310328,22.655172,0.264368,0.287356,0.448276
7653,simple_champion_confirm,7741,58,0.4,1.0,0.00,-0.5,NaN,NaN,88,...,-127739.397244,-108785.704020,-24359.084848,0.833333,-96020.719965,47.293853,22.670455,0.261364,0.295455,0.443182
12625,simple_champion_confirm,11199,68,0.9,1.0,0.00,0.3,NaN,NaN,50,...,-46038.108265,80897.248453,178039.362107,1.000000,22045.397176,55.270763,21.300000,0.360000,0.240000,0.400000
12626,simple_champion_confirm,11205,68,0.9,1.0,0.25,0.3,NaN,NaN,50,...,-46038.108265,80897.248453,178039.362107,1.000000,22045.397176,55.270763,21.300000,0.360000,0.240000,0.400000
5737,simple_champion_confirm,9253,62,0.5,1.0,0.00,-0.5,NaN,NaN,115,...,-127739.397244,-100042.443460,-39732.138399,0.833333,-84756.048640,50.353765,22.800000,0.260870,0.286957,0.452174
5653,simple_champion_confirm,9037,62,0.4,1.0,0.00,-0.5,NaN,NaN,116,...,-127739.397244,-100042.443460,-39732.138399,0.833333,-84756.048640,50.315030,22.810345,0.258621,0.293103,0.448276
9907,simple_champion_confirm,8605,58,0.9,1.0,0.00,-0.5,NaN,NaN,71,...,-127739.397244,-108785.704020,-29803.030147,0.833333,-96020.719965,47.391228,20.746479,0.380282,0.197183,0.422535
12467,simple_champion_confirm,7958,58,0.5,1.0,0.00,0.0,NaN,NaN,52,...,-66761.272072,-7597.020329,118251.773870,0.833333,-46886.223770,47.280943,23.134615,0.269231,0.250000,0.480769
12468,simple_champion_confirm,7964,58,0.5,1.0,0.25,0.0,NaN,NaN,52,...,-66761.272072,-7597.020329,118251.773870,0.833333,-46886.223770,47.280943,23.134615,0.269231,0.250000,0.480769
12348,simple_champion_confirm,7742,58,0.4,1.0,0.00,0.0,NaN,NaN,53,...,-66761.272072,-7597.020329,118251.773870,0.833333,-46886.223770,47.254143,23.150943,0.264151,0.264151,0.471698



Best by rule family, requiring at least 50 trades:


,rule_name,rule_id,rsi_cap,simple_signal_threshold,simple_z_threshold,champion_signal_threshold,champion_z_threshold,old_signal_threshold,old_z_threshold,trades,...,max_drawdown,worst_10_trade_sum,worst_20_trade_sum,positive_year_rate,min_year_pnl,avg_rsi,avg_tenor,front_pct,middle_pct,back_pct
760,champion_only,324,77,NaN,NaN,0.00,-1.0,NaN,NaN,1208,...,-2.884669e+06,-781899.001721,-1.332583e+06,0.714286,-1.866560e+06,56.336859,21.432119,0.327815,0.364238,0.307947
1007,champion_only,288,72,NaN,NaN,0.00,-1.0,NaN,NaN,1156,...,-2.810604e+06,-781899.001721,-1.332583e+06,0.714286,-1.792494e+06,55.540461,21.412630,0.328720,0.364187,0.307093
1634,champion_only,252,68,NaN,NaN,0.00,-1.0,NaN,NaN,1055,...,-2.629397e+06,-781899.001721,-1.332583e+06,0.714286,-1.611288e+06,54.185531,21.463507,0.325118,0.368720,0.306161
1783,champion_only,325,77,NaN,NaN,0.00,-0.5,NaN,NaN,987,...,-2.901101e+06,-715086.383684,-1.300969e+06,0.857143,-1.873431e+06,56.591336,21.562310,0.327254,0.350557,0.322188
2027,champion_only,289,72,NaN,NaN,0.00,-0.5,NaN,NaN,944,...,-2.827036e+06,-715086.383684,-1.306593e+06,0.857143,-1.799366e+06,55.798073,21.552966,0.328390,0.349576,0.322034
529,old_champion_confirm,6192,72,NaN,NaN,0.00,-1.0,0.5,0.5,486,...,-8.227042e+05,-604577.194349,-7.847507e+05,0.857143,-1.464835e+05,56.713271,20.246914,0.397119,0.312757,0.290123
606,old_champion_confirm,5472,68,NaN,NaN,0.00,-1.0,0.5,0.5,433,...,-7.920357e+05,-604577.194349,-7.771291e+05,0.857143,-2.173064e+05,55.124246,20.397229,0.392610,0.307159,0.300231
670,old_champion_confirm,7020,77,NaN,NaN,0.00,-1.0,0.7,0.0,624,...,-1.054022e+06,-713856.759068,-9.494692e+05,0.714286,-1.356552e+05,57.502099,20.091346,0.400641,0.312500,0.286859
673,old_champion_confirm,6912,77,NaN,NaN,0.00,-1.0,0.5,0.5,517,...,-8.227042e+05,-604577.194349,-7.847507e+05,0.857143,-6.327863e+04,57.770531,20.321083,0.392650,0.317215,0.290135
689,old_champion_confirm,6300,72,NaN,NaN,0.00,-1.0,0.7,0.0,584,...,-1.054022e+06,-713856.759068,-9.494692e+05,0.714286,-2.188601e+05,56.355277,20.023973,0.405822,0.308219,0.285959


In [25]:
# ============================================================
# Cell 23: By-tenor diagnostics for champion model and champion rules
# ============================================================

import numpy as np
import pandas as pd

SWEEP_CSV = AUDIT_DIR / "put_champion_recalibration_sweep_v0_1.csv"

SWEEP_TRADES_PARQUET = (
    PROCESSED_DATA_DIR
    / "put_champion_recalibration_sweep_selected_trades_v0_1.parquet"
)

MODEL_BY_TENOR_CSV = (
    AUDIT_DIR / "forecast_vol_model_comparison_by_tenor_v0_1.csv"
)

BY_TENOR_DIAGNOSTIC_CSV = (
    AUDIT_DIR / "put_champion_recalibration_by_tenor_diagnostic_v0_1.csv"
)

BY_TENOR_SHORTLIST_CSV = (
    AUDIT_DIR / "put_champion_recalibration_by_tenor_shortlist_v0_1.csv"
)

sweep_df = pd.read_csv(SWEEP_CSV)
sweep_trades_df = pd.read_parquet(SWEEP_TRADES_PARQUET).copy()
model_by_tenor_df = pd.read_csv(MODEL_BY_TENOR_CSV)

sweep_trades_df["trade_dt"] = pd.to_datetime(sweep_trades_df["trade_dt"])
sweep_trades_df["entry_tenor"] = sweep_trades_df["entry_tenor"].astype(int)

# ------------------------------------------------------------
# 1. Forecast-model by-tenor comparison
# ------------------------------------------------------------

print("Forecast model comparison by tenor:")

forecast_pivot = (
    model_by_tenor_df
    .pivot_table(
        index="tenor",
        columns="model",
        values=["log_rmse", "vol_rmse_pct", "log_bias", "vol_bias_pct"],
        aggfunc="first",
    )
)

display(forecast_pivot)

best_log_by_tenor = (
    model_by_tenor_df
    .sort_values(["tenor", "log_rmse"])
    .groupby("tenor", as_index=False)
    .head(3)
    [
        [
            "tenor",
            "model",
            "rows",
            "log_rmse",
            "log_bias",
            "vol_rmse_pct",
            "vol_bias_pct",
            "corr_log_prediction_target",
        ]
    ]
)

print("\nTop 3 forecast models by log RMSE for each tenor:")
display(best_log_by_tenor)

best_vol_by_tenor = (
    model_by_tenor_df
    .sort_values(["tenor", "vol_rmse_pct"])
    .groupby("tenor", as_index=False)
    .head(3)
    [
        [
            "tenor",
            "model",
            "rows",
            "log_rmse",
            "log_bias",
            "vol_rmse_pct",
            "vol_bias_pct",
            "corr_log_prediction_target",
        ]
    ]
)

print("\nTop 3 forecast models by vol RMSE for each tenor:")
display(best_vol_by_tenor)


# ------------------------------------------------------------
# 2. Pick representative/best rules from each family
# ------------------------------------------------------------

def pick_best_rules(sweep_df):
    rows = []

    selection_specs = [
        {
            "label": "best_total_old_simple_confirm",
            "rule_name": "old_simple_confirm",
            "query": "trades >= 100",
            "sort_cols": ["total_pnl", "avg_pnl"],
        },
        {
            "label": "best_total_simple_only",
            "rule_name": "simple_only",
            "query": "trades >= 100",
            "sort_cols": ["total_pnl", "avg_pnl"],
        },
        {
            "label": "best_total_champion_only",
            "rule_name": "champion_only",
            "query": "trades >= 100",
            "sort_cols": ["total_pnl", "avg_pnl"],
        },
        {
            "label": "best_total_old_champion_confirm",
            "rule_name": "old_champion_confirm",
            "query": "trades >= 100",
            "sort_cols": ["total_pnl", "avg_pnl"],
        },
        {
            "label": "best_total_simple_champion_confirm",
            "rule_name": "simple_champion_confirm",
            "query": "trades >= 100",
            "sort_cols": ["total_pnl", "avg_pnl"],
        },
        {
            "label": "best_avg_simple_champion_confirm_50_250",
            "rule_name": "simple_champion_confirm",
            "query": "trades >= 50 and trades <= 250",
            "sort_cols": ["avg_pnl", "total_pnl"],
        },
        {
            "label": "best_total_simple_rich_champion_not_hostile",
            "rule_name": "simple_rich_champion_not_hostile",
            "query": "trades >= 100",
            "sort_cols": ["total_pnl", "avg_pnl"],
        },
        {
            "label": "best_total_three_signal_confirm",
            "rule_name": "three_signal_confirm",
            "query": "trades >= 100",
            "sort_cols": ["total_pnl", "avg_pnl"],
        },
    ]

    for spec in selection_specs:
        temp = sweep_df[sweep_df["rule_name"] == spec["rule_name"]].copy()
        temp = temp.query(spec["query"]).copy()

        if temp.empty:
            continue

        best = (
            temp
            .sort_values(spec["sort_cols"], ascending=[False] * len(spec["sort_cols"]))
            .head(1)
            .copy()
        )

        best["selection_label"] = spec["label"]
        rows.append(best)

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)

shortlist_rules_df = pick_best_rules(sweep_df)

shortlist_rules_df.to_csv(BY_TENOR_SHORTLIST_CSV, index=False)

print("\nShortlisted representative rules:")
display(
    shortlist_rules_df[
        [
            "selection_label",
            "rule_name",
            "rule_id",
            "rsi_cap",
            "simple_signal_threshold",
            "simple_z_threshold",
            "champion_signal_threshold",
            "champion_z_threshold",
            "old_signal_threshold",
            "old_z_threshold",
            "trades",
            "total_pnl",
            "avg_pnl",
            "win_rate",
            "q05_pnl",
            "worst_pnl",
            "max_drawdown",
            "positive_year_rate",
            "min_year_pnl",
            "avg_tenor",
            "front_pct",
            "middle_pct",
            "back_pct",
        ]
    ]
)


# ------------------------------------------------------------
# 3. By-tenor and by-tenor-group performance for shortlisted rules
# ------------------------------------------------------------

shortlist_rule_ids = shortlist_rules_df["rule_id"].astype(int).tolist()

shortlist_trades_df = sweep_trades_df[
    sweep_trades_df["rule_id"].astype(int).isin(shortlist_rule_ids)
].copy()

shortlist_trades_df = shortlist_trades_df.merge(
    shortlist_rules_df[["rule_id", "selection_label"]],
    on="rule_id",
    how="left",
)

def summarize_group(df, group_cols):
    out = (
        df
        .groupby(group_cols, dropna=False, observed=False)
        .agg(
            trades=("trade_dt", "count"),
            total_pnl=("test_pnl", "sum"),
            avg_pnl=("test_pnl", "mean"),
            median_pnl=("test_pnl", "median"),
            win_rate=("test_pnl", lambda x: (x > 0).mean()),
            q05_pnl=("test_pnl", lambda x: x.quantile(0.05)),
            worst_pnl=("test_pnl", "min"),
            avg_rsi=("entry_rsi_14", "mean"),
            avg_simple_signal=("simple_carry_vrp_signal", "mean"),
            avg_simple_z=("simple_carry_vrp_min_z", "mean"),
            avg_champion_signal=("champion_forecast_vrp_signal", "mean"),
            avg_champion_z=("champion_forecast_vrp_min_z", "mean"),
            avg_trailing_signal=("trailing_carry_vrp_signal", "mean"),
            avg_trailing_z=("trailing_carry_vrp_min_z", "mean"),
        )
        .reset_index()
    )

    return out

by_tenor_df = summarize_group(
    shortlist_trades_df,
    ["selection_label", "rule_name", "rule_id", "entry_tenor"],
)

by_tenor_group_df = summarize_group(
    shortlist_trades_df,
    ["selection_label", "rule_name", "rule_id", "tenor_group"],
)

by_tenor_df["diagnostic_level"] = "tenor"
by_tenor_group_df["diagnostic_level"] = "tenor_group"

combined_by_tenor_df = pd.concat(
    [
        by_tenor_df.rename(columns={"entry_tenor": "bucket"}),
        by_tenor_group_df.rename(columns={"tenor_group": "bucket"}),
    ],
    ignore_index=True,
)

combined_by_tenor_df.to_csv(BY_TENOR_DIAGNOSTIC_CSV, index=False)

print("\nSaved by-tenor diagnostic:")
print(BY_TENOR_DIAGNOSTIC_CSV)
print(BY_TENOR_SHORTLIST_CSV)

print("\nBy tenor group:")
display(
    by_tenor_group_df
    .sort_values(["selection_label", "total_pnl"], ascending=[True, False])
)

print("\nBy exact tenor:")
display(
    by_tenor_df
    .sort_values(["selection_label", "entry_tenor"])
)

Forecast model comparison by tenor:


log_bias                                              \
model existing_simple    har_rv har_rv_iv har_rv_iv_vov   implied   
tenor                                                               
9           -0.086434  0.133567  0.100642     -0.011225  0.615075   
12          -0.071105  0.148816  0.146297      0.051137  0.658037   
15          -0.063482  0.063678  0.083796     -0.004424  0.589788   
18          -0.065283  0.061709  0.113373      0.036728  0.614927   
21           0.108923  0.021100  0.095995      0.041336  0.594493   
24           0.060974 -0.041869  0.101431      0.040525  0.581098   
27           0.070253  0.004291  0.125566      0.073098  0.613693   
30           0.053923 -0.041260  0.119072      0.069238  0.595311   
33           0.037600 -0.023531  0.143603      0.094894  0.622373   

                                 log_rmse                                    \
model trailing_same_tenor existing_simple    har_rv har_rv_iv har_rv_iv_vov   
tenor                                                                         
9                0.048436        0.880198  0.902649  0.790661      0.823239   
12               0.036889        0.825467  0.834850  0.753927      0.774044   
15               0.015602        0.793367  0.784293  0.726438      0.745652   
18               0.016476        0.779976  0.768570  0.713678      0.737418   
21               0.003052        0.774015  0.744090  0.703065      0.742501   
24              -0.046066        0.748327  0.711565  0.682306      0.732992   
27               0.009957        0.750378  0.708776  0.686765      0.727526   
30              -0.000378        0.729665  0.684345  0.673388      0.718195   
33              -0.005116        0.726560  0.684819  0.679616      0.726969   

       ... vol_bias_pct                                              \
model  ...    har_rv_iv har_rv_iv_vov   implied trailing_same_tenor   
tenor  ...                                                            
9      ...    -0.129048     -0.869594  3.888830            0.277187   
12     ...     0.341254     -0.258199  4.392836            0.250177   
15     ...    -0.020710     -0.566266  4.032645            0.105007   
18     ...     0.224710     -0.194997  4.291576            0.101231   
21     ...     0.120339     -0.094315  4.224752           -0.002078   
24     ...     0.209568     -0.019959  4.259106           -0.350969   
27     ...     0.433960      0.224867  4.541587            0.057681   
30     ...     0.406594      0.200355  4.463610           -0.004577   
33     ...     0.648744      0.465466  4.750902           -0.047609   

         vol_rmse_pct                                              \
model existing_simple    har_rv har_rv_iv har_rv_iv_vov   implied   
tenor                                                               
9            7.437750  7.358861  6.652603      8.163074  7.471859   
12           6.861241  6.732400  6.289042      7.771790  7.440335   
15           6.873259  6.684562  6.403123      7.857842  7.315108   
18           6.702340  6.493944  6.277297      7.864712  7.353653   
21           6.870649  6.471830  6.362241      8.239565  7.358640   
24           6.756395  6.337235  6.329689      8.533201  7.370225   
27           6.754171  6.242948  6.325095      8.182784  7.481132   
30           6.697490  6.165220  6.312193      8.208432  7.397021   
33           6.674262  6.133696  6.353896      8.241525  7.587339   

                           
model trailing_same_tenor  
tenor                      
9                8.335793  
12               7.712557  
15               7.643558  
18               7.383855  
21               7.295608  
24               7.070650  
27               7.099417  
30               7.054307  
33               7.041816  

[9 rows x 24 columns]


Top 3 forecast models by log RMSE for each tenor:


,tenor,model,rows,log_rmse,log_bias,vol_rmse_pct,vol_bias_pct,corr_log_prediction_target
36,9,har_rv_iv,1440,0.790661,0.100642,6.652603,-0.129048,0.625950
45,9,har_rv_iv_vov,1440,0.823239,-0.011225,8.163074,-0.869594,0.581202
18,9,existing_simple,1440,0.880198,-0.086434,7.437750,-1.445511,0.520101
37,12,har_rv_iv,1481,0.753927,0.146297,6.289042,0.341254,0.617239
46,12,har_rv_iv_vov,1481,0.774044,0.051137,7.771790,-0.258199,0.580066
19,12,existing_simple,1481,0.825467,-0.071105,6.861241,-1.125253,0.525006
38,15,har_rv_iv,1488,0.726438,0.083796,6.403123,-0.020710,0.593728
47,15,har_rv_iv_vov,1488,0.745652,-0.004424,7.857842,-0.566266,0.567454
29,15,har_rv,1488,0.784293,0.063678,6.684562,-0.460435,0.482011
39,18,har_rv_iv,1455,0.713678,0.113373,6.277297,0.224710,0.590210



Top 3 forecast models by vol RMSE for each tenor:


...

,selection_label,rule_name,rule_id,rsi_cap,simple_signal_threshold,simple_z_threshold,champion_signal_threshold,champion_z_threshold,old_signal_threshold,old_z_threshold,...,win_rate,q05_pnl,worst_pnl,max_drawdown,positive_year_rate,min_year_pnl,avg_tenor,front_pct,middle_pct,back_pct
0,best_total_old_simple_confirm,old_simple_confirm,2521,72,0.40,0.0,NaN,NaN,0.5,0.0,...,0.785358,-28107.751777,-111511.425505,-9.432520e+05,0.857143,-3.192041e+04,21.229617,0.391015,0.204659,0.404326
1,best_total_simple_only,simple_only,108,72,0.40,-0.5,NaN,NaN,NaN,NaN,...,0.746835,-26998.641143,-97591.734861,-7.058454e+05,0.857143,-1.829727e+05,15.322785,0.686709,0.159283,0.154008
2,best_total_champion_only,champion_only,324,77,NaN,NaN,0.0,-1.0,NaN,NaN,...,0.733444,-48806.910772,-129837.937694,-2.884669e+06,0.714286,-1.866560e+06,21.432119,0.327815,0.364238,0.307947
3,best_total_old_champion_confirm,old_champion_confirm,6192,72,NaN,NaN,0.0,-1.0,0.5,0.5,...,0.790123,-29496.628660,-112316.486943,-8.227042e+05,0.857143,-1.464835e+05,20.246914,0.397119,0.312757,0.290123
4,best_total_simple_champion_confirm,simple_champion_confirm,10800,68,0.75,-0.5,0.0,-1.0,NaN,NaN,...,0.776842,-30063.278161,-100779.661310,-4.847647e+05,0.857143,-3.053541e+04,18.688421,0.494737,0.221053,0.284211
5,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,58,0.50,1.0,0.0,-0.5,NaN,NaN,...,0.839080,-18835.933522,-47172.123854,-1.277394e+05,0.833333,-9.602072e+04,22.655172,0.264368,0.287356,0.448276
6,best_total_simple_rich_champion_not_hostile,simple_rich_champion_not_hostile,17050,72,0.50,-0.5,0.0,-2.0,NaN,NaN,...,0.744755,-26894.493048,-97591.734861,-5.697968e+05,0.857143,-2.632517e+05,14.178322,0.744755,0.158508,0.096737
7,best_total_three_signal_confirm,three_signal_confirm,14520,72,0.40,-0.5,0.0,-0.5,0.7,-0.5,...,0.783333,-30069.479568,-129837.937694,-6.623453e+05,0.714286,-2.033579e+04,21.757143,0.330952,0.285714,0.383333



Saved by-tenor diagnostic:
C:\Users\patri\vrp_project\data\audit\put_champion_recalibration_by_tenor_diagnostic_v0_1.csv
C:\Users\patri\vrp_project\data\audit\put_champion_recalibration_by_tenor_shortlist_v0_1.csv

By tenor group:


,selection_label,rule_name,rule_id,tenor_group,trades,total_pnl,avg_pnl,median_pnl,win_rate,q05_pnl,worst_pnl,avg_rsi,avg_simple_signal,avg_simple_z,avg_champion_signal,avg_champion_z,avg_trailing_signal,avg_trailing_z,diagnostic_level
0,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,back_27_33d,39,5.494417e+05,14088.248338,16633.561318,0.871795,-30346.403984,-47172.123854,45.803797,1.100716,1.421314,0.527164,0.295050,1.210269,1.153850,tenor_group
2,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,middle_18_24d,25,4.062587e+05,16250.348060,14743.626094,1.000000,11646.782286,8337.898844,46.459678,0.942909,1.529301,0.555808,0.101962,1.024750,1.134804,tenor_group
1,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,front_9_15d,23,4.662192e+04,2027.040148,7664.545391,0.608696,-19848.596338,-46038.108265,50.789501,1.249331,1.388311,0.592499,0.317697,0.923534,0.497963,tenor_group
3,best_total_champion_only,champion_only,324,back_27_33d,372,1.600536e+06,4302.517188,15260.427681,0.776882,-59280.814242,-129837.937694,54.680564,0.523424,-0.412496,0.504829,0.217156,0.594756,-0.306007,tenor_group
5,best_total_champion_only,champion_only,324,middle_18_24d,440,7.377438e+05,1676.690364,10811.467730,0.709091,-55756.946698,-98732.347272,54.499001,0.417867,-0.263901,0.565036,0.180088,0.556771,-0.089116,tenor_group
4,best_total_champion_only,champion_only,324,front_9_15d,396,2.003617e+05,505.963918,5675.678551,0.719697,-30354.196887,-99982.493610,59.934836,0.644838,-0.352706,0.593428,0.370002,0.598531,-0.086327,tenor_group
6,best_total_old_champion_confirm,old_champion_confirm,6192,back_27_33d,141,1.290679e+06,9153.753124,14267.804213,0.843972,-33137.147895,-112316.486943,54.144046,0.975256,0.839002,0.497020,0.217376,1.230059,1.028287,tenor_group
8,best_total_old_champion_confirm,old_champion_confirm,6192,middle_18_24d,152,6.983212e+05,4594.218711,11241.662271,0.750000,-35016.516200,-71883.010669,57.334666,0.769232,0.733416,0.552284,0.166695,1.162927,1.192553,tenor_group
7,best_total_old_champion_confirm,old_champion_confirm,6192,front_9_15d,193,6.977625e+05,3615.349673,7149.215447,0.782383,-22218.688754,-46038.108265,58.100880,0.701015,0.045016,0.599865,0.278966,1.303633,1.144130,tenor_group
9,best_total_old_simple_confirm,old_simple_confirm,2521,back_27_33d,243,2.318711e+06,9542.020704,14178.080866,0.835391,-26152.709907,-111511.425505,53.784244,0.975724,0.927245,0.475841,-0.253719,1.169380,0.981395,tenor_group



By exact tenor:


,selection_label,rule_name,rule_id,entry_tenor,trades,total_pnl,avg_pnl,median_pnl,win_rate,q05_pnl,worst_pnl,avg_rsi,avg_simple_signal,avg_simple_z,avg_champion_signal,avg_champion_z,avg_trailing_signal,avg_trailing_z,diagnostic_level
0,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,9,16,-18071.430092,-1129.464381,-73.269512,0.500000,-26775.722875,-46038.108265,51.433464,1.243022,1.497899,0.568491,0.217380,0.902699,0.581936,tenor
1,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,12,5,41447.278455,8289.455691,8390.477824,0.800000,-10700.381855,-15291.613667,49.271876,1.228351,1.083435,0.681038,0.595293,1.011867,0.437828,tenor
2,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,15,2,23246.075040,11623.037520,11623.037520,1.000000,10708.981018,10607.419185,49.431857,1.352249,1.273796,0.563215,0.426249,0.869387,-0.023483,tenor
3,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,18,5,71942.835891,14388.567178,12307.011661,1.000000,11716.976301,11634.965601,52.023645,1.329023,1.372240,0.597830,0.606548,1.296344,0.887191,tenor
4,best_avg_simple_champion_confirm_50_250,simple_champion_confirm,7957,21,2,35086.893941,17543.446971,17543.446971,1.000000,9258.453656,8337.898844,51.702266,1.012216,1.044130,0.375002,-0.386199,0.930043,0.168503,tenor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,best_total_three_signal_confirm,three_signal_confirm,14520,21,25,281299.202353,11251.968094,12351.383332,0.960000,2411.274085,-16391.340739,57.056524,0.764909,0.403558,0.561548,0.382621,1.091708,0.538928,tenor
68,best_total_three_signal_confirm,three_signal_confirm,14520,24,56,438881.206028,7837.164393,11842.858598,0.803571,-24816.242825,-68207.378153,54.441989,0.890572,0.991564,0.595023,0.286215,1.131868,0.926788,tenor
69,best_total_three_signal_confirm,three_signal_confirm,14520,27,29,94846.651271,3270.574182,12770.137525,0.758621,-40942.492678,-47172.123854,57.271197,0.852095,0.508323,0.553305,0.496680,1.118743,0.696414,tenor
70,best_total_three_signal_confirm,three_signal_confirm,14520,30,55,703871.705244,12797.667368,14685.666387,0.890909,-14957.240879,-27822.697682,55.693255,1.002027,0.928814,0.498578,0.243798,1.193926,0.949014,tenor


In [26]:
# ============================================================
# Cell 24: Save by-tenor champion diagnostic conclusion
# ============================================================

import json
import pandas as pd

CHAMPION_BY_TENOR_CONFIG_JSON = (
    AUDIT_DIR / "put_champion_by_tenor_diagnostic_config_v0_1.json"
)

CHAMPION_BY_TENOR_SUMMARY_CSV = (
    AUDIT_DIR / "put_champion_by_tenor_diagnostic_summary_v0_1.csv"
)

champion_by_tenor_config = {
    "strategy_version": "put_champion_by_tenor_diagnostic_v0_1",

    "core_conclusions": {
        "primary_trade_signal": "old_simple_confirm",
        "champion_model": "har_rv_iv",
        "champion_model_role": "model_expected_vrp_denominator_and_quality_context",
        "champion_as_standalone_signal": False,
        "champion_as_broad_hard_filter": False,
        "champion_as_middle_back_quality_tag": True,
        "front_tenor_champion_use": "avoid as hard filter; noisy/weak in 9-15d",
        "middle_back_tenor_champion_use": "candidate quality tag or size-up flag after portfolio testing",
    },

    "forecast_model_conclusions": {
        "har_rv_iv": "best log-variance RMSE across tenors",
        "har_rv": "competitive/better vol-point RMSE in 27-33d",
        "har_rv_iv_vov": "not promoted; underperformed har_rv_iv",
        "next_denominator_tests": [
            "per_tenor_har_rv_iv",
            "tenor_group_har_rv_iv",
            "back_tenor_har_rv_or_blend",
            "bias_adjusted_har_rv_iv",
        ],
    },

    "trading_conclusions": {
        "old_simple_confirm": "main production candidate",
        "simple_only": "useful carry signal but front-tenor heavy",
        "champion_only": "not a replacement signal",
        "simple_champion_confirm": "useful selective quality setup, especially middle/back",
        "three_signal_confirm": "not clearly superior yet",
        "vol_of_vol": "defer; test later as risk/regime overlay after denominator variants",
    },

    "next_stage": {
        "stage": "improved_denominator_tests",
        "then": "portfolio engine comparison",
        "then_after": "vol_of_vol regime overlay / sizing overlay",
    },
}

with open(CHAMPION_BY_TENOR_CONFIG_JSON, "w") as f:
    json.dump(champion_by_tenor_config, f, indent=4)

summary_rows = [
    {
        "topic": "production_signal",
        "conclusion": "Keep old_simple_confirm as the primary trade-selection family.",
        "action": "Do not replace with champion-only.",
    },
    {
        "topic": "champion_forecast",
        "conclusion": "har_rv_iv remains the clean model-expected VRP denominator.",
        "action": "Use as diagnostic / denominator candidate, not standalone trade engine.",
    },
    {
        "topic": "front_tenors",
        "conclusion": "Champion confirmation is weak/noisy in 9-15d.",
        "action": "Keep front tenors mostly driven by simple/EWMA carry VRP.",
    },
    {
        "topic": "middle_back_tenors",
        "conclusion": "Champion confirmation is more promising in 18-33d.",
        "action": "Test as quality tag / size-up flag in portfolio engine.",
    },
    {
        "topic": "back_tenor_denominator",
        "conclusion": "har_rv may be better than har_rv_iv on vol-point RMSE for 27-33d.",
        "action": "Test back-tenor HAR-only or HAR/HAR-IV blend.",
    },
    {
        "topic": "vol_of_vol",
        "conclusion": "Not yet a core signal input.",
        "action": "Bring in later as sizing/hedge/regime overlay.",
    },
]

champion_by_tenor_summary_df = pd.DataFrame(summary_rows)
champion_by_tenor_summary_df.to_csv(CHAMPION_BY_TENOR_SUMMARY_CSV, index=False)

print("Saved by-tenor champion diagnostic config:")
print(CHAMPION_BY_TENOR_CONFIG_JSON)

print("\nSaved by-tenor champion diagnostic summary:")
print(CHAMPION_BY_TENOR_SUMMARY_CSV)

display(champion_by_tenor_summary_df)

Saved by-tenor champion diagnostic config:
C:\Users\patri\vrp_project\data\audit\put_champion_by_tenor_diagnostic_config_v0_1.json

Saved by-tenor champion diagnostic summary:
C:\Users\patri\vrp_project\data\audit\put_champion_by_tenor_diagnostic_summary_v0_1.csv


,topic,conclusion,action
0,production_signal,Keep old_simple_confirm as the primary trade-s...,Do not replace with champion-only.
1,champion_forecast,har_rv_iv remains the clean model-expected VRP...,"Use as diagnostic / denominator candidate, not..."
2,front_tenors,Champion confirmation is weak/noisy in 9-15d.,Keep front tenors mostly driven by simple/EWMA...
3,middle_back_tenors,Champion confirmation is more promising in 18-...,Test as quality tag / size-up flag in portfoli...
4,back_tenor_denominator,har_rv may be better than har_rv_iv on vol-poi...,Test back-tenor HAR-only or HAR/HAR-IV blend.
5,vol_of_vol,Not yet a core signal input.,Bring in later as sizing/hedge/regime overlay.


In [27]:
# ============================================================
# Cell 25: Test per-tenor and tenor-group denominator models
# Pure numpy ridge, no sklearn dependency
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

COMPLETE_PANEL_PATH = (
    PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_complete_targets_v0_1.parquet"
)

if not COMPLETE_PANEL_PATH.exists():
    COMPLETE_PANEL_PATH = (
        PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_v0_1.parquet"
    )

BASE_PRED_PATH = (
    PROCESSED_DATA_DIR / "forecast_vol_model_predictions_v0_1.parquet"
)

STRUCTURE_PRED_PATH = (
    PROCESSED_DATA_DIR / "forecast_vol_model_structure_predictions_v0_1.parquet"
)

STRUCTURE_COMPARISON_CSV = (
    AUDIT_DIR / "forecast_vol_model_structure_comparison_v0_1.csv"
)

STRUCTURE_COMPARISON_BY_TENOR_CSV = (
    AUDIT_DIR / "forecast_vol_model_structure_comparison_by_tenor_v0_1.csv"
)

STRUCTURE_CONFIG_JSON = (
    AUDIT_DIR / "forecast_vol_model_structure_test_config_v0_1.json"
)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def safe_corr(a, b):
    temp = pd.DataFrame({"a": a, "b": b}).dropna()
    if len(temp) < 5:
        return np.nan
    if temp["a"].std() == 0 or temp["b"].std() == 0:
        return np.nan
    return temp["a"].corr(temp["b"])


def fit_predict_standardized_ridge(X_train, y_train, X_test, alpha=5.0):
    X_train = np.asarray(X_train, dtype=float)
    y_train = np.asarray(y_train, dtype=float)
    X_test = np.asarray(X_test, dtype=float)

    x_mean = np.nanmean(X_train, axis=0)
    x_std = np.nanstd(X_train, axis=0)

    x_std = np.where((x_std == 0) | np.isnan(x_std), 1.0, x_std)

    X_train_z = (X_train - x_mean) / x_std
    X_test_z = (X_test - x_mean) / x_std

    X_train_z = np.where(np.isfinite(X_train_z), X_train_z, 0.0)
    X_test_z = np.where(np.isfinite(X_test_z), X_test_z, 0.0)

    # Add intercept. Do not penalize intercept.
    X_design = np.column_stack([np.ones(len(X_train_z)), X_train_z])
    X_test_design = np.column_stack([np.ones(len(X_test_z)), X_test_z])

    penalty = np.eye(X_design.shape[1]) * alpha
    penalty[0, 0] = 0.0

    beta = np.linalg.solve(
        X_design.T @ X_design + penalty,
        X_design.T @ y_train,
    )

    return X_test_design @ beta


def tenor_group_from_tenor(t):
    t = int(t)
    if t in [9, 12, 15]:
        return "front_9_15d"
    if t in [18, 21, 24]:
        return "middle_18_24d"
    if t in [27, 30, 33]:
        return "back_27_33d"
    return "other"


def detect_feature_sets(df):
    cols = list(df.columns)

    # Prefer prior notebook definitions if available.
    try:
        feature_sets = {
            "har_rv": [c for c in MODEL_FEATURE_SETS["har_rv"] if c in cols],
            "har_rv_iv": [c for c in MODEL_FEATURE_SETS["har_rv_iv"] if c in cols],
        }

        if len(feature_sets["har_rv"]) >= 3 and len(feature_sets["har_rv_iv"]) >= 5:
            return feature_sets

    except Exception:
        pass

    bad_tokens = [
        "forward",
        "target",
        "pred_",
        "actual",
        "pnl",
    ]

    rv_cols = []
    for c in cols:
        cl = c.lower()

        if any(tok in cl for tok in bad_tokens):
            continue

        if (
            ("log_trailing" in cl and "variance" in cl)
            or ("trailing_realized" in cl and "log" in cl and "variance" in cl)
        ):
            rv_cols.append(c)

    # Include main trailing log variance if named differently.
    for c in ["log_trailing_realized_variance"]:
        if c in cols and c not in rv_cols:
            rv_cols.append(c)

    iv_cols = []
    exact_iv_candidates = [
        "log_implied_variance",
        "implied_variance",
        "vix_style_vol",
        "spx_rsi_14",
    ]

    for c in exact_iv_candidates:
        if c in cols:
            iv_cols.append(c)

    for c in cols:
        cl = c.lower()

        if any(tok in cl for tok in bad_tokens):
            continue

        if any(tok in cl for tok in ["slope", "curve", "term", "zscore", "z_score"]):
            if c not in rv_cols and c not in iv_cols:
                iv_cols.append(c)

    feature_sets = {
        "har_rv": sorted(rv_cols),
        "har_rv_iv": sorted(list(dict.fromkeys(rv_cols + iv_cols))),
    }

    return feature_sets


def evaluate_predictions(df, pred_cols, model_label_map=None):
    rows = []

    for pred_col in pred_cols:
        temp = df.dropna(subset=[pred_col, "target_log_variance"]).copy()

        if temp.empty:
            continue

        model_name = model_label_map.get(pred_col, pred_col) if model_label_map else pred_col

        temp["pred_var"] = np.exp(temp[pred_col])
        temp["target_var"] = np.exp(temp["target_log_variance"])

        temp["pred_vol_pct"] = np.sqrt(temp["pred_var"].clip(lower=1e-12)) * 100.0
        temp["target_vol_pct_calc"] = np.sqrt(temp["target_var"].clip(lower=1e-12)) * 100.0

        rows.append({
            "model": model_name,
            "rows": len(temp),
            "start_date": temp["trade_dt"].min(),
            "end_date": temp["trade_dt"].max(),

            "log_rmse": np.sqrt(np.mean((temp[pred_col] - temp["target_log_variance"]) ** 2)),
            "log_mae": np.mean(np.abs(temp[pred_col] - temp["target_log_variance"])),
            "log_bias": np.mean(temp[pred_col] - temp["target_log_variance"]),

            "vol_rmse_pct": np.sqrt(np.mean((temp["pred_vol_pct"] - temp["target_vol_pct_calc"]) ** 2)),
            "vol_mae_pct": np.mean(np.abs(temp["pred_vol_pct"] - temp["target_vol_pct_calc"])),
            "vol_bias_pct": np.mean(temp["pred_vol_pct"] - temp["target_vol_pct_calc"]),

            "corr_log_prediction_target": safe_corr(temp[pred_col], temp["target_log_variance"]),
            "corr_vol_prediction_target": safe_corr(temp["pred_vol_pct"], temp["target_vol_pct_calc"]),
        })

    return pd.DataFrame(rows)


def evaluate_predictions_by_tenor(df, pred_cols, model_label_map=None):
    rows = []

    for tenor, tenor_df in df.groupby("tenor"):
        eval_df = evaluate_predictions(
            tenor_df,
            pred_cols,
            model_label_map=model_label_map,
        )

        if not eval_df.empty:
            eval_df.insert(1, "tenor", tenor)
            rows.append(eval_df)

    if not rows:
        return pd.DataFrame()

    return pd.concat(rows, ignore_index=True)


# ------------------------------------------------------------
# Load modeling panel
# ------------------------------------------------------------

model_df = pd.read_parquet(COMPLETE_PANEL_PATH).copy()

if "trade_dt" not in model_df.columns:
    if "trade_date" in model_df.columns:
        model_df["trade_dt"] = pd.to_datetime(
            model_df["trade_date"].astype(str),
            format="%Y%m%d",
            errors="coerce",
        )
    else:
        raise ValueError("No trade_dt or trade_date column found.")

model_df["trade_dt"] = pd.to_datetime(model_df["trade_dt"])
model_df["tenor"] = model_df["tenor"].astype(int)
model_df["tenor_group"] = model_df["tenor"].map(tenor_group_from_tenor)

if "target_log_variance" not in model_df.columns:
    if "log_forward_realized_variance" in model_df.columns:
        model_df["target_log_variance"] = model_df["log_forward_realized_variance"]
    else:
        raise ValueError("Could not find target log variance column.")

feature_sets = detect_feature_sets(model_df)

print("Using modeling panel:")
print(COMPLETE_PANEL_PATH)
print("Rows:", len(model_df))
print("Dates:", model_df["trade_dt"].min(), "to", model_df["trade_dt"].max())

print("\nFeature set sizes:")
for name, cols in feature_sets.items():
    print(name, len(cols))
    print(cols[:20], "..." if len(cols) > 20 else "")

# ------------------------------------------------------------
# Merge pooled/base predictions if available
# ------------------------------------------------------------

pred_df = model_df.copy()

if BASE_PRED_PATH.exists():
    base_pred_df = pd.read_parquet(BASE_PRED_PATH).copy()
    base_pred_df["trade_dt"] = pd.to_datetime(base_pred_df["trade_dt"])
    base_pred_df["tenor"] = base_pred_df["tenor"].astype(int)

    base_cols = [
        "trade_dt",
        "tenor",
        "pred_logvar_har_rv",
        "pred_logvar_har_rv_iv",
    ]

    base_cols = [c for c in base_cols if c in base_pred_df.columns]

    pred_df = pred_df.merge(
        base_pred_df[base_cols],
        on=["trade_dt", "tenor"],
        how="left",
        suffixes=("", "_base"),
    )

# ------------------------------------------------------------
# Walk-forward model variants
# ------------------------------------------------------------

RIDGE_ALPHA = 5.0
MIN_TRAIN_OBS = 504
REFIT_FREQ = "MS"

model_variants = [
    {
        "model_name": "har_rv_per_tenor",
        "feature_set": "har_rv",
        "group_col": "tenor",
    },
    {
        "model_name": "har_rv_iv_per_tenor",
        "feature_set": "har_rv_iv",
        "group_col": "tenor",
    },
    {
        "model_name": "har_rv_by_tenor_group",
        "feature_set": "har_rv",
        "group_col": "tenor_group",
    },
    {
        "model_name": "har_rv_iv_by_tenor_group",
        "feature_set": "har_rv_iv",
        "group_col": "tenor_group",
    },
]

all_prediction_blocks = []

for variant in model_variants:
    model_name = variant["model_name"]
    feature_cols = feature_sets[variant["feature_set"]]
    group_col = variant["group_col"]

    print("\nRunning:", model_name, "| group_col:", group_col, "| features:", len(feature_cols))

    if len(feature_cols) == 0:
        print("Skipping; no features.")
        continue

    work_df = pred_df[
        ["trade_dt", "trade_date", "tenor", "tenor_group", "target_log_variance"] + feature_cols
    ].copy()

    work_df = work_df.dropna(
        subset=["trade_dt", "tenor", group_col, "target_log_variance"] + feature_cols
    ).copy()

    pred_blocks = []

    for group_value, group_df in work_df.groupby(group_col):
        group_df = group_df.sort_values("trade_dt").copy()

        unique_dates = pd.Series(group_df["trade_dt"].unique()).sort_values()

        if len(unique_dates) <= MIN_TRAIN_OBS:
            continue

        first_test_date = unique_dates.iloc[MIN_TRAIN_OBS]

        refit_dates = pd.date_range(
            start=first_test_date,
            end=unique_dates.iloc[-1],
            freq=REFIT_FREQ,
        )

        # Keep only refit dates that have data after them.
        refit_dates = [d for d in refit_dates if d <= unique_dates.iloc[-1]]

        for i, refit_date in enumerate(refit_dates):
            next_refit_date = (
                refit_dates[i + 1]
                if i + 1 < len(refit_dates)
                else unique_dates.iloc[-1] + pd.Timedelta(days=1)
            )

            train_mask = group_df["trade_dt"] < refit_date
            test_mask = (
                (group_df["trade_dt"] >= refit_date)
                & (group_df["trade_dt"] < next_refit_date)
            )

            train_df = group_df[train_mask].copy()
            test_df = group_df[test_mask].copy()

            if len(train_df) < MIN_TRAIN_OBS or test_df.empty:
                continue

            y_pred = fit_predict_standardized_ridge(
                train_df[feature_cols].values,
                train_df["target_log_variance"].values,
                test_df[feature_cols].values,
                alpha=RIDGE_ALPHA,
            )

            out = test_df[
                ["trade_dt", "trade_date", "tenor", "tenor_group", "target_log_variance"]
            ].copy()

            out[f"pred_logvar_{model_name}"] = y_pred
            out["refit_date"] = refit_date
            out["model_group_value"] = group_value

            pred_blocks.append(out)

    if pred_blocks:
        variant_pred_df = pd.concat(pred_blocks, ignore_index=True)
        all_prediction_blocks.append(variant_pred_df)

        print("Pred rows:", len(variant_pred_df))
        print("Dates:", variant_pred_df["trade_dt"].min(), "to", variant_pred_df["trade_dt"].max())
    else:
        print("No predictions produced.")

# ------------------------------------------------------------
# Combine predictions
# ------------------------------------------------------------

structure_pred_df = pred_df[
    [
        "trade_date",
        "trade_dt",
        "tenor",
        "tenor_group",
        "target_log_variance",
    ]
].copy()

# Carry pooled/base columns through if available.
for col in ["pred_logvar_har_rv", "pred_logvar_har_rv_iv"]:
    if col in pred_df.columns:
        structure_pred_df[col] = pred_df[col]

for block in all_prediction_blocks:
    pred_cols = [c for c in block.columns if c.startswith("pred_logvar_")]

    structure_pred_df = structure_pred_df.merge(
        block[
            [
                "trade_dt",
                "tenor",
            ] + pred_cols
        ],
        on=["trade_dt", "tenor"],
        how="left",
    )

# Variance and vol versions.
pred_logvar_cols = [c for c in structure_pred_df.columns if c.startswith("pred_logvar_")]

for col in pred_logvar_cols:
    suffix = col.replace("pred_logvar_", "")

    structure_pred_df[f"pred_var_{suffix}"] = np.exp(structure_pred_df[col])
    structure_pred_df[f"pred_vol_pct_{suffix}"] = (
        np.sqrt(structure_pred_df[f"pred_var_{suffix}"].clip(lower=1e-12)) * 100.0
    )

structure_pred_df.to_parquet(STRUCTURE_PRED_PATH, index=False)

# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

model_label_map = {
    "pred_logvar_har_rv": "pooled_har_rv",
    "pred_logvar_har_rv_iv": "pooled_har_rv_iv",
    "pred_logvar_har_rv_per_tenor": "per_tenor_har_rv",
    "pred_logvar_har_rv_iv_per_tenor": "per_tenor_har_rv_iv",
    "pred_logvar_har_rv_by_tenor_group": "tenor_group_har_rv",
    "pred_logvar_har_rv_iv_by_tenor_group": "tenor_group_har_rv_iv",
}

comparison_df = evaluate_predictions(
    structure_pred_df,
    pred_logvar_cols,
    model_label_map=model_label_map,
)

comparison_by_tenor_df = evaluate_predictions_by_tenor(
    structure_pred_df,
    pred_logvar_cols,
    model_label_map=model_label_map,
)

comparison_df = comparison_df.sort_values("log_rmse").reset_index(drop=True)
comparison_by_tenor_df = comparison_by_tenor_df.sort_values(
    ["tenor", "log_rmse"]
).reset_index(drop=True)

comparison_df.to_csv(STRUCTURE_COMPARISON_CSV, index=False)
comparison_by_tenor_df.to_csv(STRUCTURE_COMPARISON_BY_TENOR_CSV, index=False)

print("\nSaved structure predictions:")
print(STRUCTURE_PRED_PATH)

print("\nSaved structure model comparison:")
print(STRUCTURE_COMPARISON_CSV)
print(STRUCTURE_COMPARISON_BY_TENOR_CSV)

print("\nOverall model structure comparison:")
display(comparison_df)

print("\nBest models by tenor using log RMSE:")
display(
    comparison_by_tenor_df
    .sort_values(["tenor", "log_rmse"])
    .groupby("tenor", as_index=False)
    .head(3)
)

print("\nBest models by tenor using vol RMSE:")
display(
    comparison_by_tenor_df
    .sort_values(["tenor", "vol_rmse_pct"])
    .groupby("tenor", as_index=False)
    .head(3)
)

Using modeling panel:
C:\Users\patri\vrp_project\data\processed\forecast_vol_modeling_panel_complete_targets_v0_1.parquet
Rows: 16458
Dates: 2018-06-25 00:00:00 to 2026-06-17 00:00:00

Feature set sizes:
har_rv 11
['log_trailing_realized_variance', 'log_trailing_rv_5d_variance', 'log_trailing_rv_9d_variance', 'log_trailing_rv_12d_variance', 'log_trailing_rv_15d_variance', 'log_trailing_rv_21d_variance', 'log_trailing_rv_30d_variance', 'log_trailing_rv_42d_variance', 'log_trailing_rv_63d_variance', 'log_trailing_rv_126d_variance', 'log_trailing_rv_252d_variance'] 
har_rv_iv 22
['log_trailing_realized_variance', 'log_trailing_rv_5d_variance', 'log_trailing_rv_9d_variance', 'log_trailing_rv_12d_variance', 'log_trailing_rv_15d_variance', 'log_trailing_rv_21d_variance', 'log_trailing_rv_30d_variance', 'log_trailing_rv_42d_variance', 'log_trailing_rv_63d_variance', 'log_trailing_rv_126d_variance', 'log_trailing_rv_252d_variance', 'log_implied_variance', 'vix_style_vol', 'spx_rsi_14', 'iv_slo

,model,rows,start_date,end_date,log_rmse,log_mae,log_bias,vol_rmse_pct,vol_mae_pct,vol_bias_pct,corr_log_prediction_target,corr_vol_prediction_target
0,per_tenor_har_rv_iv,11362,2020-10-01,2026-06-17,0.704790,0.551202,0.074279,6.288608,4.227501,-0.133126,0.600823,0.566814
1,pooled_har_rv_iv,12260,2020-07-01,2026-06-17,0.715400,0.557820,0.114267,6.370327,4.337304,0.239952,0.587839,0.547804
2,tenor_group_har_rv_iv,11632,2020-10-01,2026-06-17,0.716441,0.560604,0.094303,6.401186,4.332174,0.066277,0.587571,0.548880
3,per_tenor_har_rv,11835,2020-07-01,2026-06-17,0.761867,0.598006,0.031894,6.540524,4.485717,-0.640152,0.492245,0.487366
4,tenor_group_har_rv,12069,2020-07-01,2026-06-17,0.764022,0.599963,0.035701,6.556809,4.517917,-0.584502,0.488486,0.479235
5,pooled_har_rv,12260,2020-07-01,2026-06-17,0.765510,0.604364,0.041096,6.542045,4.541975,-0.554804,0.478708,0.474514



Best models by tenor using log RMSE:


,model,tenor,rows,start_date,end_date,log_rmse,log_mae,log_bias,vol_rmse_pct,vol_mae_pct,vol_bias_pct,corr_log_prediction_target,corr_vol_prediction_target
0,per_tenor_har_rv_iv,9,1355,2020-11-02,2026-06-17,0.756449,0.581571,0.015255,6.349706,4.146388,-0.589790,0.665224,0.661538
1,tenor_group_har_rv_iv,9,1377,2020-10-01,2026-06-17,0.766545,0.590136,0.036182,6.526539,4.267209,-0.462985,0.655581,0.642068
2,pooled_har_rv_iv,9,1440,2020-07-01,2026-06-17,0.790661,0.614587,0.100642,6.652603,4.470745,-0.129048,0.625950,0.612317
6,per_tenor_har_rv_iv,12,1417,2020-10-01,2026-06-15,0.715759,0.563566,0.039493,6.067994,4.040950,-0.344355,0.655197,0.635941
7,tenor_group_har_rv_iv,12,1417,2020-10-01,2026-06-15,0.723526,0.570289,0.077742,6.116084,4.109669,-0.048905,0.651783,0.632060
8,pooled_har_rv_iv,12,1481,2020-07-01,2026-06-15,0.753927,0.595073,0.146297,6.289042,4.330254,0.341254,0.617239,0.597317
12,tenor_group_har_rv_iv,15,1424,2020-10-01,2026-06-11,0.700191,0.549540,0.018166,6.237848,4.138359,-0.404795,0.632826,0.604226
13,per_tenor_har_rv_iv,15,1424,2020-10-01,2026-06-11,0.704654,0.552440,0.068286,6.256722,4.176359,-0.105917,0.625234,0.597888
14,pooled_har_rv_iv,15,1488,2020-07-01,2026-06-11,0.726438,0.570516,0.083796,6.403123,4.338419,-0.020710,0.593728,0.565518
18,per_tenor_har_rv_iv,18,1391,2020-10-01,2026-06-09,0.693179,0.543942,0.073477,6.163660,4.139407,-0.077372,0.614174,0.581012



Best models by tenor using vol RMSE:


,model,tenor,rows,start_date,end_date,log_rmse,log_mae,log_bias,vol_rmse_pct,vol_mae_pct,vol_bias_pct,corr_log_prediction_target,corr_vol_prediction_target
0,per_tenor_har_rv_iv,9,1355,2020-11-02,2026-06-17,0.756449,0.581571,0.015255,6.349706,4.146388,-0.589790,0.665224,0.661538
1,tenor_group_har_rv_iv,9,1377,2020-10-01,2026-06-17,0.766545,0.590136,0.036182,6.526539,4.267209,-0.462985,0.655581,0.642068
2,pooled_har_rv_iv,9,1440,2020-07-01,2026-06-17,0.790661,0.614587,0.100642,6.652603,4.470745,-0.129048,0.625950,0.612317
6,per_tenor_har_rv_iv,12,1417,2020-10-01,2026-06-15,0.715759,0.563566,0.039493,6.067994,4.040950,-0.344355,0.655197,0.635941
7,tenor_group_har_rv_iv,12,1417,2020-10-01,2026-06-15,0.723526,0.570289,0.077742,6.116084,4.109669,-0.048905,0.651783,0.632060
8,pooled_har_rv_iv,12,1481,2020-07-01,2026-06-15,0.753927,0.595073,0.146297,6.289042,4.330254,0.341254,0.617239,0.597317
12,tenor_group_har_rv_iv,15,1424,2020-10-01,2026-06-11,0.700191,0.549540,0.018166,6.237848,4.138359,-0.404795,0.632826,0.604226
13,per_tenor_har_rv_iv,15,1424,2020-10-01,2026-06-11,0.704654,0.552440,0.068286,6.256722,4.176359,-0.105917,0.625234,0.597888
14,pooled_har_rv_iv,15,1488,2020-07-01,2026-06-11,0.726438,0.570516,0.083796,6.403123,4.338419,-0.020710,0.593728,0.565518
18,per_tenor_har_rv_iv,18,1391,2020-10-01,2026-06-09,0.693179,0.543942,0.073477,6.163660,4.139407,-0.077372,0.614174,0.581012


In [28]:
# ============================================================
# Cell 27: Build denominator candidate panel v0.2
# ============================================================

import numpy as np
import pandas as pd

STRUCTURE_PRED_PATH = (
    PROCESSED_DATA_DIR / "forecast_vol_model_structure_predictions_v0_1.parquet"
)

COMPLETE_PANEL_PATH = (
    PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_complete_targets_v0_1.parquet"
)

if not COMPLETE_PANEL_PATH.exists():
    COMPLETE_PANEL_PATH = (
        PROCESSED_DATA_DIR / "forecast_vol_modeling_panel_v0_1.parquet"
    )

DENOM_CANDIDATE_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_denominator_candidate_panel_v0_2.parquet"
)

DENOM_CANDIDATE_PANEL_CSV = (
    AUDIT_DIR / "forecast_vol_denominator_candidate_panel_sample_v0_2.csv"
)

structure_df = pd.read_parquet(STRUCTURE_PRED_PATH).copy()
info_df = pd.read_parquet(COMPLETE_PANEL_PATH).copy()

for df in [structure_df, info_df]:
    if "trade_dt" not in df.columns:
        df["trade_dt"] = pd.to_datetime(
            df["trade_date"].astype(str),
            format="%Y%m%d",
            errors="coerce",
        )
    else:
        df["trade_dt"] = pd.to_datetime(df["trade_dt"])

    df["tenor"] = df["tenor"].astype(int)

if "target_log_variance" not in info_df.columns:
    if "log_forward_realized_variance" in info_df.columns:
        info_df["target_log_variance"] = info_df["log_forward_realized_variance"]
    else:
        raise ValueError("Could not find target_log_variance.")

if "target_variance" not in info_df.columns:
    if "forward_realized_variance" in info_df.columns:
        info_df["target_variance"] = info_df["forward_realized_variance"]
    else:
        info_df["target_variance"] = np.exp(info_df["target_log_variance"])

if "target_vol_pct" not in info_df.columns:
    info_df["target_vol_pct"] = np.sqrt(info_df["target_variance"].clip(lower=1e-12)) * 100.0

base_cols = [
    "trade_date",
    "trade_dt",
    "tenor",
    "spx_close",
    "spx_rsi_14",
    "implied_variance",
    "vix_style_vol",
    "target_variance",
    "target_vol_pct",
    "target_log_variance",
]

base_cols = [c for c in base_cols if c in info_df.columns]

pred_cols = [
    c for c in structure_df.columns
    if c.startswith("pred_logvar_")
]

denom_df = info_df[base_cols].merge(
    structure_df[
        [
            "trade_dt",
            "tenor",
        ] + pred_cols
    ],
    on=["trade_dt", "tenor"],
    how="left",
)

# ------------------------------------------------------------
# Helper: choose prediction by tenor with fallback
# ------------------------------------------------------------

def choose_by_tenor(df, tenor_to_suffix, fallback_suffix):
    out = pd.Series(np.nan, index=df.index, dtype=float)

    for tenor, suffix in tenor_to_suffix.items():
        col = f"pred_logvar_{suffix}"

        if col not in df.columns:
            col = f"pred_logvar_{fallback_suffix}"

        mask = df["tenor"].eq(int(tenor))

        if col in df.columns:
            out.loc[mask] = df.loc[mask, col]

    fallback_col = f"pred_logvar_{fallback_suffix}"

    if fallback_col in df.columns:
        out = out.fillna(df[fallback_col])

    return out

# Hybrid 1:
# Best-ish log objective from structure diagnostics.
hybrid_log_map = {
    9: "har_rv_iv_per_tenor",
    12: "har_rv_iv_per_tenor",
    15: "har_rv_iv_by_tenor_group",
    18: "har_rv_iv_per_tenor",
    21: "har_rv_iv",
    24: "har_rv_iv_by_tenor_group",
    27: "har_rv_iv",
    30: "har_rv_iv",
    33: "har_rv_iv",
}

# Hybrid 2:
# HAR-IV for front/middle, HAR-only for back.
hybrid_back_har_only_map = {
    9: "har_rv_iv_per_tenor",
    12: "har_rv_iv_per_tenor",
    15: "har_rv_iv_by_tenor_group",
    18: "har_rv_iv_per_tenor",
    21: "har_rv_iv",
    24: "har_rv_iv",
    27: "har_rv",
    30: "har_rv",
    33: "har_rv",
}

denom_df["pred_logvar_hybrid_log_objective"] = choose_by_tenor(
    denom_df,
    hybrid_log_map,
    fallback_suffix="har_rv_iv",
)

denom_df["pred_logvar_hybrid_back_har_only"] = choose_by_tenor(
    denom_df,
    hybrid_back_har_only_map,
    fallback_suffix="har_rv_iv",
)

candidate_suffixes = [
    c.replace("pred_logvar_", "")
    for c in denom_df.columns
    if c.startswith("pred_logvar_")
]

candidate_suffixes = sorted(set(candidate_suffixes))

eps = 1e-12

for suffix in candidate_suffixes:
    log_col = f"pred_logvar_{suffix}"

    denom_df[f"pred_var_{suffix}"] = np.exp(denom_df[log_col])

    denom_df[f"pred_vol_pct_{suffix}"] = (
        np.sqrt(denom_df[f"pred_var_{suffix}"].clip(lower=eps)) * 100.0
    )

    denom_df[f"vrp_signal_{suffix}"] = (
        np.log(denom_df["implied_variance"].clip(lower=eps))
        - denom_df[log_col]
    )

    denom_df[f"vrp_vol_ratio_{suffix}"] = (
        denom_df["vix_style_vol"]
        / denom_df[f"pred_vol_pct_{suffix}"].clip(lower=eps)
    )

denom_df = denom_df.sort_values(["trade_dt", "tenor"]).reset_index(drop=True)

denom_df.to_parquet(DENOM_CANDIDATE_PANEL_PARQUET, index=False)
denom_df.head(2000).to_csv(DENOM_CANDIDATE_PANEL_CSV, index=False)

print("Saved denominator candidate panel:")
print(DENOM_CANDIDATE_PANEL_PARQUET)
print(DENOM_CANDIDATE_PANEL_CSV)

print("\nRows:", len(denom_df))
print("Dates:", denom_df["trade_dt"].min(), "to", denom_df["trade_dt"].max())
print("Tenors:", sorted(denom_df["tenor"].dropna().unique()))

print("\nCandidate suffixes:")
for suffix in candidate_suffixes:
    print(suffix)

print("\nMissingness of key candidate VRP signals:")
vrp_cols = [
    f"vrp_signal_{suffix}"
    for suffix in candidate_suffixes
]

display(
    denom_df[vrp_cols]
    .isna()
    .mean()
    .sort_values()
    .to_frame("missing_pct")
)

print("\nTail sample:")
display(
    denom_df[
        [
            "trade_dt",
            "tenor",
            "vix_style_vol",
            "target_vol_pct",
            "pred_vol_pct_har_rv_iv",
            "pred_vol_pct_har_rv_iv_per_tenor",
            "pred_vol_pct_hybrid_log_objective",
            "pred_vol_pct_hybrid_back_har_only",
            "vrp_signal_har_rv_iv",
            "vrp_signal_har_rv_iv_per_tenor",
            "vrp_signal_hybrid_log_objective",
            "vrp_signal_hybrid_back_har_only",
        ]
    ].tail(30)
)

Saved denominator candidate panel:
C:\Users\patri\vrp_project\data\processed\forecast_vol_denominator_candidate_panel_v0_2.parquet
C:\Users\patri\vrp_project\data\audit\forecast_vol_denominator_candidate_panel_sample_v0_2.csv

Rows: 16458
Dates: 2018-06-25 00:00:00 to 2026-06-17 00:00:00
Tenors: [np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21), np.int64(24), np.int64(27), np.int64(30), np.int64(33)]

Candidate suffixes:
har_rv
har_rv_by_tenor_group
har_rv_iv
har_rv_iv_by_tenor_group
har_rv_iv_per_tenor
har_rv_per_tenor
hybrid_back_har_only
hybrid_log_objective

Missingness of key candidate VRP signals:


,missing_pct
vrp_signal_har_rv,0.255074
vrp_signal_har_rv_iv,0.255074
vrp_signal_hybrid_back_har_only,0.255074
vrp_signal_hybrid_log_objective,0.255074
vrp_signal_har_rv_by_tenor_group,0.266679
vrp_signal_har_rv_per_tenor,0.280897
vrp_signal_har_rv_iv_by_tenor_group,0.293231
vrp_signal_har_rv_iv_per_tenor,0.309637



Tail sample:


,trade_dt,tenor,vix_style_vol,target_vol_pct,pred_vol_pct_har_rv_iv,pred_vol_pct_har_rv_iv_per_tenor,pred_vol_pct_hybrid_log_objective,pred_vol_pct_hybrid_back_har_only,vrp_signal_har_rv_iv,vrp_signal_har_rv_iv_per_tenor,vrp_signal_hybrid_log_objective,vrp_signal_hybrid_back_har_only
16428,2026-06-03,18,15.035602,19.683591,11.087884,10.653038,10.653038,10.653038,0.609136,0.689152,0.689152,0.689152
16429,2026-06-03,21,15.315963,19.262255,11.311058,11.479151,11.311058,11.311058,0.606230,0.576726,0.606230,0.606230
16430,2026-06-04,9,12.405099,23.201164,9.304364,8.311384,8.311384,8.311384,0.575248,0.800963,0.800963,0.800963
16431,2026-06-04,12,13.330823,22.256367,9.972846,8.953933,8.953933,8.953933,0.580426,0.795972,0.795972,0.795972
16432,2026-06-04,15,13.859725,21.469690,10.345092,10.122035,9.584046,9.584046,0.584950,0.628545,0.737775,0.737775
16433,2026-06-04,18,14.204839,19.670204,10.624582,10.219460,10.219460,10.219460,0.580825,0.658578,0.658578,0.658578
16434,2026-06-04,21,14.446303,19.188272,10.813296,11.088255,10.813296,10.813296,0.579324,0.529104,0.579324,0.579324
16435,2026-06-05,9,21.280719,15.715545,17.899589,17.134299,17.134299,17.134299,0.346047,0.433438,0.433438,0.433438
16436,2026-06-05,12,21.320016,17.952708,17.902170,17.042263,17.042263,17.042263,0.349449,0.447900,0.447900,0.447900
16437,2026-06-05,15,21.029687,16.916211,17.653435,17.547220,16.838013,16.838013,0.350009,0.362079,0.444592,0.444592


In [29]:
# ============================================================
# Cell 28: Build denominator candidate VRP z-score panel v0.2
# ============================================================

import numpy as np
import pandas as pd

DENOM_CANDIDATE_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_denominator_candidate_panel_v0_2.parquet"
)

DENOM_CANDIDATE_ZSCORE_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_denominator_candidate_zscore_panel_v0_2.parquet"
)

DENOM_CANDIDATE_ZSCORE_PANEL_CSV = (
    AUDIT_DIR / "forecast_vol_denominator_candidate_zscore_panel_sample_v0_2.csv"
)

z_df = pd.read_parquet(DENOM_CANDIDATE_PANEL_PARQUET).copy()

z_df["trade_dt"] = pd.to_datetime(z_df["trade_dt"])
z_df["tenor"] = z_df["tenor"].astype(int)

z_df = z_df.sort_values(["tenor", "trade_dt"]).reset_index(drop=True)

vrp_signal_cols = [
    c for c in z_df.columns
    if c.startswith("vrp_signal_")
]

g = z_df.groupby("tenor", group_keys=False)

for signal_col in vrp_signal_cols:
    suffix = signal_col.replace("vrp_signal_", "")

    for window, label in [(63, "3m"), (252, "1y")]:
        mean_col = f"{signal_col}_mean_{label}"
        std_col = f"{signal_col}_std_{label}"
        z_col = f"{signal_col}_z_{label}"

        z_df[mean_col] = (
            g[signal_col]
            .rolling(window, min_periods=max(20, window // 3))
            .mean()
            .reset_index(level=0, drop=True)
        )

        z_df[std_col] = (
            g[signal_col]
            .rolling(window, min_periods=max(20, window // 3))
            .std()
            .reset_index(level=0, drop=True)
        )

        z_df[z_col] = (
            (z_df[signal_col] - z_df[mean_col])
            / z_df[std_col].replace(0, np.nan)
        )

    z_df[f"vrp_signal_{suffix}_min_z"] = z_df[
        [
            f"{signal_col}_z_3m",
            f"{signal_col}_z_1y",
        ]
    ].min(axis=1)

z_df.to_parquet(DENOM_CANDIDATE_ZSCORE_PANEL_PARQUET, index=False)
z_df.head(2000).to_csv(DENOM_CANDIDATE_ZSCORE_PANEL_CSV, index=False)

print("Saved denominator candidate z-score panel:")
print(DENOM_CANDIDATE_ZSCORE_PANEL_PARQUET)
print(DENOM_CANDIDATE_ZSCORE_PANEL_CSV)

print("\nRows:", len(z_df))
print("Dates:", z_df["trade_dt"].min(), "to", z_df["trade_dt"].max())

key_cols = [
    "trade_dt",
    "tenor",
    "vix_style_vol",
    "vrp_signal_har_rv_iv",
    "vrp_signal_har_rv_iv_min_z",
    "vrp_signal_har_rv_iv_per_tenor",
    "vrp_signal_har_rv_iv_per_tenor_min_z",
    "vrp_signal_hybrid_log_objective",
    "vrp_signal_hybrid_log_objective_min_z",
    "vrp_signal_hybrid_back_har_only",
    "vrp_signal_hybrid_back_har_only_min_z",
]

key_cols = [c for c in key_cols if c in z_df.columns]

display(z_df[key_cols].tail(30))

print("\nZ-score missingness:")
z_cols = [
    c for c in z_df.columns
    if c.endswith("_min_z")
]

display(
    z_df[z_cols]
    .isna()
    .mean()
    .sort_values()
    .to_frame("missing_pct")
)

Saved denominator candidate z-score panel:
C:\Users\patri\vrp_project\data\processed\forecast_vol_denominator_candidate_zscore_panel_v0_2.parquet
C:\Users\patri\vrp_project\data\audit\forecast_vol_denominator_candidate_zscore_panel_sample_v0_2.csv

Rows: 16458
Dates: 2018-06-25 00:00:00 to 2026-06-17 00:00:00


,trade_dt,tenor,vix_style_vol,vrp_signal_har_rv_iv,vrp_signal_har_rv_iv_min_z,vrp_signal_har_rv_iv_per_tenor,vrp_signal_har_rv_iv_per_tenor_min_z,vrp_signal_hybrid_log_objective,vrp_signal_hybrid_log_objective_min_z,vrp_signal_hybrid_back_har_only,vrp_signal_hybrid_back_har_only_min_z
16428,2026-04-07,33,25.951809,0.202474,-2.314326,0.364922,-1.792172,0.202474,-2.314326,1.106689,0.461219
16429,2026-04-08,33,21.094953,0.502558,-0.051237,0.432593,-0.667502,0.502558,-0.051237,0.338566,-2.206554
16430,2026-04-09,33,19.584822,0.575568,0.310022,0.442917,-0.500722,0.575568,0.310022,0.303961,-2.211072
16431,2026-04-10,33,19.403247,0.562405,0.245176,0.459296,-0.395710,0.562405,0.245176,0.329063,-2.036804
16432,2026-04-13,33,19.188112,0.457344,-0.273032,0.436641,-0.702649,0.457344,-0.273032,0.398556,-1.765154
16433,2026-04-14,33,18.478441,0.458234,-0.268350,0.381797,-1.684624,0.458234,-0.268350,0.251633,-2.131845
16434,2026-04-15,33,18.308379,0.466157,-0.227280,0.390167,-1.523352,0.466157,-0.227280,0.226997,-2.102932
16435,2026-04-16,33,18.127184,0.481828,-0.147540,0.418721,-1.016806,0.481828,-0.147540,0.226149,-2.025212
16436,2026-04-17,33,17.805406,0.507878,-0.017875,0.387876,-1.511728,0.507878,-0.017875,0.256999,-1.855013
16437,2026-04-20,33,19.018559,0.385020,-0.624525,0.312713,-2.630898,0.385020,-0.624525,0.490869,-1.166261



Z-score missingness:


,missing_pct
vrp_signal_har_rv_min_z,0.266010
vrp_signal_har_rv_iv_min_z,0.266010
vrp_signal_hybrid_back_har_only_min_z,0.266010
vrp_signal_hybrid_log_objective_min_z,0.266010
vrp_signal_har_rv_by_tenor_group_min_z,0.277616
vrp_signal_har_rv_per_tenor_min_z,0.291834
vrp_signal_har_rv_iv_by_tenor_group_min_z,0.304168
vrp_signal_har_rv_iv_per_tenor_min_z,0.320574


In [ ]:
# ============================================================
# Cell 29: Trade-rule sweep across v0.2 denominator candidates
# Common-sample comparison
# ============================================================

import numpy as np
import pandas as pd
from itertools import product

RECAL_UNIVERSE_PARQUET = (
    PROCESSED_DATA_DIR / "put_champion_recalibration_universe_v0_1.parquet"
)

DENOM_ZSCORE_PANEL_PARQUET = (
    PROCESSED_DATA_DIR / "forecast_vol_denominator_candidate_zscore_panel_v0_2.parquet"
)

DENOM_TRADE_SWEEP_CSV = (
    AUDIT_DIR / "put_denominator_candidate_trade_sweep_v0_2.csv"
)

DENOM_TRADE_SWEEP_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_denominator_candidate_trade_sweep_trades_v0_2.parquet"
)

recal_df = pd.read_parquet(RECAL_UNIVERSE_PARQUET).copy()
denom_z_df = pd.read_parquet(DENOM_ZSCORE_PANEL_PARQUET).copy()

recal_df["trade_dt"] = pd.to_datetime(recal_df["trade_dt"])
denom_z_df["trade_dt"] = pd.to_datetime(denom_z_df["trade_dt"])

recal_df["entry_tenor"] = recal_df["entry_tenor"].astype(int)
denom_z_df["tenor"] = denom_z_df["tenor"].astype(int)

main_denom_suffixes = [
    "har_rv_iv",                    # pooled champion v0.1
    "har_rv_iv_per_tenor",          # per-tenor champion v0.2 candidate
    "hybrid_log_objective",         # best-ish log-RMSE hybrid
    "hybrid_back_har_only",         # HAR-IV front/middle, HAR-only back
]

available_suffixes = []

for suffix in main_denom_suffixes:
    signal_col = f"vrp_signal_{suffix}"
    z_col = f"vrp_signal_{suffix}_min_z"

    if signal_col in denom_z_df.columns and z_col in denom_z_df.columns:
        available_suffixes.append(suffix)
    else:
        print("Missing denominator candidate:", suffix)

main_denom_suffixes = available_suffixes

merge_cols = ["trade_dt", "tenor"]

for suffix in main_denom_suffixes:
    merge_cols += [
        f"vrp_signal_{suffix}",
        f"vrp_signal_{suffix}_min_z",
        f"pred_vol_pct_{suffix}",
    ]

merge_cols = [c for c in merge_cols if c in denom_z_df.columns]

trade_df = recal_df.merge(
    denom_z_df[merge_cols],
    left_on=["trade_dt", "entry_tenor"],
    right_on=["trade_dt", "tenor"],
    how="left",
    suffixes=("", "_denom"),
)

# Common sample across all main denominator candidates.
required_cols = [
    "test_pnl",
    "simple_carry_vrp_signal",
    "simple_carry_vrp_min_z",
    "trailing_carry_vrp_signal",
    "trailing_carry_vrp_min_z",
    "entry_rsi_14",
]

for suffix in main_denom_suffixes:
    required_cols += [
        f"vrp_signal_{suffix}",
        f"vrp_signal_{suffix}_min_z",
    ]

clean_df = trade_df.dropna(subset=required_cols).copy()
clean_df = clean_df.sort_values(["trade_dt", "entry_tenor"]).reset_index(drop=True)

clean_df["trade_year"] = clean_df["trade_dt"].dt.year

total_trade_dates = clean_df["trade_dt"].nunique()

print("Common denominator trade sample:")
print("Rows:", len(clean_df))
print("Dates:", clean_df["trade_dt"].min(), "to", clean_df["trade_dt"].max())
print("Unique trade dates:", total_trade_dates)
print("Tenors:", sorted(clean_df["entry_tenor"].unique()))
print("Denominator candidates:", main_denom_suffixes)

display(
    clean_df[
        [
            "test_pnl",
            "simple_carry_vrp_signal",
            "simple_carry_vrp_min_z",
            "trailing_carry_vrp_signal",
            "trailing_carry_vrp_min_z",
            "entry_rsi_14",
        ]
        + [f"vrp_signal_{s}" for s in main_denom_suffixes]
        + [f"vrp_signal_{s}_min_z" for s in main_denom_suffixes]
    ].describe().T
)


def max_drawdown_from_pnl(pnl_series):
    pnl = pd.Series(pnl_series).fillna(0.0)
    equity = pnl.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    return float(dd.min())


def summarize_selected_trades(selected_df, rule_name, params):
    selected_df = selected_df.sort_values("trade_dt").copy()

    if selected_df.empty:
        return {
            "rule_name": rule_name,
            **params,
            "trades": 0,
            "frequency": 0.0,
            "total_pnl": 0.0,
            "avg_pnl": np.nan,
            "median_pnl": np.nan,
            "win_rate": np.nan,
            "q05_pnl": np.nan,
            "worst_pnl": np.nan,
            "max_drawdown": np.nan,
            "worst_10_trade_sum": np.nan,
            "worst_20_trade_sum": np.nan,
            "positive_year_rate": np.nan,
            "min_year_pnl": np.nan,
            "avg_rsi": np.nan,
            "avg_tenor": np.nan,
            "front_pct": np.nan,
            "middle_pct": np.nan,
            "back_pct": np.nan,
        }

    pnl = selected_df["test_pnl"].astype(float)
    yearly = selected_df.groupby("trade_year")["test_pnl"].sum()

    roll10 = pnl.rolling(10).sum()
    roll20 = pnl.rolling(20).sum()

    return {
        "rule_name": rule_name,
        **params,
        "trades": len(selected_df),
        "frequency": len(selected_df) / total_trade_dates,
        "total_pnl": pnl.sum(),
        "avg_pnl": pnl.mean(),
        "median_pnl": pnl.median(),
        "win_rate": (pnl > 0).mean(),
        "q05_pnl": pnl.quantile(0.05),
        "worst_pnl": pnl.min(),
        "max_drawdown": max_drawdown_from_pnl(pnl),
        "worst_10_trade_sum": roll10.min(),
        "worst_20_trade_sum": roll20.min(),
        "positive_year_rate": (yearly > 0).mean(),
        "min_year_pnl": yearly.min(),
        "avg_rsi": selected_df["entry_rsi_14"].mean(),
        "avg_tenor": selected_df["entry_tenor"].mean(),
        "front_pct": selected_df["tenor_group"].eq("front_9_15d").mean(),
        "middle_pct": selected_df["tenor_group"].eq("middle_18_24d").mean(),
        "back_pct": selected_df["tenor_group"].eq("back_27_33d").mean(),
    }


def select_one_per_day(candidate_df, score_col):
    if candidate_df.empty:
        return candidate_df.copy()

    selected = (
        candidate_df
        .sort_values(
            ["trade_dt", score_col, "entry_tenor"],
            ascending=[True, False, False],
        )
        .groupby("trade_dt", as_index=False)
        .head(1)
        .copy()
    )

    return selected.sort_values("trade_dt").reset_index(drop=True)


# ------------------------------------------------------------
# Parameter grids
# ------------------------------------------------------------

rsi_caps = [58, 62, 68, 72, 77]

old_signal_thresholds = [0.50, 0.70, 0.90, 1.10]
old_z_thresholds = [-0.50, 0.00, 0.30, 0.50]

simple_signal_thresholds = [0.40, 0.50, 0.60, 0.75, 0.90, 1.10]
simple_z_thresholds = [-0.50, 0.00, 0.30, 0.50, 0.70, 1.00]

denom_signal_thresholds = [0.00, 0.25, 0.40, 0.50, 0.60]
denom_z_thresholds = [-1.00, -0.50, 0.00, 0.30, 0.50]


summary_rows = []
selected_trade_blocks = []

rule_id = 0


# ------------------------------------------------------------
# Reference family: old_simple_confirm on same common sample
# ------------------------------------------------------------

for rsi_cap, old_sig_thr, old_z_thr, simple_sig_thr, simple_z_thr in product(
    rsi_caps,
    old_signal_thresholds,
    old_z_thresholds,
    simple_signal_thresholds,
    simple_z_thresholds,
):
    temp = clean_df.copy()

    mask = (
        (temp["entry_rsi_14"] < rsi_cap)
        & (temp["trailing_carry_vrp_signal"] > old_sig_thr)
        & (temp["trailing_carry_vrp_min_z"] > old_z_thr)
        & (temp["simple_carry_vrp_signal"] > simple_sig_thr)
        & (temp["simple_carry_vrp_min_z"] > simple_z_thr)
    )

    candidates = temp[mask].copy()

    candidates["score"] = (
        np.minimum(
            candidates["trailing_carry_vrp_signal"],
            candidates["simple_carry_vrp_signal"],
        )
        + 0.50 * np.minimum(
            candidates["trailing_carry_vrp_min_z"],
            candidates["simple_carry_vrp_min_z"],
        )
    )

    selected = select_one_per_day(candidates, "score")

    params = {
        "rule_id": rule_id,
        "denominator_model": "none_reference",
        "rsi_cap": rsi_cap,
        "old_signal_threshold": old_sig_thr,
        "old_z_threshold": old_z_thr,
        "simple_signal_threshold": simple_sig_thr,
        "simple_z_threshold": simple_z_thr,
        "denom_signal_threshold": np.nan,
        "denom_z_threshold": np.nan,
    }

    summary_rows.append(
        summarize_selected_trades(selected, "old_simple_confirm_reference", params)
    )

    selected["rule_id"] = rule_id
    selected["rule_name"] = "old_simple_confirm_reference"
    selected["denominator_model"] = "none_reference"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Reference family: simple_only on same common sample
# ------------------------------------------------------------

for rsi_cap, simple_sig_thr, simple_z_thr in product(
    rsi_caps,
    simple_signal_thresholds,
    simple_z_thresholds,
):
    temp = clean_df.copy()

    mask = (
        (temp["entry_rsi_14"] < rsi_cap)
        & (temp["simple_carry_vrp_signal"] > simple_sig_thr)
        & (temp["simple_carry_vrp_min_z"] > simple_z_thr)
    )

    candidates = temp[mask].copy()

    candidates["score"] = (
        candidates["simple_carry_vrp_signal"]
        + 0.50 * candidates["simple_carry_vrp_min_z"]
    )

    selected = select_one_per_day(candidates, "score")

    params = {
        "rule_id": rule_id,
        "denominator_model": "none_reference",
        "rsi_cap": rsi_cap,
        "old_signal_threshold": np.nan,
        "old_z_threshold": np.nan,
        "simple_signal_threshold": simple_sig_thr,
        "simple_z_threshold": simple_z_thr,
        "denom_signal_threshold": np.nan,
        "denom_z_threshold": np.nan,
    }

    summary_rows.append(
        summarize_selected_trades(selected, "simple_only_reference", params)
    )

    selected["rule_id"] = rule_id
    selected["rule_name"] = "simple_only_reference"
    selected["denominator_model"] = "none_reference"
    selected_trade_blocks.append(selected)

    rule_id += 1


# ------------------------------------------------------------
# Candidate denominator families
# ------------------------------------------------------------

for suffix in main_denom_suffixes:
    denom_signal_col = f"vrp_signal_{suffix}"
    denom_z_col = f"vrp_signal_{suffix}_min_z"

    # Candidate-only.
    for rsi_cap, denom_sig_thr, denom_z_thr in product(
        rsi_caps,
        denom_signal_thresholds,
        denom_z_thresholds,
    ):
        temp = clean_df.copy()

        mask = (
            (temp["entry_rsi_14"] < rsi_cap)
            & (temp[denom_signal_col] > denom_sig_thr)
            & (temp[denom_z_col] > denom_z_thr)
        )

        candidates = temp[mask].copy()

        candidates["score"] = (
            candidates[denom_signal_col]
            + 0.50 * candidates[denom_z_col]
        )

        selected = select_one_per_day(candidates, "score")

        params = {
            "rule_id": rule_id,
            "denominator_model": suffix,
            "rsi_cap": rsi_cap,
            "old_signal_threshold": np.nan,
            "old_z_threshold": np.nan,
            "simple_signal_threshold": np.nan,
            "simple_z_threshold": np.nan,
            "denom_signal_threshold": denom_sig_thr,
            "denom_z_threshold": denom_z_thr,
        }

        summary_rows.append(
            summarize_selected_trades(selected, "denom_only", params)
        )

        selected["rule_id"] = rule_id
        selected["rule_name"] = "denom_only"
        selected["denominator_model"] = suffix
        selected_trade_blocks.append(selected)

        rule_id += 1

    # Simple + denominator confirm.
    for rsi_cap, simple_sig_thr, simple_z_thr, denom_sig_thr, denom_z_thr in product(
        rsi_caps,
        simple_signal_thresholds,
        simple_z_thresholds,
        denom_signal_thresholds,
        denom_z_thresholds,
    ):
        temp = clean_df.copy()

        mask = (
            (temp["entry_rsi_14"] < rsi_cap)
            & (temp["simple_carry_vrp_signal"] > simple_sig_thr)
            & (temp["simple_carry_vrp_min_z"] > simple_z_thr)
            & (temp[denom_signal_col] > denom_sig_thr)
            & (temp[denom_z_col] > denom_z_thr)
        )

        candidates = temp[mask].copy()

        candidates["score"] = (
            np.minimum(
                candidates["simple_carry_vrp_signal"],
                candidates[denom_signal_col],
            )
            + 0.50 * np.minimum(
                candidates["simple_carry_vrp_min_z"],
                candidates[denom_z_col],
            )
        )

        selected = select_one_per_day(candidates, "score")

        params = {
            "rule_id": rule_id,
            "denominator_model": suffix,
            "rsi_cap": rsi_cap,
            "old_signal_threshold": np.nan,
            "old_z_threshold": np.nan,
            "simple_signal_threshold": simple_sig_thr,
            "simple_z_threshold": simple_z_thr,
            "denom_signal_threshold": denom_sig_thr,
            "denom_z_threshold": denom_z_thr,
        }

        summary_rows.append(
            summarize_selected_trades(selected, "simple_denom_confirm", params)
        )

        selected["rule_id"] = rule_id
        selected["rule_name"] = "simple_denom_confirm"
        selected["denominator_model"] = suffix
        selected_trade_blocks.append(selected)

        rule_id += 1

    # Old + simple + denominator confirm.
    for (
        rsi_cap,
        old_sig_thr,
        old_z_thr,
        simple_sig_thr,
        simple_z_thr,
        denom_sig_thr,
        denom_z_thr,
    ) in product(
        rsi_caps,
        old_signal_thresholds,
        old_z_thresholds,
        [0.40, 0.50, 0.75],
        [-0.50, 0.00, 0.50],
        [0.00, 0.25, 0.40],
        [-1.00, -0.50, 0.00],
    ):
        temp = clean_df.copy()

        mask = (
            (temp["entry_rsi_14"] < rsi_cap)
            & (temp["trailing_carry_vrp_signal"] > old_sig_thr)
            & (temp["trailing_carry_vrp_min_z"] > old_z_thr)
            & (temp["simple_carry_vrp_signal"] > simple_sig_thr)
            & (temp["simple_carry_vrp_min_z"] > simple_z_thr)
            & (temp[denom_signal_col] > denom_sig_thr)
            & (temp[denom_z_col] > denom_z_thr)
        )

        candidates = temp[mask].copy()

        candidates["score"] = (
            np.minimum.reduce([
                candidates["trailing_carry_vrp_signal"].values,
                candidates["simple_carry_vrp_signal"].values,
                candidates[denom_signal_col].values,
            ])
            + 0.50 * np.minimum.reduce([
                candidates["trailing_carry_vrp_min_z"].values,
                candidates["simple_carry_vrp_min_z"].values,
                candidates[denom_z_col].values,
            ])
        )

        selected = select_one_per_day(candidates, "score")

        params = {
            "rule_id": rule_id,
            "denominator_model": suffix,
            "rsi_cap": rsi_cap,
            "old_signal_threshold": old_sig_thr,
            "old_z_threshold": old_z_thr,
            "simple_signal_threshold": simple_sig_thr,
            "simple_z_threshold": simple_z_thr,
            "denom_signal_threshold": denom_sig_thr,
            "denom_z_threshold": denom_z_thr,
        }

        summary_rows.append(
            summarize_selected_trades(selected, "old_simple_denom_confirm", params)
        )

        selected["rule_id"] = rule_id
        selected["rule_name"] = "old_simple_denom_confirm"
        selected["denominator_model"] = suffix
        selected_trade_blocks.append(selected)

        rule_id += 1


sweep_df = pd.DataFrame(summary_rows)

selected_trades_df = pd.concat(
    selected_trade_blocks,
    ignore_index=True,
) if selected_trade_blocks else pd.DataFrame()

sweep_df = sweep_df.sort_values(
    ["total_pnl", "avg_pnl", "win_rate"],
    ascending=[False, False, False],
).reset_index(drop=True)

sweep_df.to_csv(DENOM_TRADE_SWEEP_CSV, index=False)
selected_trades_df.to_parquet(DENOM_TRADE_SWEEP_TRADES_PARQUET, index=False)

print("Saved denominator trade sweep:")
print(DENOM_TRADE_SWEEP_CSV)
print(DENOM_TRADE_SWEEP_TRADES_PARQUET)

print("\nSweep rows:", len(sweep_df))
print("Selected trade rows:", len(selected_trades_df))

print("\nTop 30 by total P&L, at least 100 trades:")
display(
    sweep_df
    .query("trades >= 100")
    .sort_values(["total_pnl", "avg_pnl"], ascending=[False, False])
    .head(30)
)

print("\nTop 30 by avg P&L, 50-250 trades:")
display(
    sweep_df
    .query("trades >= 50 and trades <= 250")
    .sort_values(["avg_pnl", "total_pnl"], ascending=[False, False])
    .head(30)
)

print("\nBest by rule name / denominator model, at least 50 trades:")
display(
    sweep_df
    .query("trades >= 50")
    .sort_values(["rule_name", "denominator_model", "total_pnl"], ascending=[True, True, False])
    .groupby(["rule_name", "denominator_model"], as_index=False)
    .head(3)
    .sort_values(["rule_name", "denominator_model", "total_pnl"], ascending=[True, True, False])
)

print("\nBest denominator model inside simple_denom_confirm, at least 50 trades:")
display(
    sweep_df
    .query("rule_name == 'simple_denom_confirm' and trades >= 50")
    .sort_values(["denominator_model", "total_pnl"], ascending=[True, False])
    .groupby("denominator_model", as_index=False)
    .head(5)
)

print("\nBest denominator model inside old_simple_denom_confirm, at least 50 trades:")
display(
    sweep_df
    .query("rule_name == 'old_simple_denom_confirm' and trades >= 50")
    .sort_values(["denominator_model", "total_pnl"], ascending=[True, False])
    .groupby("denominator_model", as_index=False)
    .head(5)
)

In [11]:
# ============================================================
# Cell 30: Shortlist denominator candidate rules
# ============================================================

import numpy as np
import pandas as pd

DENOM_TRADE_SWEEP_CSV = (
    AUDIT_DIR / "put_denominator_candidate_trade_sweep_v0_2.csv"
)

DENOM_TRADE_SWEEP_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_denominator_candidate_trade_sweep_trades_v0_2.parquet"
)

SHORTLIST_RULES_CSV = (
    AUDIT_DIR / "put_denominator_candidate_shortlist_rules_v0_2.csv"
)

SHORTLIST_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_denominator_candidate_shortlist_trades_v0_2.parquet"
)

SHORTLIST_BY_YEAR_CSV = (
    AUDIT_DIR / "put_denominator_candidate_shortlist_by_year_v0_2.csv"
)

SHORTLIST_BY_TENOR_CSV = (
    AUDIT_DIR / "put_denominator_candidate_shortlist_by_tenor_v0_2.csv"
)

sweep_df = pd.read_csv(DENOM_TRADE_SWEEP_CSV)
trades_df = pd.read_parquet(DENOM_TRADE_SWEEP_TRADES_PARQUET).copy()

trades_df["trade_dt"] = pd.to_datetime(trades_df["trade_dt"])
trades_df["trade_year"] = trades_df["trade_dt"].dt.year

# Helper score: total P&L relative to drawdown.
sweep_df["pnl_to_drawdown"] = (
    sweep_df["total_pnl"]
    / sweep_df["max_drawdown"].abs().replace(0, np.nan)
)

shortlist_blocks = []

def add_best(label, query, sort_cols, ascending=None, n=1):
    temp = sweep_df.query(query).copy()

    if temp.empty:
        print("No rows for:", label)
        return

    if ascending is None:
        ascending = [False] * len(sort_cols)

    selected = temp.sort_values(sort_cols, ascending=ascending).head(n).copy()
    selected["shortlist_label"] = label

    shortlist_blocks.append(selected)

# 1. Robust primary old/simple reference.
add_best(
    label="primary_old_simple_robust",
    query=(
        "rule_name == 'old_simple_confirm_reference' "
        "and denominator_model == 'none_reference' "
        "and trades >= 100 "
        "and positive_year_rate >= 1.0"
    ),
    sort_cols=["total_pnl", "pnl_to_drawdown"],
)

# 2. Simple-only robust reference.
add_best(
    label="simple_only_robust",
    query=(
        "rule_name == 'simple_only_reference' "
        "and denominator_model == 'none_reference' "
        "and trades >= 100 "
        "and positive_year_rate >= 1.0"
    ),
    sort_cols=["total_pnl", "pnl_to_drawdown"],
)

# 3. Broad pooled HAR-IV quality candidate.
add_best(
    label="pooled_har_iv_quality",
    query=(
        "rule_name == 'simple_denom_confirm' "
        "and denominator_model == 'har_rv_iv' "
        "and trades >= 100 "
        "and max_drawdown >= -600000 "
        "and positive_year_rate >= 0.85"
    ),
    sort_cols=["total_pnl", "pnl_to_drawdown"],
)

# 4. Strict back-tenor exceptional candidate.
add_best(
    label="hybrid_back_exceptional",
    query=(
        "rule_name == 'simple_denom_confirm' "
        "and denominator_model == 'hybrid_back_har_only' "
        "and trades >= 50 and trades <= 250 "
        "and back_pct >= 0.85 "
        "and max_drawdown >= -150000 "
        "and positive_year_rate >= 1.0"
    ),
    sort_cols=["avg_pnl", "total_pnl"],
)

# 5. Broader hybrid-back candidate with old/simple confirmation.
add_best(
    label="hybrid_back_old_simple_broad",
    query=(
        "rule_name == 'old_simple_denom_confirm' "
        "and denominator_model == 'hybrid_back_har_only' "
        "and trades >= 100 "
        "and max_drawdown >= -1050000 "
        "and min_year_pnl >= -125000"
    ),
    sort_cols=["total_pnl", "pnl_to_drawdown"],
)

# 6. Include best per-tenor HAR-IV for comparison, even if not champion.
add_best(
    label="per_tenor_har_iv_best_comparison",
    query=(
        "rule_name == 'simple_denom_confirm' "
        "and denominator_model == 'har_rv_iv_per_tenor' "
        "and trades >= 100 "
        "and positive_year_rate >= 0.85"
    ),
    sort_cols=["total_pnl", "pnl_to_drawdown"],
)

shortlist_rules_df = pd.concat(shortlist_blocks, ignore_index=True)

shortlist_rules_df = shortlist_rules_df.sort_values(
    ["shortlist_label", "total_pnl"],
    ascending=[True, False],
).reset_index(drop=True)

shortlist_rules_df.to_csv(SHORTLIST_RULES_CSV, index=False)

shortlist_rule_ids = shortlist_rules_df["rule_id"].astype(int).tolist()

shortlist_trades_df = trades_df[
    trades_df["rule_id"].astype(int).isin(shortlist_rule_ids)
].copy()

shortlist_trades_df = shortlist_trades_df.merge(
    shortlist_rules_df[
        [
            "rule_id",
            "shortlist_label",
            "rule_name",
            "denominator_model",
            "rsi_cap",
            "old_signal_threshold",
            "old_z_threshold",
            "simple_signal_threshold",
            "simple_z_threshold",
            "denom_signal_threshold",
            "denom_z_threshold",
        ]
    ],
    on=["rule_id", "rule_name", "denominator_model"],
    how="left",
    suffixes=("", "_rule"),
)

shortlist_trades_df.to_parquet(SHORTLIST_TRADES_PARQUET, index=False)

by_year_df = (
    shortlist_trades_df
    .groupby(["shortlist_label", "rule_id", "trade_year"], dropna=False)
    .agg(
        trades=("trade_dt", "count"),
        total_pnl=("test_pnl", "sum"),
        avg_pnl=("test_pnl", "mean"),
        win_rate=("test_pnl", lambda x: (x > 0).mean()),
        worst_pnl=("test_pnl", "min"),
        avg_tenor=("entry_tenor", "mean"),
    )
    .reset_index()
)

by_tenor_df = (
    shortlist_trades_df
    .groupby(["shortlist_label", "rule_id", "entry_tenor"], dropna=False)
    .agg(
        trades=("trade_dt", "count"),
        total_pnl=("test_pnl", "sum"),
        avg_pnl=("test_pnl", "mean"),
        win_rate=("test_pnl", lambda x: (x > 0).mean()),
        worst_pnl=("test_pnl", "min"),
        avg_rsi=("entry_rsi_14", "mean"),
    )
    .reset_index()
)

by_year_df.to_csv(SHORTLIST_BY_YEAR_CSV, index=False)
by_tenor_df.to_csv(SHORTLIST_BY_TENOR_CSV, index=False)

print("Saved shortlist files:")
print(SHORTLIST_RULES_CSV)
print(SHORTLIST_TRADES_PARQUET)
print(SHORTLIST_BY_YEAR_CSV)
print(SHORTLIST_BY_TENOR_CSV)

print("\nShortlisted rules:")
display(
    shortlist_rules_df[
        [
            "shortlist_label",
            "rule_name",
            "rule_id",
            "denominator_model",
            "rsi_cap",
            "old_signal_threshold",
            "old_z_threshold",
            "simple_signal_threshold",
            "simple_z_threshold",
            "denom_signal_threshold",
            "denom_z_threshold",
            "trades",
            "total_pnl",
            "avg_pnl",
            "win_rate",
            "q05_pnl",
            "worst_pnl",
            "max_drawdown",
            "pnl_to_drawdown",
            "positive_year_rate",
            "min_year_pnl",
            "avg_tenor",
            "front_pct",
            "middle_pct",
            "back_pct",
        ]
    ]
)

print("\nShortlist by year:")
display(by_year_df)

print("\nShortlist by tenor:")
display(by_tenor_df)

# Pairwise overlap by trade date + tenor.
overlap_base = shortlist_trades_df[
    [
        "shortlist_label",
        "rule_id",
        "trade_dt",
        "entry_tenor",
        "test_pnl",
    ]
].copy()

labels = sorted(overlap_base["shortlist_label"].dropna().unique())

overlap_rows = []

for a in labels:
    a_df = overlap_base[overlap_base["shortlist_label"] == a][
        ["trade_dt", "entry_tenor"]
    ].drop_duplicates()

    for b in labels:
        b_df = overlap_base[overlap_base["shortlist_label"] == b][
            ["trade_dt", "entry_tenor"]
        ].drop_duplicates()

        merged = a_df.merge(
            b_df,
            on=["trade_dt", "entry_tenor"],
            how="inner",
        )

        overlap_rows.append({
            "label_a": a,
            "label_b": b,
            "trades_a": len(a_df),
            "trades_b": len(b_df),
            "overlap_trades": len(merged),
            "overlap_pct_of_a": len(merged) / len(a_df) if len(a_df) else np.nan,
            "overlap_pct_of_b": len(merged) / len(b_df) if len(b_df) else np.nan,
        })

overlap_df = pd.DataFrame(overlap_rows)

print("\nPairwise overlap:")
display(overlap_df)

Saved shortlist files:
C:\Users\patri\vrp_project\data\audit\put_denominator_candidate_shortlist_rules_v0_2.csv
C:\Users\patri\vrp_project\data\processed\put_denominator_candidate_shortlist_trades_v0_2.parquet
C:\Users\patri\vrp_project\data\audit\put_denominator_candidate_shortlist_by_year_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_denominator_candidate_shortlist_by_tenor_v0_2.csv

Shortlisted rules:


,shortlist_label,rule_name,rule_id,denominator_model,rsi_cap,old_signal_threshold,old_z_threshold,simple_signal_threshold,simple_z_threshold,denom_signal_threshold,...,q05_pnl,worst_pnl,max_drawdown,pnl_to_drawdown,positive_year_rate,min_year_pnl,avg_tenor,front_pct,middle_pct,back_pct
0,hybrid_back_exceptional,simple_denom_confirm,37298,hybrid_back_har_only,58,NaN,NaN,1.10,0.0,0.6,...,-12555.074014,-57319.352129,-1.024663e+05,12.203578,1.000000,37944.127056,27.293478,0.054348,0.032609,0.913043
1,hybrid_back_old_simple_broad,old_simple_denom_confirm,46211,hybrid_back_har_only,77,0.5,-0.5,0.50,-0.5,0.0,...,-29887.659858,-111511.425505,-9.456972e+05,4.506233,0.857143,-50862.410585,23.311813,0.306319,0.119505,0.574176
2,per_tenor_har_iv_best_comparison,simple_denom_confirm,17890,har_rv_iv_per_tenor,77,NaN,NaN,0.40,-0.5,0.0,...,-33802.630384,-129837.937694,-1.273988e+06,2.282077,0.857143,-275799.411372,18.412811,0.549229,0.153025,0.297746
3,pooled_har_iv_quality,simple_denom_confirm,5435,har_rv_iv,68,NaN,NaN,0.75,-0.5,0.0,...,-28477.253323,-100779.661310,-4.847647e+05,4.608242,0.857143,-30535.408260,18.792325,0.478555,0.239278,0.282167
4,primary_old_simple_robust,old_simple_confirm_reference,1729,none_reference,72,0.5,-0.5,0.40,0.0,NaN,...,-27785.830930,-111511.425505,-9.447469e+05,3.751762,1.000000,102416.500948,21.019934,0.392027,0.224252,0.383721
5,simple_only_robust,simple_only_reference,2989,none_reference,72,NaN,NaN,0.40,0.0,NaN,...,-26236.160838,-78574.577043,-5.257750e+05,5.305411,1.000000,52222.742204,15.651982,0.679883,0.145374,0.174743



Shortlist by year:


,shortlist_label,rule_id,trade_year,trades,total_pnl,avg_pnl,win_rate,worst_pnl,avg_tenor
0,hybrid_back_exceptional,37298,2021,35,5.860103e+05,16743.152604,0.942857,-1714.133546,26.228571
1,hybrid_back_exceptional,37298,2022,4,9.531293e+04,23828.231324,1.000000,11172.006267,27.000000
2,hybrid_back_exceptional,37298,2023,5,3.794413e+04,7588.825411,0.800000,-624.494308,29.400000
3,hybrid_back_exceptional,37298,2024,13,1.660923e+05,12776.330618,0.846154,-18753.860127,26.538462
4,hybrid_back_exceptional,37298,2025,6,1.118594e+05,18643.234247,1.000000,15446.997490,28.000000
5,hybrid_back_exceptional,37298,2026,29,2.532358e+05,8732.269748,0.689655,-57319.352129,28.448276
6,hybrid_back_old_simple_broad,46211,2020,23,2.985094e+05,12978.668464,0.956522,-1389.519670,12.652174
7,hybrid_back_old_simple_broad,46211,2021,164,1.759963e+06,10731.482905,0.871951,-26652.128198,24.402439
8,hybrid_back_old_simple_broad,46211,2022,50,6.774519e+04,1354.903840,0.580000,-60499.029000,21.720000
9,hybrid_back_old_simple_broad,46211,2023,144,9.339178e+05,6485.540029,0.798611,-28975.444462,21.729167



Shortlist by tenor:


,shortlist_label,rule_id,entry_tenor,trades,total_pnl,avg_pnl,win_rate,worst_pnl,avg_rsi
0,hybrid_back_exceptional,37298,9,1,1.319812e+04,13198.116054,1.000000,13198.116054,43.503702
1,hybrid_back_exceptional,37298,12,1,1.995122e+04,19951.218748,1.000000,19951.218748,45.763127
2,hybrid_back_exceptional,37298,15,3,9.363989e+03,3121.329567,0.666667,-10549.349570,56.241858
3,hybrid_back_exceptional,37298,18,3,7.307890e+03,2435.963483,0.666667,-18753.860127,46.357677
4,hybrid_back_exceptional,37298,27,61,8.802719e+05,14430.686136,0.852459,-57319.352129,43.190212
5,hybrid_back_exceptional,37298,30,5,9.406937e+04,18813.874030,1.000000,13532.736494,44.407787
6,hybrid_back_exceptional,37298,33,18,2.262925e+05,12571.804516,0.833333,-12448.425789,47.483361
7,hybrid_back_old_simple_broad,46211,9,73,9.025783e+04,1236.408687,0.780822,-47197.552837,61.335879
8,hybrid_back_old_simple_broad,46211,12,101,3.889244e+05,3850.736881,0.811881,-60499.029000,61.829599
9,hybrid_back_old_simple_broad,46211,15,49,1.145219e+05,2337.182555,0.653061,-37139.021346,58.852271



Pairwise overlap:


,label_a,label_b,trades_a,trades_b,overlap_trades,overlap_pct_of_a,overlap_pct_of_b
0,hybrid_back_exceptional,hybrid_back_exceptional,92,92,92,1.000000,1.000000
1,hybrid_back_exceptional,hybrid_back_old_simple_broad,92,728,53,0.576087,0.072802
2,hybrid_back_exceptional,per_tenor_har_iv_best_comparison,92,843,14,0.152174,0.016607
3,hybrid_back_exceptional,pooled_har_iv_quality,92,443,10,0.108696,0.022573
4,hybrid_back_exceptional,primary_old_simple_robust,92,602,20,0.217391,0.033223
5,hybrid_back_exceptional,simple_only_robust,92,681,6,0.065217,0.008811
6,hybrid_back_old_simple_broad,hybrid_back_exceptional,728,92,53,0.072802,0.576087
7,hybrid_back_old_simple_broad,hybrid_back_old_simple_broad,728,728,728,1.000000,1.000000
8,hybrid_back_old_simple_broad,per_tenor_har_iv_best_comparison,728,843,285,0.391484,0.338078
9,hybrid_back_old_simple_broad,pooled_har_iv_quality,728,443,107,0.146978,0.241535


In [12]:
# ============================================================
# Cell 31: Combine shortlisted sleeves with one-put-per-day priority
# ============================================================

import numpy as np
import pandas as pd

SHORTLIST_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_denominator_candidate_shortlist_trades_v0_2.parquet"
)

COMBO_SUMMARY_CSV = (
    AUDIT_DIR / "put_shortlist_sleeve_combo_summary_v0_2.csv"
)

COMBO_BY_YEAR_CSV = (
    AUDIT_DIR / "put_shortlist_sleeve_combo_by_year_v0_2.csv"
)

COMBO_BY_TENOR_CSV = (
    AUDIT_DIR / "put_shortlist_sleeve_combo_by_tenor_v0_2.csv"
)

COMBO_SELECTED_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_shortlist_sleeve_combo_selected_trades_v0_2.parquet"
)

trades_df = pd.read_parquet(SHORTLIST_TRADES_PARQUET).copy()

trades_df["trade_dt"] = pd.to_datetime(trades_df["trade_dt"])
trades_df["trade_year"] = trades_df["trade_dt"].dt.year
trades_df["entry_tenor"] = trades_df["entry_tenor"].astype(int)

def max_drawdown_from_pnl(pnl_series):
    pnl = pd.Series(pnl_series).fillna(0.0)
    equity = pnl.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    return float(dd.min())

def summarize_strategy(df, strategy_name):
    df = df.sort_values("trade_dt").copy()

    if df.empty:
        return {
            "strategy_name": strategy_name,
            "trades": 0,
            "total_pnl": 0.0,
            "avg_pnl": np.nan,
            "median_pnl": np.nan,
            "win_rate": np.nan,
            "q05_pnl": np.nan,
            "worst_pnl": np.nan,
            "max_drawdown": np.nan,
            "pnl_to_drawdown": np.nan,
            "worst_10_trade_sum": np.nan,
            "worst_20_trade_sum": np.nan,
            "positive_year_rate": np.nan,
            "min_year_pnl": np.nan,
            "avg_tenor": np.nan,
            "front_pct": np.nan,
            "middle_pct": np.nan,
            "back_pct": np.nan,
        }

    pnl = df["test_pnl"].astype(float)
    yearly = df.groupby("trade_year")["test_pnl"].sum()

    roll10 = pnl.rolling(10).sum()
    roll20 = pnl.rolling(20).sum()

    max_dd = max_drawdown_from_pnl(pnl)

    return {
        "strategy_name": strategy_name,
        "trades": len(df),
        "start_date": df["trade_dt"].min(),
        "end_date": df["trade_dt"].max(),
        "total_pnl": pnl.sum(),
        "avg_pnl": pnl.mean(),
        "median_pnl": pnl.median(),
        "win_rate": (pnl > 0).mean(),
        "q05_pnl": pnl.quantile(0.05),
        "worst_pnl": pnl.min(),
        "max_drawdown": max_dd,
        "pnl_to_drawdown": pnl.sum() / abs(max_dd) if max_dd != 0 else np.nan,
        "worst_10_trade_sum": roll10.min(),
        "worst_20_trade_sum": roll20.min(),
        "positive_year_rate": (yearly > 0).mean(),
        "min_year_pnl": yearly.min(),
        "avg_tenor": df["entry_tenor"].mean(),
        "front_pct": df["tenor_group"].eq("front_9_15d").mean(),
        "middle_pct": df["tenor_group"].eq("middle_18_24d").mean(),
        "back_pct": df["tenor_group"].eq("back_27_33d").mean(),
    }

def select_one_per_day_by_priority(df, priority_map):
    temp = df.copy()

    temp["priority"] = temp["shortlist_label"].map(priority_map)

    temp = temp.dropna(subset=["priority"]).copy()
    temp["priority"] = temp["priority"].astype(int)

    # No look-ahead tie-breakers.
    # Priority first, then prefer longer tenor, then stable label order.
    selected = (
        temp
        .sort_values(
            ["trade_dt", "priority", "entry_tenor", "shortlist_label"],
            ascending=[True, True, False, True],
        )
        .groupby("trade_dt", as_index=False)
        .head(1)
        .copy()
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )

    return selected

combo_specs = {
    "primary_only": {
        "primary_old_simple_robust": 1,
    },
    "simple_only": {
        "simple_only_robust": 1,
    },
    "hybrid_back_exceptional_only": {
        "hybrid_back_exceptional": 1,
    },
    "primary_plus_hybrid_priority_primary": {
        "primary_old_simple_robust": 1,
        "hybrid_back_exceptional": 2,
    },
    "primary_plus_hybrid_priority_hybrid": {
        "hybrid_back_exceptional": 1,
        "primary_old_simple_robust": 2,
    },
    "primary_plus_simple_priority_primary": {
        "primary_old_simple_robust": 1,
        "simple_only_robust": 2,
    },
    "primary_plus_simple_priority_simple": {
        "simple_only_robust": 1,
        "primary_old_simple_robust": 2,
    },
    "primary_plus_simple_plus_hybrid_priority_hybrid_primary_simple": {
        "hybrid_back_exceptional": 1,
        "primary_old_simple_robust": 2,
        "simple_only_robust": 3,
    },
    "primary_plus_simple_plus_hybrid_priority_primary_hybrid_simple": {
        "primary_old_simple_robust": 1,
        "hybrid_back_exceptional": 2,
        "simple_only_robust": 3,
    },
}

summary_rows = []
selected_blocks = []

for strategy_name, priority_map in combo_specs.items():
    selected = select_one_per_day_by_priority(trades_df, priority_map)
    selected["combo_strategy"] = strategy_name

    summary_rows.append(summarize_strategy(selected, strategy_name))
    selected_blocks.append(selected)

combo_summary_df = pd.DataFrame(summary_rows)

combo_selected_df = pd.concat(selected_blocks, ignore_index=True)

combo_by_year_df = (
    combo_selected_df
    .groupby(["combo_strategy", "trade_year"], dropna=False)
    .agg(
        trades=("trade_dt", "count"),
        total_pnl=("test_pnl", "sum"),
        avg_pnl=("test_pnl", "mean"),
        win_rate=("test_pnl", lambda x: (x > 0).mean()),
        worst_pnl=("test_pnl", "min"),
        avg_tenor=("entry_tenor", "mean"),
        front_pct=("tenor_group", lambda x: (x == "front_9_15d").mean()),
        middle_pct=("tenor_group", lambda x: (x == "middle_18_24d").mean()),
        back_pct=("tenor_group", lambda x: (x == "back_27_33d").mean()),
    )
    .reset_index()
)

combo_by_tenor_df = (
    combo_selected_df
    .groupby(["combo_strategy", "entry_tenor"], dropna=False)
    .agg(
        trades=("trade_dt", "count"),
        total_pnl=("test_pnl", "sum"),
        avg_pnl=("test_pnl", "mean"),
        win_rate=("test_pnl", lambda x: (x > 0).mean()),
        worst_pnl=("test_pnl", "min"),
        avg_rsi=("entry_rsi_14", "mean"),
    )
    .reset_index()
)

combo_summary_df = combo_summary_df.sort_values(
    ["pnl_to_drawdown", "total_pnl"],
    ascending=[False, False],
).reset_index(drop=True)

combo_summary_df.to_csv(COMBO_SUMMARY_CSV, index=False)
combo_by_year_df.to_csv(COMBO_BY_YEAR_CSV, index=False)
combo_by_tenor_df.to_csv(COMBO_BY_TENOR_CSV, index=False)
combo_selected_df.to_parquet(COMBO_SELECTED_TRADES_PARQUET, index=False)

print("Saved combo files:")
print(COMBO_SUMMARY_CSV)
print(COMBO_BY_YEAR_CSV)
print(COMBO_BY_TENOR_CSV)
print(COMBO_SELECTED_TRADES_PARQUET)

print("\nCombo summary:")
display(combo_summary_df)

print("\nCombo by year:")
display(combo_by_year_df)

print("\nCombo by tenor:")
display(combo_by_tenor_df)

# Incremental diagnostics versus primary.
primary_dates = set(
    combo_selected_df
    .query("combo_strategy == 'primary_only'")["trade_dt"]
)

hybrid_dates = set(
    combo_selected_df
    .query("combo_strategy == 'hybrid_back_exceptional_only'")["trade_dt"]
)

print("\nDate overlap diagnostics:")
print("Primary trade dates:", len(primary_dates))
print("Hybrid exceptional trade dates:", len(hybrid_dates))
print("Overlap dates:", len(primary_dates & hybrid_dates))
print("Hybrid dates not in primary:", len(hybrid_dates - primary_dates))
print("Primary dates not in hybrid:", len(primary_dates - hybrid_dates))

Saved combo files:
C:\Users\patri\vrp_project\data\audit\put_shortlist_sleeve_combo_summary_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_shortlist_sleeve_combo_by_year_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_shortlist_sleeve_combo_by_tenor_v0_2.csv
C:\Users\patri\vrp_project\data\processed\put_shortlist_sleeve_combo_selected_trades_v0_2.parquet

Combo summary:


,strategy_name,trades,start_date,end_date,total_pnl,avg_pnl,median_pnl,win_rate,q05_pnl,worst_pnl,max_drawdown,pnl_to_drawdown,worst_10_trade_sum,worst_20_trade_sum,positive_year_rate,min_year_pnl,avg_tenor,front_pct,middle_pct,back_pct
0,hybrid_back_exceptional_only,92,2021-01-27,2026-03-30,1.250455e+06,13591.901301,17069.001692,0.847826,-12555.074014,-57319.352129,-102466.257845,12.203578,-102466.257845,21075.310568,1.0,37944.127056,27.293478,0.054348,0.032609,0.913043
1,simple_only,681,2020-10-29,2026-06-11,2.789453e+06,4096.112666,8721.260162,0.751836,-26236.160838,-78574.577043,-525775.028019,5.305411,-339564.638176,-430542.302861,1.0,52222.742204,15.651982,0.679883,0.145374,0.174743
2,primary_plus_simple_priority_simple,681,2020-10-29,2026-06-11,2.789453e+06,4096.112666,8721.260162,0.751836,-26236.160838,-78574.577043,-525775.028019,5.305411,-339564.638176,-430542.302861,1.0,52222.742204,15.651982,0.679883,0.145374,0.174743
3,primary_plus_hybrid_priority_hybrid,602,2020-10-29,2026-06-05,3.733363e+06,6201.599183,10798.464284,0.780731,-26231.923947,-111511.425505,-944746.911842,3.951707,-533576.106123,-941213.622876,1.0,110810.890918,21.772425,0.360465,0.184385,0.455150
4,primary_only,602,2020-10-29,2026-06-05,3.544466e+06,5887.817014,10442.162535,0.779070,-27785.830930,-111511.425505,-944746.911842,3.751762,-533576.106123,-941213.622876,1.0,102416.500948,21.019934,0.392027,0.224252,0.383721
5,primary_plus_hybrid_priority_primary,602,2020-10-29,2026-06-05,3.544466e+06,5887.817014,10442.162535,0.779070,-27785.830930,-111511.425505,-944746.911842,3.751762,-533576.106123,-941213.622876,1.0,102416.500948,21.019934,0.392027,0.224252,0.383721
6,primary_plus_simple_plus_hybrid_priority_hybri...,681,2020-10-29,2026-06-11,3.706968e+06,5443.417940,10324.881201,0.765051,-27822.697682,-111511.425505,-994297.470817,3.728228,-528416.858288,-918717.522718,1.0,83213.698112,20.638767,0.419971,0.170338,0.409692
7,primary_plus_simple_priority_primary,681,2020-10-29,2026-06-11,3.518071e+06,5166.036346,10046.035299,0.763583,-29285.502598,-111511.425505,-994297.470817,3.538248,-528416.858288,-918717.522718,1.0,74819.308142,19.973568,0.447871,0.205580,0.346549
8,primary_plus_simple_plus_hybrid_priority_prima...,681,2020-10-29,2026-06-11,3.518071e+06,5166.036346,10046.035299,0.763583,-29285.502598,-111511.425505,-994297.470817,3.538248,-528416.858288,-918717.522718,1.0,74819.308142,19.973568,0.447871,0.205580,0.346549



Combo by year:


,combo_strategy,trade_year,trades,total_pnl,avg_pnl,win_rate,worst_pnl,avg_tenor,front_pct,middle_pct,back_pct
0,hybrid_back_exceptional_only,2021,35,586010.341151,16743.152604,0.942857,-1714.133546,26.228571,0.114286,0.028571,0.857143
1,hybrid_back_exceptional_only,2022,4,95312.925298,23828.231324,1.000000,11172.006267,27.000000,0.000000,0.000000,1.000000
2,hybrid_back_exceptional_only,2023,5,37944.127056,7588.825411,0.800000,-624.494308,29.400000,0.000000,0.200000,0.800000
3,hybrid_back_exceptional_only,2024,13,166092.298039,12776.330618,0.846154,-18753.860127,26.538462,0.076923,0.076923,0.846154
4,hybrid_back_exceptional_only,2025,6,111859.405480,18643.234247,1.000000,15446.997490,28.000000,0.000000,0.000000,1.000000
...,...,...,...,...,...,...,...,...,...,...,...
57,simple_only,2022,59,173483.995562,2940.406704,0.644068,-51422.095318,10.372881,0.983051,0.016949,0.000000
58,simple_only,2023,158,533976.653998,3379.599076,0.740506,-49261.410409,15.056962,0.708861,0.151899,0.139241
59,simple_only,2024,122,370003.751946,3032.817639,0.745902,-39035.223257,15.516393,0.647541,0.188525,0.163934
60,simple_only,2025,117,347157.239398,2967.155892,0.786325,-78574.577043,16.307692,0.649573,0.145299,0.205128



Combo by tenor:


,combo_strategy,entry_tenor,trades,total_pnl,avg_pnl,win_rate,worst_pnl,avg_rsi
0,hybrid_back_exceptional_only,9,1,13198.116054,13198.116054,1.000000,13198.116054,43.503702
1,hybrid_back_exceptional_only,12,1,19951.218748,19951.218748,1.000000,19951.218748,45.763127
2,hybrid_back_exceptional_only,15,3,9363.988702,3121.329567,0.666667,-10549.349570,56.241858
3,hybrid_back_exceptional_only,18,3,7307.890450,2435.963483,0.666667,-18753.860127,46.357677
4,hybrid_back_exceptional_only,27,61,880271.854323,14430.686136,0.852459,-57319.352129,43.190212
...,...,...,...,...,...,...,...,...
74,simple_only,21,2,-16626.002782,-8313.001391,0.500000,-37832.763639,47.510359
75,simple_only,24,5,61748.374029,12349.674806,1.000000,10676.422786,63.671637
76,simple_only,27,19,99050.453495,5213.181763,0.736842,-38264.225718,61.992591
77,simple_only,30,14,155696.267622,11121.161973,0.928571,-27071.544809,60.107155



Date overlap diagnostics:
Primary trade dates: 602
Hybrid exceptional trade dates: 92
Overlap dates: 92
Hybrid dates not in primary: 0
Primary dates not in hybrid: 510


In [18]:
# ============================================================
# Cell 32: Attribution of hybrid priority override - corrected
# ============================================================

import numpy as np
import pandas as pd

COMBO_SELECTED_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_shortlist_sleeve_combo_selected_trades_v0_2.parquet"
)

OVERRIDE_ATTRIBUTION_CSV = (
    AUDIT_DIR / "put_hybrid_override_attribution_v0_2.csv"
)

OVERRIDE_BY_YEAR_CSV = (
    AUDIT_DIR / "put_hybrid_override_attribution_by_year_v0_2.csv"
)

OVERRIDE_BY_TENOR_PAIR_CSV = (
    AUDIT_DIR / "put_hybrid_override_attribution_by_tenor_pair_v0_2.csv"
)

df = pd.read_parquet(COMBO_SELECTED_TRADES_PARQUET).copy()

df["trade_dt"] = pd.to_datetime(df["trade_dt"])
df["trade_year"] = df["trade_dt"].dt.year
df["entry_tenor"] = df["entry_tenor"].astype(int)

primary = (
    df.query("combo_strategy == 'primary_only'")
    .copy()
    .sort_values("trade_dt")
)

hybrid = (
    df.query("combo_strategy == 'hybrid_back_exceptional_only'")
    .copy()
    .sort_values("trade_dt")
)

override = primary.merge(
    hybrid,
    on="trade_dt",
    how="inner",
    suffixes=("_primary", "_hybrid"),
)

# Recreate clean unsuffixed year column after merge.
override["trade_year"] = override["trade_dt"].dt.year

override["incremental_pnl"] = (
    override["test_pnl_hybrid"] - override["test_pnl_primary"]
)

override["tenor_shift"] = (
    override["entry_tenor_hybrid"] - override["entry_tenor_primary"]
)

override["same_tenor"] = (
    override["entry_tenor_hybrid"] == override["entry_tenor_primary"]
)

override["primary_group"] = override["tenor_group_primary"]
override["hybrid_group"] = override["tenor_group_hybrid"]

override.to_csv(OVERRIDE_ATTRIBUTION_CSV, index=False)

by_year = (
    override
    .groupby("trade_year")
    .agg(
        overrides=("trade_dt", "count"),
        primary_pnl=("test_pnl_primary", "sum"),
        hybrid_pnl=("test_pnl_hybrid", "sum"),
        incremental_pnl=("incremental_pnl", "sum"),
        avg_incremental_pnl=("incremental_pnl", "mean"),
        win_rate_primary=("test_pnl_primary", lambda x: (x > 0).mean()),
        win_rate_hybrid=("test_pnl_hybrid", lambda x: (x > 0).mean()),
        primary_worst=("test_pnl_primary", "min"),
        hybrid_worst=("test_pnl_hybrid", "min"),
        avg_primary_tenor=("entry_tenor_primary", "mean"),
        avg_hybrid_tenor=("entry_tenor_hybrid", "mean"),
    )
    .reset_index()
)

by_tenor_pair = (
    override
    .groupby(["entry_tenor_primary", "entry_tenor_hybrid"])
    .agg(
        overrides=("trade_dt", "count"),
        primary_pnl=("test_pnl_primary", "sum"),
        hybrid_pnl=("test_pnl_hybrid", "sum"),
        incremental_pnl=("incremental_pnl", "sum"),
        avg_incremental_pnl=("incremental_pnl", "mean"),
        primary_win_rate=("test_pnl_primary", lambda x: (x > 0).mean()),
        hybrid_win_rate=("test_pnl_hybrid", lambda x: (x > 0).mean()),
        primary_worst=("test_pnl_primary", "min"),
        hybrid_worst=("test_pnl_hybrid", "min"),
    )
    .reset_index()
    .sort_values("incremental_pnl", ascending=False)
)

by_group_pair = (
    override
    .groupby(["primary_group", "hybrid_group"])
    .agg(
        overrides=("trade_dt", "count"),
        primary_pnl=("test_pnl_primary", "sum"),
        hybrid_pnl=("test_pnl_hybrid", "sum"),
        incremental_pnl=("incremental_pnl", "sum"),
        avg_incremental_pnl=("incremental_pnl", "mean"),
    )
    .reset_index()
    .sort_values("incremental_pnl", ascending=False)
)

by_year.to_csv(OVERRIDE_BY_YEAR_CSV, index=False)
by_tenor_pair.to_csv(OVERRIDE_BY_TENOR_PAIR_CSV, index=False)

print("Saved override attribution:")
print(OVERRIDE_ATTRIBUTION_CSV)
print(OVERRIDE_BY_YEAR_CSV)
print(OVERRIDE_BY_TENOR_PAIR_CSV)

print("\nOverride summary:")
display(
    pd.DataFrame([
        {
            "override_dates": len(override),
            "primary_pnl_on_override_dates": override["test_pnl_primary"].sum(),
            "hybrid_pnl_on_override_dates": override["test_pnl_hybrid"].sum(),
            "incremental_pnl": override["incremental_pnl"].sum(),
            "avg_incremental_pnl": override["incremental_pnl"].mean(),
            "median_incremental_pnl": override["incremental_pnl"].median(),
            "pct_overrides_helped": (override["incremental_pnl"] > 0).mean(),
            "worst_incremental_pnl": override["incremental_pnl"].min(),
            "best_incremental_pnl": override["incremental_pnl"].max(),
            "same_tenor_pct": override["same_tenor"].mean(),
            "avg_primary_tenor": override["entry_tenor_primary"].mean(),
            "avg_hybrid_tenor": override["entry_tenor_hybrid"].mean(),
        }
    ])
)

print("\nOverride attribution by year:")
display(by_year)

print("\nOverride attribution by tenor pair:")
display(by_tenor_pair)

print("\nOverride attribution by tenor group pair:")
display(by_group_pair)

print("\nWorst 20 substitutions:")
display(
    override[
        [
            "trade_dt",
            "entry_tenor_primary",
            "entry_tenor_hybrid",
            "tenor_group_primary",
            "tenor_group_hybrid",
            "test_pnl_primary",
            "test_pnl_hybrid",
            "incremental_pnl",
            "entry_rsi_14_primary",
            "entry_rsi_14_hybrid",
        ]
    ]
    .sort_values("incremental_pnl")
    .head(20)
)

print("\nBest 20 substitutions:")
display(
    override[
        [
            "trade_dt",
            "entry_tenor_primary",
            "entry_tenor_hybrid",
            "tenor_group_primary",
            "tenor_group_hybrid",
            "test_pnl_primary",
            "test_pnl_hybrid",
            "incremental_pnl",
            "entry_rsi_14_primary",
            "entry_rsi_14_hybrid",
        ]
    ]
    .sort_values("incremental_pnl", ascending=False)
    .head(20)
)

Saved override attribution:
C:\Users\patri\vrp_project\data\audit\put_hybrid_override_attribution_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_hybrid_override_attribution_by_year_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_hybrid_override_attribution_by_tenor_pair_v0_2.csv

Override summary:


,override_dates,primary_pnl_on_override_dates,hybrid_pnl_on_override_dates,incremental_pnl,avg_incremental_pnl,median_incremental_pnl,pct_overrides_helped,worst_incremental_pnl,best_incremental_pnl,same_tenor_pct,avg_primary_tenor,avg_hybrid_tenor
0,92,1.061558e+06,1.250455e+06,188896.865911,2053.226803,0.0,0.413043,-30851.95277,51417.133707,0.217391,22.369565,27.293478



Override attribution by year:


,trade_year,overrides,primary_pnl,hybrid_pnl,incremental_pnl,avg_incremental_pnl,win_rate_primary,win_rate_hybrid,primary_worst,hybrid_worst,avg_primary_tenor,avg_hybrid_tenor
0,2021,35,476999.652654,586010.341151,109010.688496,3114.591100,0.885714,0.942857,-15324.951104,-1714.133546,21.257143,26.228571
1,2022,4,105318.164231,95312.925298,-10005.238933,-2501.309733,1.000000,1.000000,23468.695934,11172.006267,21.750000,27.000000
2,2023,5,35284.475379,37944.127056,2659.651677,531.930335,0.800000,0.800000,-624.494308,-624.494308,30.000000,29.400000
3,2024,13,151002.076659,166092.298039,15090.221380,1160.786260,0.846154,0.846154,-10549.349570,-18753.860127,18.230769,26.538462
4,2025,6,103465.015510,111859.405480,8394.389970,1399.064995,1.000000,1.000000,8257.236463,15446.997490,26.500000,28.000000
5,2026,29,189488.669371,253235.822692,63747.153320,2198.177701,0.724138,0.689655,-39615.671891,-57319.352129,23.482759,28.448276



Override attribution by tenor pair:


...

,primary_group,hybrid_group,overrides,primary_pnl,hybrid_pnl,incremental_pnl,avg_incremental_pnl
4,middle_18_24d,back_27_33d,23,151348.587685,303618.049348,152269.461663,6620.411377
3,front_9_15d,back_27_33d,24,36462.323511,135013.849910,98551.526398,4106.313600
6,middle_18_24d,middle_18_24d,1,12903.257161,12903.257161,0.000000,0.000000
5,middle_18_24d,front_9_15d,3,25853.596347,22599.985232,-3253.611115,-1084.537038
1,back_27_33d,front_9_15d,2,25019.558844,19913.338272,-5106.220572,-2553.110286
0,back_27_33d,back_27_33d,37,778785.542217,762001.806503,-16783.735715,-453.614479
2,back_27_33d,middle_18_24d,2,31185.188038,-5595.366711,-36780.554749,-18390.277374



Worst 20 substitutions:


,trade_dt,entry_tenor_primary,entry_tenor_hybrid,tenor_group_primary,tenor_group_hybrid,test_pnl_primary,test_pnl_hybrid,incremental_pnl,entry_rsi_14_primary,entry_rsi_14_hybrid
44,2024-07-19,30,18,back_27_33d,middle_18_24d,12098.092643,-18753.860127,-30851.952770,49.528842,49.528842
67,2026-01-30,9,33,front_9_15d,back_27_33d,6730.047283,-12448.425789,-19178.473072,53.359087,53.359087
71,2026-03-02,15,27,front_9_15d,back_27_33d,-39615.671891,-57319.352129,-17703.680238,48.647576,48.647576
35,2022-01-24,18,27,middle_18_24d,back_27_33d,23468.695934,11172.006267,-12296.689667,28.552336,28.552336
65,2026-01-16,18,27,middle_18_24d,back_27_33d,9755.029171,-2165.702931,-11920.732103,56.625318,56.625318
64,2026-01-15,18,27,middle_18_24d,back_27_33d,8140.290044,-2452.310975,-10592.601019,57.286866,57.286866
21,2021-09-10,12,27,front_9_15d,back_27_33d,11281.618811,1489.263398,-9792.355414,47.542957,47.542957
63,2026-01-14,18,27,middle_18_24d,back_27_33d,9514.047296,616.464066,-8897.583230,55.345903,55.345903
69,2026-02-04,12,27,front_9_15d,back_27_33d,3816.804984,-4108.840691,-7925.645675,46.423596,46.423596
68,2026-02-03,12,33,front_9_15d,back_27_33d,-2216.019232,-9884.631119,-7668.611887,50.152892,50.152892



Best 20 substitutions:


,trade_dt,entry_tenor_primary,entry_tenor_hybrid,tenor_group_primary,tenor_group_hybrid,test_pnl_primary,test_pnl_hybrid,incremental_pnl,entry_rsi_14_primary,entry_rsi_14_hybrid
73,2026-03-04,18,33,middle_18_24d,back_27_33d,-38419.098915,12998.034791,51417.133707,48.264262,48.264262
75,2026-03-06,18,27,middle_18_24d,back_27_33d,-33182.394118,339.761603,33522.155721,38.453183,38.453183
77,2026-03-10,24,27,middle_18_24d,back_27_33d,-9306.228139,22148.557542,31454.785681,42.825712,42.825712
24,2021-09-16,18,27,middle_18_24d,back_27_33d,-15324.951104,14685.666387,30010.617491,50.908880,50.908880
50,2024-10-25,9,27,front_9_15d,back_27_33d,-5647.266241,15977.631316,21624.897557,56.431722,56.431722
4,2021-02-23,12,27,front_9_15d,back_27_33d,1221.218281,17287.710267,16066.491986,54.833172,54.833172
72,2026-03-03,15,27,front_9_15d,back_27_33d,-29210.915071,-13409.852082,15801.062989,43.035216,43.035216
76,2026-03-09,12,27,front_9_15d,back_27_33d,-28107.751777,-12685.421844,15422.329933,43.879944,43.879944
74,2026-03-05,18,27,middle_18_24d,back_27_33d,-32473.930236,-17217.829479,15256.100757,45.000430,45.000430
20,2021-09-09,12,33,front_9_15d,back_27_33d,1865.007300,12767.955703,10902.948403,55.086042,55.086042


In [19]:
# ============================================================
# Cell 33: Refined conditional hybrid override
# Only override primary when hybrid moves front/middle -> back
# ============================================================

import numpy as np
import pandas as pd

COMBO_SELECTED_TRADES_PARQUET = (
    PROCESSED_DATA_DIR / "put_shortlist_sleeve_combo_selected_trades_v0_2.parquet"
)

REFINED_OVERRIDE_SELECTED_PARQUET = (
    PROCESSED_DATA_DIR / "put_refined_hybrid_override_selected_trades_v0_2.parquet"
)

REFINED_OVERRIDE_SUMMARY_CSV = (
    AUDIT_DIR / "put_refined_hybrid_override_summary_v0_2.csv"
)

REFINED_OVERRIDE_BY_YEAR_CSV = (
    AUDIT_DIR / "put_refined_hybrid_override_by_year_v0_2.csv"
)

REFINED_OVERRIDE_BY_TENOR_CSV = (
    AUDIT_DIR / "put_refined_hybrid_override_by_tenor_v0_2.csv"
)

REFINED_OVERRIDE_ATTRIBUTION_CSV = (
    AUDIT_DIR / "put_refined_hybrid_override_attribution_v0_2.csv"
)

df = pd.read_parquet(COMBO_SELECTED_TRADES_PARQUET).copy()

df["trade_dt"] = pd.to_datetime(df["trade_dt"])
df["trade_year"] = df["trade_dt"].dt.year
df["entry_tenor"] = df["entry_tenor"].astype(int)

primary = (
    df.query("combo_strategy == 'primary_only'")
    .copy()
    .sort_values("trade_dt")
)

hybrid = (
    df.query("combo_strategy == 'hybrid_back_exceptional_only'")
    .copy()
    .sort_values("trade_dt")
)

# Merge primary with possible hybrid candidate on same date.
candidate = primary.merge(
    hybrid[
        [
            "trade_dt",
            "entry_tenor",
            "tenor_group",
            "test_pnl",
            "shortlist_label",
            "rule_id",
            "rule_name",
            "denominator_model",
            "entry_rsi_14",
        ]
    ],
    on="trade_dt",
    how="left",
    suffixes=("_primary", "_hybrid"),
)

candidate["has_hybrid"] = candidate["entry_tenor_hybrid"].notna()

candidate["primary_is_front_or_middle"] = (
    candidate["entry_tenor_primary"] < 27
)

candidate["hybrid_is_back"] = (
    candidate["entry_tenor_hybrid"] >= 27
)

candidate["use_hybrid_refined"] = (
    candidate["has_hybrid"]
    & candidate["primary_is_front_or_middle"]
    & candidate["hybrid_is_back"]
)

# Build selected trade rows.
primary_cols = primary.columns.tolist()

selected_primary = candidate[~candidate["use_hybrid_refined"]].copy()
selected_hybrid = candidate[candidate["use_hybrid_refined"]].copy()

# Convert back into the original primary-column schema.
selected_primary_out = selected_primary.copy()
rename_primary = {
    c: c.replace("_primary", "")
    for c in selected_primary_out.columns
    if c.endswith("_primary")
}

selected_primary_out = selected_primary_out.rename(columns=rename_primary)

selected_primary_out = selected_primary_out[
    [c for c in primary_cols if c in selected_primary_out.columns]
].copy()

# Hybrid rows need to be built from hybrid columns.
hybrid_rows = []

for _, row in selected_hybrid.iterrows():
    base = {}

    # Start from primary row so we preserve any columns not explicitly available.
    for c in primary_cols:
        if c in row.index:
            base[c] = row[c]
        elif f"{c}_primary" in row.index:
            base[c] = row[f"{c}_primary"]
        else:
            base[c] = np.nan

    # Override key trade identity/performance fields with hybrid values.
    override_map = {
        "entry_tenor": "entry_tenor_hybrid",
        "tenor_group": "tenor_group_hybrid",
        "test_pnl": "test_pnl_hybrid",
        "shortlist_label": "shortlist_label_hybrid",
        "rule_id": "rule_id_hybrid",
        "rule_name": "rule_name_hybrid",
        "denominator_model": "denominator_model_hybrid",
        "entry_rsi_14": "entry_rsi_14_hybrid",
    }

    for target_col, source_col in override_map.items():
        if source_col in row.index:
            base[target_col] = row[source_col]

    hybrid_rows.append(base)

selected_hybrid_out = pd.DataFrame(hybrid_rows)

refined_selected = pd.concat(
    [selected_primary_out, selected_hybrid_out],
    ignore_index=True,
)

refined_selected["trade_dt"] = pd.to_datetime(refined_selected["trade_dt"])
refined_selected["trade_year"] = refined_selected["trade_dt"].dt.year
refined_selected["entry_tenor"] = refined_selected["entry_tenor"].astype(int)
refined_selected["combo_strategy"] = "primary_with_refined_hybrid_override"

refined_selected = refined_selected.sort_values("trade_dt").reset_index(drop=True)

# Attribution only for actual refined overrides.
override_attr = candidate[candidate["use_hybrid_refined"]].copy()
override_attr["incremental_pnl"] = (
    override_attr["test_pnl_hybrid"] - override_attr["test_pnl_primary"]
)
override_attr["tenor_shift"] = (
    override_attr["entry_tenor_hybrid"] - override_attr["entry_tenor_primary"]
)
override_attr["trade_year"] = override_attr["trade_dt"].dt.year

# Helpers.
def max_drawdown_from_pnl(pnl_series):
    pnl = pd.Series(pnl_series).fillna(0.0)
    equity = pnl.cumsum()
    running_max = equity.cummax()
    dd = equity - running_max
    return float(dd.min())

def summarize(df, name):
    df = df.sort_values("trade_dt").copy()
    pnl = df["test_pnl"].astype(float)
    yearly = df.groupby("trade_year")["test_pnl"].sum()

    roll10 = pnl.rolling(10).sum()
    roll20 = pnl.rolling(20).sum()

    max_dd = max_drawdown_from_pnl(pnl)

    return {
        "strategy_name": name,
        "trades": len(df),
        "start_date": df["trade_dt"].min(),
        "end_date": df["trade_dt"].max(),
        "total_pnl": pnl.sum(),
        "avg_pnl": pnl.mean(),
        "median_pnl": pnl.median(),
        "win_rate": (pnl > 0).mean(),
        "q05_pnl": pnl.quantile(0.05),
        "worst_pnl": pnl.min(),
        "max_drawdown": max_dd,
        "pnl_to_drawdown": pnl.sum() / abs(max_dd) if max_dd != 0 else np.nan,
        "worst_10_trade_sum": roll10.min(),
        "worst_20_trade_sum": roll20.min(),
        "positive_year_rate": (yearly > 0).mean(),
        "min_year_pnl": yearly.min(),
        "avg_tenor": df["entry_tenor"].mean(),
        "front_pct": df["tenor_group"].eq("front_9_15d").mean(),
        "middle_pct": df["tenor_group"].eq("middle_18_24d").mean(),
        "back_pct": df["tenor_group"].eq("back_27_33d").mean(),
    }

primary_summary = summarize(primary, "primary_only")
broad_hybrid_summary = summarize(
    df.query("combo_strategy == 'primary_plus_hybrid_priority_hybrid'").copy(),
    "primary_plus_hybrid_priority_hybrid",
)
refined_summary = summarize(
    refined_selected,
    "primary_with_refined_hybrid_override",
)

summary_df = pd.DataFrame(
    [primary_summary, broad_hybrid_summary, refined_summary]
)

by_year_df = (
    refined_selected
    .groupby("trade_year", dropna=False)
    .agg(
        trades=("trade_dt", "count"),
        total_pnl=("test_pnl", "sum"),
        avg_pnl=("test_pnl", "mean"),
        win_rate=("test_pnl", lambda x: (x > 0).mean()),
        worst_pnl=("test_pnl", "min"),
        avg_tenor=("entry_tenor", "mean"),
        front_pct=("tenor_group", lambda x: (x == "front_9_15d").mean()),
        middle_pct=("tenor_group", lambda x: (x == "middle_18_24d").mean()),
        back_pct=("tenor_group", lambda x: (x == "back_27_33d").mean()),
    )
    .reset_index()
)

by_tenor_df = (
    refined_selected
    .groupby("entry_tenor", dropna=False)
    .agg(
        trades=("trade_dt", "count"),
        total_pnl=("test_pnl", "sum"),
        avg_pnl=("test_pnl", "mean"),
        win_rate=("test_pnl", lambda x: (x > 0).mean()),
        worst_pnl=("test_pnl", "min"),
        avg_rsi=("entry_rsi_14", "mean"),
    )
    .reset_index()
)

refined_selected.to_parquet(REFINED_OVERRIDE_SELECTED_PARQUET, index=False)
summary_df.to_csv(REFINED_OVERRIDE_SUMMARY_CSV, index=False)
by_year_df.to_csv(REFINED_OVERRIDE_BY_YEAR_CSV, index=False)
by_tenor_df.to_csv(REFINED_OVERRIDE_BY_TENOR_CSV, index=False)
override_attr.to_csv(REFINED_OVERRIDE_ATTRIBUTION_CSV, index=False)

print("Saved refined override files:")
print(REFINED_OVERRIDE_SELECTED_PARQUET)
print(REFINED_OVERRIDE_SUMMARY_CSV)
print(REFINED_OVERRIDE_BY_YEAR_CSV)
print(REFINED_OVERRIDE_BY_TENOR_CSV)
print(REFINED_OVERRIDE_ATTRIBUTION_CSV)

print("\nRefined override summary:")
display(summary_df)

print("\nRefined override attribution:")
display(
    pd.DataFrame([
        {
            "refined_overrides": len(override_attr),
            "primary_pnl_on_refined_override_dates": override_attr["test_pnl_primary"].sum(),
            "hybrid_pnl_on_refined_override_dates": override_attr["test_pnl_hybrid"].sum(),
            "incremental_pnl": override_attr["incremental_pnl"].sum(),
            "avg_incremental_pnl": override_attr["incremental_pnl"].mean(),
            "median_incremental_pnl": override_attr["incremental_pnl"].median(),
            "pct_overrides_helped": (override_attr["incremental_pnl"] > 0).mean(),
            "worst_incremental_pnl": override_attr["incremental_pnl"].min(),
            "best_incremental_pnl": override_attr["incremental_pnl"].max(),
            "avg_primary_tenor": override_attr["entry_tenor_primary"].mean(),
            "avg_hybrid_tenor": override_attr["entry_tenor_hybrid"].mean(),
        }
    ])
)

print("\nRefined strategy by year:")
display(by_year_df)

print("\nRefined strategy by tenor:")
display(by_tenor_df)

print("\nWorst refined substitutions:")
display(
    override_attr[
        [
            "trade_dt",
            "entry_tenor_primary",
            "entry_tenor_hybrid",
            "tenor_group_primary",
            "tenor_group_hybrid",
            "test_pnl_primary",
            "test_pnl_hybrid",
            "incremental_pnl",
            "entry_rsi_14_primary",
            "entry_rsi_14_hybrid",
        ]
    ]
    .sort_values("incremental_pnl")
    .head(20)
)

print("\nBest refined substitutions:")
display(
    override_attr[
        [
            "trade_dt",
            "entry_tenor_primary",
            "entry_tenor_hybrid",
            "tenor_group_primary",
            "tenor_group_hybrid",
            "test_pnl_primary",
            "test_pnl_hybrid",
            "incremental_pnl",
            "entry_rsi_14_primary",
            "entry_rsi_14_hybrid",
        ]
    ]
    .sort_values("incremental_pnl", ascending=False)
    .head(20)
)

Saved refined override files:
C:\Users\patri\vrp_project\data\processed\put_refined_hybrid_override_selected_trades_v0_2.parquet
C:\Users\patri\vrp_project\data\audit\put_refined_hybrid_override_summary_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_refined_hybrid_override_by_year_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_refined_hybrid_override_by_tenor_v0_2.csv
C:\Users\patri\vrp_project\data\audit\put_refined_hybrid_override_attribution_v0_2.csv

Refined override summary:


,strategy_name,trades,start_date,end_date,total_pnl,avg_pnl,median_pnl,win_rate,q05_pnl,worst_pnl,max_drawdown,pnl_to_drawdown,worst_10_trade_sum,worst_20_trade_sum,positive_year_rate,min_year_pnl,avg_tenor,front_pct,middle_pct,back_pct
0,primary_only,602,2020-10-29,2026-06-05,3.544466e+06,5887.817014,10442.162535,0.779070,-27785.830930,-111511.425505,-944746.911842,3.751762,-533576.106123,-941213.622876,1.0,102416.500948,21.019934,0.392027,0.224252,0.383721
1,primary_plus_hybrid_priority_hybrid,602,2020-10-29,2026-06-05,3.733363e+06,6201.599183,10798.464284,0.780731,-26231.923947,-111511.425505,-944746.911842,3.951707,-533576.106123,-941213.622876,1.0,110810.890918,21.772425,0.360465,0.184385,0.455150
2,primary_with_refined_hybrid_override,602,2020-10-29,2026-06-05,3.795287e+06,6304.463174,10937.260361,0.782392,-26231.923947,-111511.425505,-944746.911842,4.017252,-533576.106123,-941213.622876,1.0,112187.213271,21.996678,0.352159,0.186047,0.461794



Refined override attribution:


,refined_overrides,primary_pnl_on_refined_override_dates,hybrid_pnl_on_refined_override_dates,incremental_pnl,avg_incremental_pnl,median_incremental_pnl,pct_overrides_helped,worst_incremental_pnl,best_incremental_pnl,avg_primary_tenor,avg_hybrid_tenor
0,47,187810.911197,438631.899258,250820.988061,5336.616767,3311.818016,0.765957,-19178.473072,51417.133707,15.829787,28.340426



Refined strategy by year:


...

,entry_tenor,trades,total_pnl,avg_pnl,win_rate,worst_pnl,avg_rsi
0,9,58,1.075213e+05,1853.814715,0.724138,-40441.053294,52.986863
1,12,103,1.467190e+05,1424.456749,0.660194,-49261.410409,55.626677
2,15,51,4.192003e+05,8219.614050,0.823529,-44768.311536,52.702568
3,18,72,3.673318e+05,5101.830913,0.833333,-49418.522467,59.122572
4,21,23,2.008474e+05,8732.493763,0.869565,-40497.519097,50.278298
5,24,17,1.563616e+05,9197.739174,0.764706,-33174.909732,47.593454
6,27,85,6.854119e+05,8063.669209,0.788235,-111511.425505,48.045604
7,30,44,2.471622e+05,5617.321603,0.795455,-100415.930124,50.746579
8,33,149,1.464731e+06,9830.412307,0.832215,-75007.519387,55.068612



Worst refined substitutions:


,trade_dt,entry_tenor_primary,entry_tenor_hybrid,tenor_group_primary,tenor_group_hybrid,test_pnl_primary,test_pnl_hybrid,incremental_pnl,entry_rsi_14_primary,entry_rsi_14_hybrid
562,2026-01-30,9,33.0,front_9_15d,back_27_33d,6730.047283,-12448.425789,-19178.473072,53.359087,53.359087
579,2026-03-02,15,27.0,front_9_15d,back_27_33d,-39615.671891,-57319.352129,-17703.680238,48.647576,48.647576
163,2022-01-24,18,27.0,middle_18_24d,back_27_33d,23468.695934,11172.006267,-12296.689667,28.552336,28.552336
556,2026-01-16,18,27.0,middle_18_24d,back_27_33d,9755.029171,-2165.702931,-11920.732103,56.625318,56.625318
555,2026-01-15,18,27.0,middle_18_24d,back_27_33d,8140.290044,-2452.310975,-10592.601019,57.286866,57.286866
131,2021-09-10,12,27.0,front_9_15d,back_27_33d,11281.618811,1489.263398,-9792.355414,47.542957,47.542957
554,2026-01-14,18,27.0,middle_18_24d,back_27_33d,9514.047296,616.464066,-8897.583230,55.345903,55.345903
565,2026-02-04,12,27.0,front_9_15d,back_27_33d,3816.804984,-4108.840691,-7925.645675,46.423596,46.423596
564,2026-02-03,12,33.0,front_9_15d,back_27_33d,-2216.019232,-9884.631119,-7668.611887,50.152892,50.152892
566,2026-02-05,15,27.0,front_9_15d,back_27_33d,14356.319134,11270.298894,-3086.020240,38.932069,38.932069



Best refined substitutions:


,trade_dt,entry_tenor_primary,entry_tenor_hybrid,tenor_group_primary,tenor_group_hybrid,test_pnl_primary,test_pnl_hybrid,incremental_pnl,entry_rsi_14_primary,entry_rsi_14_hybrid
581,2026-03-04,18,33.0,middle_18_24d,back_27_33d,-38419.098915,12998.034791,51417.133707,48.264262,48.264262
583,2026-03-06,18,27.0,middle_18_24d,back_27_33d,-33182.394118,339.761603,33522.155721,38.453183,38.453183
585,2026-03-10,24,27.0,middle_18_24d,back_27_33d,-9306.228139,22148.557542,31454.785681,42.825712,42.825712
135,2021-09-16,18,27.0,middle_18_24d,back_27_33d,-15324.951104,14685.666387,30010.617491,50.908880,50.908880
424,2024-10-25,9,27.0,front_9_15d,back_27_33d,-5647.266241,15977.631316,21624.897557,56.431722,56.431722
45,2021-02-23,12,27.0,front_9_15d,back_27_33d,1221.218281,17287.710267,16066.491986,54.833172,54.833172
580,2026-03-03,15,27.0,front_9_15d,back_27_33d,-29210.915071,-13409.852082,15801.062989,43.035216,43.035216
584,2026-03-09,12,27.0,front_9_15d,back_27_33d,-28107.751777,-12685.421844,15422.329933,43.879944,43.879944
582,2026-03-05,18,27.0,middle_18_24d,back_27_33d,-32473.930236,-17217.829479,15256.100757,45.000430,45.000430
130,2021-09-09,12,33.0,front_9_15d,back_27_33d,1865.007300,12767.955703,10902.948403,55.086042,55.086042


In [21]:
# ============================================================
# Cell 35: Locate actual expiry / strike / stress columns
# ============================================================

import pandas as pd
from pathlib import Path

search_dirs = [
    PROCESSED_DATA_DIR,
    AUDIT_DIR,
]

keywords = [
    "expiry", "expiration", "expir", "maturity",
    "strike", "short", "long", "width", "credit",
    "stress", "shock", "delta", "vega",
    "notional", "size", "premium", "pnl",
]

parquet_files = []

for d in search_dirs:
    parquet_files.extend(sorted(Path(d).glob("*.parquet")))

rows = []

for path in parquet_files:
    try:
        df_head = pd.read_parquet(path, engine="pyarrow")
        cols = list(df_head.columns)

        matching_cols = [
            c for c in cols
            if any(k.lower() in c.lower() for k in keywords)
        ]

        rows.append({
            "file": str(path),
            "rows": len(df_head),
            "n_cols": len(cols),
            "matching_cols": matching_cols,
        })

    except Exception as e:
        rows.append({
            "file": str(path),
            "rows": None,
            "n_cols": None,
            "matching_cols": f"ERROR: {e}",
        })

column_search_df = pd.DataFrame(rows)

pd.set_option("display.max_colwidth", 300)

display(
    column_search_df
    .assign(n_matching=lambda x: x["matching_cols"].apply(lambda y: len(y) if isinstance(y, list) else 0))
    .sort_values("n_matching", ascending=False)
    .head(30)
)

print("\nCandidate selected trade columns:")
candidate = pd.read_parquet(
    PROCESSED_DATA_DIR / "put_strategy_candidate_v0_2_selected_trades.parquet"
)
print(candidate.columns.tolist())

,file,rows,n_cols,matching_cols,n_matching
44,C:\Users\patri\vrp_project\data\processed\put_phase3_stress_zone_daily_v0_1.parquet,2697,48,"[max_same_expiry_stress_loss_pct_of_nw__down_10_vol_up_6, max_same_expiry_stress_loss_pct_of_nw__down_15_vol_up_9, max_same_expiry_stress_loss_pct_of_nw__down_20_vol_up_15, max_same_expiry_stress_loss_pct_of_nw__down_5_vol_up_3, max_same_expiry_stress_loss_pct_of_nw__dynamic_moderate_vol_up_9, o...",42
48,C:\Users\patri\vrp_project\data\processed\put_phase3_trade_vol_of_vol_features_v0_1.parquet,378,146,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expired_otm, entry_credit_points, short_option_pnl_points, contracts_for_normalized...",35
50,C:\Users\patri\vrp_project\data\processed\put_portfolio_sizing_paths_v0_1.parquet,11495,107,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expired_otm, entry_credit_points, short_option_pnl_points, contracts_for_normalized...",34
21,C:\Users\patri\vrp_project\data\processed\put_balanced_cap_portfolio_strategy_v0_1.parquet,422,99,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expired_otm, entry_credit_points, short_option_pnl_points, contracts_for_normalized...",29
46,C:\Users\patri\vrp_project\data\processed\put_phase3_trade_champion_forecast_diagnostic_v0_1.parquet,378,112,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expired_otm, entry_credit_points, short_option_pnl_points, contracts_for_normalized...",29
34,C:\Users\patri\vrp_project\data\processed\put_core_friendly_cap_accepted_trades_v0_1.parquet,2037,99,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expired_otm, entry_credit_points, short_option_pnl_points, contracts_for_normalized...",29
41,C:\Users\patri\vrp_project\data\processed\put_open_risk_cap_accepted_trades_v0_1.parquet,1884,99,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expired_otm, entry_credit_points, short_option_pnl_points, contracts_for_normalized...",29
43,C:\Users\patri\vrp_project\data\processed\put_phase3_portfolio_strategy_v0_1.parquet,378,99,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expired_otm, entry_credit_points, short_option_pnl_points, contracts_for_normalized...",29
61,C:\Users\patri\vrp_project\data\processed\trade_candidate_pricing_v0_1.parquet,1829,62,"[target_expiry_date, expiry_selection_rule, short_strike_rule, short_strike_raw, selected_expiry_date, expiry_distance_days, num_available_expiries_for_trade_date, expiry_mapping_status, short_strike_raw_actual_dte, short_strike_selected, strike_mapping_status, short_strike_raw_target_tenor, sho...",26
24,C:\Users\patri\vrp_project\data\processed\put_best_risk_weighted_strategy_v0_1.parquet,605,96,"[target_expiry_date, expiry_mapping_status, selected_expiry_date, expiry_distance_days, strike_mapping_status, short_strike_selected, raw_short_strike, expiry_spx_close, expiry_spx_status, expiry_intrinsic_value, expire


Candidate selected trade columns:
['trade_date', 'tenor', 'spx_close', 'spx_rsi_14', 'vix_style_vol', 'implied_variance', 'old_vrp_signal', 'old_vol', 'old_vrp_3m_z', 'old_vrp_1y_z', 'old_vrp_blended_z', 'simple_forecast_model', 'simple_forecast_variance', 'forecast_vol', 'forecast_vrp_signal', 'forecast_vrp_3m_z', 'forecast_vrp_1y_z', 'forecast_vrp_blended_z', 'target_tenor', 'side', 'source_universe', 'trade_date_ts', 'target_expiry_date', 'entry_spx_close', 'tenor_group', 'trade_year', 'expiry_mapping_status', 'selected_expiry_date', 'selected_chain_file', 'expiry_distance_days', 'actual_dte', 'strike_mapping_status', 'short_strike_selected', 'raw_short_strike', 'put_moneyness', 'put_pct_otm', 'entry_quote_status', 'entry_bid', 'entry_ask', 'entry_mid', 'entry_spread', 'entry_spread_pct_mid', 'mapping_status', 'expiry_spx_close', 'expiry_spx_status', 'expiry_intrinsic_value', 'expired_otm', 'entry_credit_points', 'short_option_pnl_points', 'dollars_per_contract', 'contracts_for_nor